# Pixel3DMM: хронологическая обработка фото с Google Drive
[Официальный код](https://github.com/SimonGiebenhain/pixel3dmm). Каждое фото получает независимую
FLAME-реконструкцию: свою 3D-модель, каноническую модель, параметры, ландмарки, маски и превью.

## Запуск
1. Выбери GPU T4/L4 в **Runtime > Change runtime type**.
2. Фото `.jpg`, `.jpeg`, `.png` положи в **`MyDrive/input`**. Подпапки поддерживаются.
3. Лицензированные **`FLAME2020.zip` и `FLAME2023.zip`** положи прямо в корень `MyDrive`.
4. Выполни ячейки **1 > 2 > 3 > 4 > 5**. Выбирать режимы, запускать демо и отдельно делать `plan` не нужно.
5. Ячейка **6** показывает сохраненный статус и последнее готовое фото. Ей не нужен GPU.

Каждая исполняемая ячейка показывает живую панель: текущий шаг, имя фото, счетчики, время и последние события.
Таймер обновляется раз в секунду даже при молчащей CUDA-компиляции. Полный вывод сохраняется в
`/content/logs/notebook/`; журналы обработки фото также сохраняются на Drive.

## Продолжение
`START_FROM = ""` в ячейке 1 означает: весь корпус по дате в имени, готовые фото автоматически пропускаются.
Для ручного продолжения укажи **полное имя с расширением**, например `START_FROM = "2016_05_20.jpg"`.
Если одинаковые имена есть в разных подпапках, укажи относительный путь: `put/frontal/2016_05_20.jpg`.
Указанное фото включается; все предыдущие в хронологии пропускаются еще до чтения изображений и запуска GPU.
При одинаковой дате порядок определяется именем и относительным путем. Опечатка в `START_FROM` останавливает
запуск, а не начинает корпус заново.

Поддерживаются даты `YYYY_MM_DD`, `YYYY-MM-DD`, `YYYY.MM.DD`, `YYYYMMDD`, `DD.MM.YYYY` (также `_`/`-`)
и время `HHMMSS`/`HH_MM_SS` после даты. Фото без корректной даты **не анализируются**: их список сохраняется
в `OUTPUT/_state/undated.json`. Дата берется именно из имени, не из времени загрузки на Drive.

Состояние сохраняется в `OUTPUT/_state/resume.json`: текущее фото, последнее готовое и следующая точка продолжения.
После отключения Colab снова выполни 1 > 2 > 3 > 4 > 5. `START_FROM` можно оставить пустым:
готовые папки с `done.json` не пересчитываются. Не запускай два ноутбука одновременно с одним `OUTPUT`.

## Скорость
Первую сборку CUDA полностью убрать нельзя. Совместимые Python 3.9 / PyTorch cu118 / GCC 11 устанавливаются
в `/content/p3dmm-env`, не затрагивая системный Python Colab. Исправная среда не переустанавливается,
проверенные веса не скачиваются заново. После ячейки 4 среда и веса автоматически кешируются в `OUTPUT/_env`;
ячейка 3 восстанавливает их в следующей сессии. Понадобится дополнительное место на Drive.
При смене GPU или повреждении кеша несовместимые CUDA-сборки придется восстановить.

Для ~1900 фото нужны несколько сессий. `SESSION_MAX_HOURS` ограничивает один запуск; перед подготовкой новой
группы и перед каждым фото проверяются время и свободное место. Оптимизация остается 800 итераций на фото:
качество ради скорости установки не уменьшается.


## 1. Параметры и Google Drive
Для продолжения меняй только `START_FROM`. Пути и бюджет сессии находятся в начале ячейки.


In [ ]:
import collections, html, json, os, pathlib, signal, subprocess, sys, threading, time
from IPython.display import HTML, Image, display


class CellStatus:
    """One live output per cell; complete subprocess output stays in a local log."""

    def __init__(self, title, total=None):
        self.title, self.total = title, total
        self.completed, self.phase, self.current, self.stats = 0, "Запуск", "", ""
        self.lines = collections.deque(maxlen=12)
        self.state, self.warning = "running", False
        self.started = time.monotonic()
        self.stop = threading.Event()
        self.lock = threading.RLock()
        self.handle = None

    def __enter__(self):
        self.handle = display(HTML(self.render()), display_id=True)
        self.worker = threading.Thread(target=self.refresh, daemon=True)
        self.worker.start()
        return self

    def render(self):
        with self.lock:
            esc = html.escape
            elapsed = int(time.monotonic() - self.started)
            clock = f"{elapsed // 3600:02d}:{elapsed // 60 % 60:02d}:{elapsed % 60:02d}"
            labels = {"running": "Выполняется", "done": "Готово", "warning": "Есть предупреждения",
                      "error": "Ошибка", "stopped": "Остановлено"}
            color = {"done": "#27865b", "warning": "#ae7415", "error": "#c44545",
                     "stopped": "#ae7415"}.get(self.state, "#3979b9")
            if self.total:
                value = min(self.total, max(0, self.completed))
                meter = f'<progress max="{self.total}" value="{value}" style="width:100%"></progress>'
                count = f"{value}/{self.total} ({100 * value / self.total:.1f}%)"
            else:
                meter, count = '<progress style="width:100%"></progress>', ""
            recent = esc("\n".join(self.lines) or "Ожидание вывода. Таймер обновляется каждую секунду.")
            return (f'<section style="border:1px solid {color};border-left:5px solid {color};'
                    'border-radius:8px;padding:14px;font:14px/1.5 system-ui">'
                    f'<div style="display:flex;justify-content:space-between;gap:12px">'
                    f'<strong>{esc(self.title)}</strong><span>{labels[self.state]} | {clock}</span></div>'
                    f'<div style="margin-top:8px">{esc(self.phase)}</div>'
                    f'<div style="font-family:monospace;overflow-wrap:anywhere">{esc(self.current)}</div>'
                    f'<div>{esc(self.stats)}</div>{meter}<div>{count}</div>'
                    '<details open><summary>Последние события (полный лог: /content/logs/notebook/)</summary>'
                    f'<pre style="white-space:pre-wrap;overflow-wrap:anywhere;max-height:220px;'
                    f'overflow:auto;font-size:12px;margin-bottom:0">{recent}</pre></details></section>')

    def refresh(self):
        while not self.stop.wait(1):
            self.handle.update(HTML(self.render()))

    def update(self, **values):
        with self.lock:
            for key in ("phase", "current", "stats", "completed", "total"):
                if key in values:
                    setattr(self, key, values[key])

    def note(self, text):
        with self.lock:
            for line in str(text).replace("\r", "\n").splitlines():
                if line.startswith("__P3DMM_STATUS__"):
                    try:
                        event = json.loads(line[len("__P3DMM_STATUS__"):])
                        self.update(**event)
                    except (ValueError, TypeError):
                        self.lines.append(line[:600])
                elif line.strip():
                    self.lines.append(line[:600])

    def warn(self, text):
        self.warning = True
        self.note("ПРЕДУПРЕЖДЕНИЕ: " + text)

    def run(self, command, label, *, cwd="/content", check=True):
        self.update(phase=label, current="")
        folder = pathlib.Path("/content/logs/notebook")
        folder.mkdir(parents=True, exist_ok=True)
        name = "".join(c if c.isalnum() else "_" for c in self.title)[:60]
        env = dict(os.environ, PYTHONUNBUFFERED="1", P3DMM_NOTEBOOK_STATUS="1",
                   PIP_DISABLE_PIP_VERSION_CHECK="1", PIP_PREFER_BINARY="1")
        with (folder / (name + ".log")).open("a", encoding="utf-8") as log:
            log.write("\n$ " + " ".join(map(str, command)) + "\n")
            log.flush()
            proc = subprocess.Popen(command, cwd=cwd, env=env, stdout=subprocess.PIPE,
                                    stderr=subprocess.STDOUT, text=True, errors="replace",
                                    bufsize=1, start_new_session=True)

            def read_output():
                for line in proc.stdout:
                    log.write(line)
                    log.flush()
                    self.note(line)

            reader = threading.Thread(target=read_output, daemon=True)
            reader.start()
            try:
                rc = proc.wait()
            except BaseException:
                try:
                    os.killpg(proc.pid, signal.SIGINT)
                    proc.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    os.killpg(proc.pid, signal.SIGKILL)
                    proc.wait()
                except ProcessLookupError:
                    pass
                raise
            finally:
                reader.join(timeout=10)
                proc.stdout.close()
            if check and rc:
                raise subprocess.CalledProcessError(rc, command)
            return rc

    def __exit__(self, kind, error, traceback):
        self.stop.set()
        self.worker.join(timeout=2)
        if kind:
            self.state = "stopped" if issubclass(kind, KeyboardInterrupt) else "error"
            self.note(str(error) or "Выполнение прервано. Готовые результаты остаются на Drive.")
        else:
            self.state = "warning" if self.warning else "done"
            self.phase = "Завершено"
            if self.total:
                self.completed = self.total
        self.handle.update(HTML(self.render()))

INPUT = "MyDrive/input"       # фиксированная папка с фото, включая подпапки
OUTPUT = "MyDrive/output"     # готовые результаты и состояние между сессиями
START_FROM = ""               # @param {type:"string"}
SESSION_MAX_HOURS = 12.0      # @param {type:"number"}

with CellStatus("1. Google Drive и параметры", total=4) as status:
    status.update(phase="Подключение Google Drive")
    DRIVE = pathlib.Path("/content/drive/MyDrive")
    if not DRIVE.is_dir():
        from google.colab import drive
        drive.mount("/content/drive")
    if not DRIVE.is_dir():
        raise RuntimeError("Google Drive не подключен. Повтори эту ячейку и разреши доступ.")

    status.update(phase="Проверка папок", completed=1)
    def drive_path(raw):
        raw = str(raw).strip().strip('"')
        path = pathlib.Path(raw) if raw.startswith("/") else DRIVE / raw.removeprefix("MyDrive/")
        path = path.resolve()
        if not path.is_relative_to(DRIVE.resolve()):
            raise ValueError("Вход и выход должны находиться в MyDrive.")
        return path

    src, dst = drive_path(INPUT), drive_path(OUTPUT)
    if not src.is_dir():
        raise FileNotFoundError(f"Нет папки {src}. Фото должны лежать в MyDrive/input.")
    if dst.is_relative_to(src):
        raise ValueError("OUTPUT не должен находиться внутри INPUT: результаты не являются входными фото.")
    dst.mkdir(parents=True, exist_ok=True)
    status.note(f"Фото: {src}\nРезультаты: {dst}")

    status.update(phase="Проверка FLAME в корне MyDrive", completed=2)
    for filename in ("FLAME2020.zip", "FLAME2023.zip"):
        archive = DRIVE / filename
        if not archive.is_file() or not archive.stat().st_size:
            raise FileNotFoundError(f"Положи {filename} в корень Google Drive: {archive}")
        status.note(f"{filename}: {archive.stat().st_size / 1e6:.1f} МБ")
    if not 0 < float(SESSION_MAX_HOURS) <= 12:
        raise ValueError("SESSION_MAX_HOURS должен быть больше 0 и не больше 12.")

    status.update(phase="Сохранение параметров", completed=3)
    CONFIG = {"input": str(src), "output": str(dst), "start_from": str(START_FROM).strip(),
              "session_max_hours": float(SESSION_MAX_HOURS), "unit": "photo",
              "export": "photos", "use_flame2023": "true", "torch_compile": False,
              "sync_to_drive": True, "keep_archive": False}
    pathlib.Path("/content/p3dmm_config.json").write_text(
        json.dumps(CONFIG, ensure_ascii=False, indent=1), encoding="utf-8")
    for folder in ("/content/work", "/content/stage", "/content/logs",
                   "/content/p3dmm/preprocessed", "/content/p3dmm/tracking"):
        pathlib.Path(folder).mkdir(parents=True, exist_ok=True)
    env_dir = pathlib.Path.home() / ".config/pixel3dmm"
    env_dir.mkdir(parents=True, exist_ok=True)
    (env_dir / ".env").write_text(
        "PIXEL3DMM_CODE_BASE=/content/pixel3dmm/\n"
        "PIXEL3DMM_PREPROCESSED_DATA=/content/p3dmm/preprocessed/\n"
        "PIXEL3DMM_TRACKING_OUTPUT=/content/p3dmm/tracking/\n", encoding="utf-8")
    resume = dst / "_state/resume.json"
    if resume.is_file():
        try:
            saved = json.loads(resume.read_text(encoding="utf-8"))
            next_photo = (saved.get("next_photo") or {}).get("relpath")
            if next_photo:
                status.note(f"Сохраненная точка продолжения: START_FROM = {next_photo!r}")
        except (OSError, ValueError):
            status.warn("Не удалось прочитать resume.json; готовые папки все равно будут пропущены.")
    status.note("START_FROM: " + (START_FROM or "с самого раннего незавершенного фото"))


## 2. Ядро и очередь фото
Быстро читает имена файлов, проверяет `START_FROM` и сохраняет хронологический план. Не копирует весь корпус,
не открывает изображения и не расходует GPU на предварительный анализ уже пропущенных фото.


In [ ]:
PIPELINE_SOURCE = '# -*- coding: utf-8 -*-\n"""Colab pipeline: independent photo fits, chronological queue and durable Drive state.\n\nThe notebook runs plan/status without GPU dependencies, and preflight/run in the isolated\nPython environment. Input is a Drive photo folder. Finished per-photo folders are preserved\nacross runtime resets; freeze/warm cache the installation, not the processing results.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport os\nimport posixpath\nimport re\nimport shutil\nimport signal\nimport subprocess\nimport sys\nimport tarfile\nimport time\nfrom collections import Counter\nfrom datetime import datetime\nfrom pathlib import Path\n\nENV_PY = "/content/p3dmm-env/bin/python"\nENV_ROOT = Path("/content/p3dmm-env")\nCODE = Path("/content/pixel3dmm")\nPREPRO = Path("/content/p3dmm/preprocessed")\nTRACK = Path("/content/p3dmm/tracking")\nSTAGE = Path("/content/stage")\nLOGS = Path("/content/logs")\nWORK = Path("/content/work")\nCFG_FILE = Path("/content/p3dmm_config.json")\nPLAN_FILE = Path("/content/work/plan.json")\nLOCAL_LOG = Path("/content/work/batch_log.jsonl")\nPHOTO_LOG = Path("/content/work/photos_log.jsonl")     # unit=photo: журнал по каждому фото\nSTOP_FILE = Path("/content/STOP")\nDRIVE_ROOT = Path("/content/drive/MyDrive")\nFLAME_DATA = CODE / "src/pixel3dmm/preprocessing/MICA/data"\nPIPNET = CODE / "src/pixel3dmm/preprocessing/PIPNet"\nIMG_EXT = (".jpg", ".jpeg", ".png")\nVIDEO_EXT = (".mp4", ".mov", ".avi", ".mkv", ".webm", ".m4v")\n# обязательные файлы FLAME: без них трекинг падает сразу после инициализации\nREQUIRED_FLAME2020 = ("generic_model.pkl", "landmark_embedding.npy",\n                      "FLAME_masks/FLAME_masks.pkl")\nFLAME2023_PKL = "FLAME2023/flame2023_no_jaw.pkl"\nHF_REV = "7f5141a5edb10fa57caa705e8cb029f44630f788"\nHF_FILES = {\n    "generic_model.pkl": (53_023_716, "efcd14cc4a69f3a3d9af8ded80146b5b6b50df3bd74cf69108213b144eba725b"),\n    "mica.tar": (502_647_880, "4542a467d9e8f7521474a1d00eac89552bebef0b331b72bf7fbd6f065ff64d7b"),\n    "epoch59.pth": (49_393_015, "44002daaf3187e6e4b99e0fd9a52f60ffa05e06ec5df3721ace950fd2f2cc120"),\n    "uv.ckpt": (2_246_794_577, "dff9d73feec47914b704759f57ebffb8c58d2aef550b426013fe31eae21707b8"),\n    "normals.ckpt": (1_469_022_184, "e856799d55db54c7537c8ee3c5a4938c13cc0b24082ce7e4e7f35f0d0f0e28da"),\n}\nfor _model in ("antelopev2", "buffalo_l"):\n    for _name, _size, _sha in (\n        ("1k3d68.onnx", 143_607_619, "df5c06b8a0c12e422b2ed8947b8869faa4105387f199c477af038aa01f9a45cc"),\n        ("2d106det.onnx", 5_030_888, "f001b856447c413801ef5c42091ed0cd516fcd21f2d6b79635b1e733a7109dbf"),\n        ("genderage.onnx", 1_322_532, "4fde69b1c810857b88c64a335084f1c3fe8f01246c9a191b48c7bb756d6652fb"),\n        ("scrfd_10g_bnkps.onnx" if _model == "antelopev2" else "det_10g.onnx", 16_923_827,\n         "5838f7fe053675b1c7a08b633df49e7af5495cee0493c7dcf6697200b85b5b91"),\n    ):\n        HF_FILES[f"insightface/models/{_model}/{_name}"] = (_size, _sha)\nHF_FILES["insightface/models/antelopev2/glintr100.onnx"] = (\n    260_665_334, "4ab1d6435d639628a6f3e5008dd4f929edf4c4124b1a7169e1048f9fef534cdf")\nHF_FILES["insightface/models/buffalo_l/w600k_r50.onnx"] = (\n    174_383_860, "4c06341c33c2ca1f86781dab0e829f88ad5b64be9fba56e56bc9ebdefc619e43")\nWEIGHTS = (("pretrained_weights/uv.ckpt", HF_FILES["uv.ckpt"][0]),\n           ("pretrained_weights/normals.ckpt", HF_FILES["normals.ckpt"][0]),\n           (str(FLAME_DATA.relative_to(CODE)) + "/pretrained/mica.tar", HF_FILES["mica.tar"][0]),\n           ("src/pixel3dmm/preprocessing/PIPNet/snapshots/WFLW/pip_32_16_60_r18_l2_l1_10_1_nb10/epoch59.pth", HF_FILES["epoch59.pth"][0]))\n# что из предобработки уезжает в Drive вместе с результатом\nPREPROC_SUBSET = ("cropped", "seg_og", "PIPnet_landmarks", "p3dmm/normals", "p3dmm/uv_map")\n# unit=photo: (имя файла в папке фото, подпапка предобработки, расширение) — всё по одному снимку\nPHOTO_PREPROC = (("crop.jpg", "cropped", ".jpg"),\n                 ("segmentation.png", "seg_og", ".png"),\n                 ("landmarks_pipnet.npy", "PIPnet_landmarks", ".npy"),\n                 ("normals.png", "p3dmm/normals", ".png"),\n                 ("uv_map.png", "p3dmm/uv_map", ".png"),\n                 ("arcface.npy", "arcface", ".npy"))\n# MICA пишет на кадр отдельной папкой mica/<номер кадра>/ — это тоже данные фото\nPHOTO_MICA = (("mica_identity.npy", "identity.npy"),\n              ("mica_mesh.ply", "mesh.ply"),\n              ("mica_kpt7.npy", "kpt7.npy"),\n              ("mica_kpt68.npy", "kpt68.npy"))\n# mica_demo.py берёт кадры с шагом len(cropped)//10: за один вызов больше 10 — и часть фото\n# останется без своего mica_identity/arcface, поэтому недостающее предоббатываем группами по 10\nMICA_GROUP = 10\n# ядро, без которого трекер не читает кадр; остальное (arcface + MICA) даёт тот же вызов MICA\nPHOTO_CORE = tuple(p for p in PHOTO_PREPROC if p[1] != "arcface")\n# по этому набору решаем, что фото посчитано и папку можно пропускать\nPHOTO_REQUIRED = ("mesh.ply", "canonical.ply", "params.json", "done.json")\n# --- наблюдаемость лица на снимке: те же классы FaceR, из которых read_data() собирает свои маски ---\n# без этой правки потом нельзя отличить «сторона не видна» от «сторона другая»\nSEG_FG = (1, 2, 4, 5, 6, 7, 8, 9, 10, 12, 13)      # fg_mask: силуэт головы вообще\nSEG_UV = (2, 6, 7, 10, 12, 13)                     # uv_mask: где реально работали UV-сопоставления\nSEG_NORMAL = SEG_UV + (11,)                        # normal_mask: где работали нормали (big_normal_mask выкл)\nMASK_SIZE = 256                                    # config.size — в этом разрешении трекер видел кадр\n# PIPNet WFLW: первые 68 точек идут в порядке iBUG-68, дальше дополнительный контур\nPIPNET_ZONES = ((range(0, 17), "jaw"), (range(17, 22), "brow_right"), (range(22, 27), "brow_left"),\n                (range(27, 36), "nose"), (range(36, 42), "eye_right"), (range(42, 48), "eye_left"),\n                (range(48, 60), "mouth_outer"), (range(60, 68), "mouth_inner"))\n# ракурс считаем не «на глаз по 2D»: та же девятибинная таблица и та же математика, что в\n# pixel3dmm_analysis.py (разбор корпуса), чтобы метка в params.json и метка при сравнении не разъехались\nVIEW_BINS = (\n    ("left_profile", -95.0, -50.0, -70.0),\n    ("left_deep", -50.0, -40.0, -45.0),\n    ("left_mid", -40.0, -25.0, -32.5),\n    ("left_light", -25.0, -10.0, -17.5),\n    ("frontal", -10.0, 10.0, 0.0),\n    ("right_light", 10.0, 25.0, 17.5),\n    ("right_mid", 25.0, 40.0, 32.5),\n    ("right_deep", 40.0, 50.0, 45.0),\n    ("right_profile", 50.0, 95.0, 70.0),\n)\nPOSE_CONVENTION = "yaw positive nose-to-image-right; pitch positive up; roll camera-up"\n# tracker.py: MAX_STEPS = min(len(cropped) - start_frame, 1000) — дальше кадры просто не пишутся\nMAX_TRACK_FRAMES = 1000\n# типичная независимая примерка одного фото на T4 (iters=800) — оценка до первого замера в журнале\nPHOTO_DEFAULT_S = 90.0\n\nDEFAULT_CONFIG = {\n    "input": "MyDrive/input",\n    "output": "MyDrive/output",\n    "start_from": "",\n    "export": "photos",\n    "face_filter": True,\n    "face_min_score": 0.75,\n    "use_flame2023": "true",\n    "torch_compile": False,\n    "sync_to_drive": True,\n    "single_iters": 800,\n    "attempts": 3,\n    "min_free_gb": 2.0,\n    "session_max_hours": 12.0,\n    "session_hard_limit_hours": 0.0,\n    "batch_timeout_min": 90,\n    "keep_preproc": True,\n    "unit": "photo",\n    "photos_per_chunk": 40,\n    "max_photos_per_run": 500,\n    "photo_timeout_min": 15,\n    "keep_archive": False,\n}\n\n\n# ----------------------------------------------------------------------------- база\n\ndef say(text=""):\n    print(text, flush=True)\n\n\ndef progress(**values):\n    """Structured updates are consumed by the notebook\'s persistent status panel."""\n    if os.environ.get("P3DMM_NOTEBOOK_STATUS") == "1":\n        say("__P3DMM_STATUS__" + json.dumps(values, ensure_ascii=False))\n    else:\n        text = " | ".join(str(values[k]) for k in ("phase", "current", "stats") if values.get(k))\n        if text:\n            say(text)\n\n\ndef load_config():\n    if not CFG_FILE.is_file():\n        raise SystemExit("нет /content/p3dmm_config.json — сначала выполни ячейку «1. ВХОД -> ВЫХОД»")\n    cfg = dict(DEFAULT_CONFIG)\n    cfg.update(json.loads(CFG_FILE.read_text(encoding="utf-8")))\n    if cfg["unit"] != "photo" or cfg["export"] != "photos" or cfg["keep_archive"]:\n        raise ValueError("Ноутбук поддерживает только независимые фото без архивов результатов. Повтори ячейку 1.")\n    return cfg\n\n\ndef apply_env(cfg):\n    """Все переменные, в которые упирались ошибок: пути Pixel3DMM, компилятор nvdiffrast, weights_only."""\n    os.environ["PIXEL3DMM_CODE_BASE"] = str(CODE) + "/"\n    os.environ["PIXEL3DMM_PREPROCESSED_DATA"] = str(PREPRO) + "/"\n    os.environ["PIXEL3DMM_TRACKING_OUTPUT"] = str(TRACK) + "/"\n    os.environ.pop("TORCH_FORCE_WEIGHTS_ONLY_LOAD", None)\n    os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"\n    os.environ["TOKENIZERS_PARALLELISM"] = "false"\n    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:256"\n    os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5 8.9"\n    os.environ["PATH"] = str(ENV_ROOT / "bin") + ":" + os.environ.get("PATH", "")\n    os.environ["CONDA_PREFIX"] = str(ENV_ROOT)\n    os.environ["CUDA_HOME"] = str(ENV_ROOT)\n    os.environ["LD_LIBRARY_PATH"] = str(ENV_ROOT / "lib") + ":" + os.environ.get("LD_LIBRARY_PATH", "")\n    gcc = str(ENV_ROOT / "bin/x86_64-conda-linux-gnu-gcc")\n    gxx = str(ENV_ROOT / "bin/x86_64-conda-linux-gnu-g++")\n    if os.path.exists(gcc):\n        os.environ["CC"] = gcc\n        os.environ["CXX"] = gxx\n        os.environ["CUDAHOSTCXX"] = gxx\n    os.environ["MAX_JOBS"] = "2"\n    try:\n        import torch\n        if torch.cuda.is_available():\n            arch = torch.cuda.get_device_capability()\n            os.environ["TORCH_CUDA_ARCH_LIST"] = f"{arch[0]}.{arch[1]}" if arch <= (9, 0) else "8.9+PTX"\n    except ImportError:\n        pass\n    for d in (PREPRO, TRACK, STAGE, LOGS, WORK):\n        d.mkdir(parents=True, exist_ok=True)\n\n\ndef patch_compile(enable):\n    """COMPILE в tracker.py по умолчанию True -> torch.compile/triton. выключаем, если не просили иначе."""\n    path = CODE / "src/pixel3dmm/tracking/tracker.py"\n    if not path.is_file():\n        return "(pixel3dmm ещё не склонирован)"\n    text = path.read_text(encoding="utf-8", errors="ignore")\n    want = "COMPILE = True" if enable else "COMPILE = False"\n    fixed = re.sub(r"^COMPILE = (True|False)$", want, text, count=1, flags=re.M)\n    if fixed != text:\n        path.write_text(fixed, encoding="utf-8")\n    return want\n\n\n# upstream-трекер умеет начинать с start_frame, но число кадров ограничивает только длиной папки,\n# а MICA берёт средним по всем кадрам. для unit="photo" это два blockers: прогон одного кадра\n# превращается в прогон «всего, что осталось после start_frame», а форма инициализируется чужими лицами.\nNUM_FRAMES_MARK = "# p3dmm-colab: num_frames"\nMICA_FRAME_MARK = "# p3dmm-colab: per-frame mica"\n\n\ndef patch_single_frame_support():\n    """Делает возможным независимую примерку ровно одного кадра (num_frames=1). Идемпотентен."""\n    path = CODE / "src/pixel3dmm/tracking/tracker.py"\n    if not path.is_file():\n        return "нет tracker.py — выполни ячейку «3. Окружение»"\n    text = path.read_text(encoding="utf-8", errors="ignore")\n    applied = []\n    if NUM_FRAMES_MARK not in text:\n        anchor = "        self.BATCH_SIZE = min(self.config.batch_size, self.MAX_STEPS)"\n        step = ("        " + NUM_FRAMES_MARK + "\\n"\n                "        if getattr(self.config, \'num_frames\', 0):\\n"\n                "            self.MAX_STEPS = min(max(self.MAX_STEPS, 1), int(self.config.num_frames))\\n")\n        if anchor in text:\n            text = text.replace(anchor, step + anchor, 1)\n            applied.append("num_frames")\n    if MICA_FRAME_MARK not in text:\n        anchor = "            mica_shape = np.mean(mica_shapes, axis=0)"\n        step = ("            " + MICA_FRAME_MARK + "\\n"\n                "            if getattr(self.config, \'num_frames\', 0) and os.path.isdir("\n                "f\'{mica_folder}/{timestep:05d}\'):\\n"\n                "                mica_shape = np.squeeze(np.load("\n                "f\'{mica_folder}/{timestep:05d}/identity.npy\'))\\n"\n                "            elif getattr(self.config, \'num_frames\', 0):\\n"\n                "                raise RuntimeError(\'нет собственной MICA identity для кадра \' + str(timestep))\\n"\n                "            else:\\n"\n                "                mica_shape = np.mean(mica_shapes, axis=0)")\n        if anchor in text:\n            text = text.replace(anchor, step, 1)\n            applied.append("per-frame mica")\n    if applied:\n        path.write_text(text, encoding="utf-8")\n    return ", ".join(applied) if applied else "уже на месте (num_frames + per-frame mica)"\n\n\ndef slug(text):\n    """Имя батча должно пережить os.system без кавычек и парсинг OmegaConf CLI."""\n    out = re.sub(r"[^0-9A-Za-z_-]", "_", str(text)).strip("_")\n    return out[:60] or "x"\n\n\n# OmegaConf.from_cli разбирает одиночное значение как yaml: 123 -> int, true -> bool, null -> None.\n# такой id в video_name=... ломает пути (data_folder=<int>) -> добавляем суффикс.\nCLI_UNSAFE = {"true", "false", "null", "none", "~", "yes", "no", "on", "off", ""}\n\n\ndef safe_id(text):\n    out = slug(text)\n    # консервативно: если id не начинается с буквы, yaml может распознать его как число (1_5e3 -> float)\n    if not out[:1].isalpha() or out.lower() in CLI_UNSAFE:\n        out += "_v"\n    return out\n\n\ndef photo_folder(text):\n    """Имя папки фото в Drive: оставляем исходное имя файла (в т.ч. кириллицу), режем только запрещённое."""\n    stem = Path(str(text)).stem.strip()\n    out = re.sub(r\'[\\\\/:*?"<>|\\x00-\\x1f]\', "_", stem)[:60].strip(" .") or "frame"\n    return out\n\n\ndef file_sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef free_gb():\n    for probe in ("/content", "/tmp", "/"):\n        try:\n            return round(shutil.disk_usage(probe).free / 1e9, 1)\n        except OSError:\n            continue\n    return -1.0\n\n\ndef human(sec):\n    if sec < 90:\n        return f"{sec:.0f} с"\n    if sec < 5400:\n        return f"{sec / 60:.1f} мин"\n    return f"{sec / 3600:.1f} ч"\n\n\ndef run_cmd(cmd, log_path, timeout_min=None, cwd=None):\n    """Keep the full child log and expose new output while long GPU steps are running."""\n    log_path = Path(log_path)\n    log_path.parent.mkdir(parents=True, exist_ok=True)\n    cmd = [str(c) for c in cmd]\n    if cmd[0] in ("python", "python3"):\n        cmd[0] = ENV_PY\n    seconds = None if not timeout_min else float(timeout_min) * 60\n    started = time.monotonic()\n    deadline = started + seconds if seconds else None\n    with log_path.open("a", encoding="utf-8", errors="ignore") as fh:\n        fh.write("\\n$ " + " ".join(map(str, cmd)) + "\\n")\n        fh.flush()\n        output_start = fh.tell()\n        progress(phase="Выполняется " + Path(cmd[1]).name)\n        try:\n            child_env = dict(os.environ, PYTHONUNBUFFERED="1")\n            proc = subprocess.Popen(cmd, cwd=str(cwd or CODE), stdout=fh, env=child_env,\n                                    stderr=subprocess.STDOUT, start_new_session=True)\n            with log_path.open("r", encoding="utf-8", errors="replace") as live:\n                live.seek(output_start)\n                try:\n                    while True:\n                        remaining = deadline - time.monotonic() if deadline else 2.0\n                        if remaining <= 0:\n                            raise subprocess.TimeoutExpired(cmd, seconds)\n                        try:\n                            rc = proc.wait(timeout=min(2.0, remaining))\n                        except subprocess.TimeoutExpired:\n                            rc = None\n                        output = live.read().replace("\\r", "\\n").splitlines()\n                        for line in output[-3:]:\n                            if line.strip():\n                                say(line[:600])\n                        if rc is not None:\n                            break\n                except (subprocess.TimeoutExpired, KeyboardInterrupt):\n                    try:\n                        os.killpg(proc.pid, signal.SIGKILL)\n                    except ProcessLookupError:\n                        pass\n                    proc.wait()\n                    raise\n            reason = "" if rc == 0 else f"код {rc}"\n        except subprocess.TimeoutExpired:\n            rc, reason = -9, f"таймаут {timeout_min} мин"\n        except OSError as error:\n            rc, reason = 127, f"{type(error).__name__}: {error}"\n        fh.write(f"=== {reason or \'ok\'} ===\\n")\n    return rc, reason\n\n\ndef tail(path, lines=25):\n    try:\n        return "\\n".join(Path(path).read_text(encoding="utf-8", errors="ignore").splitlines()[-lines:])\n    except OSError:\n        return "(лога нет)"\n\n\n# ----------------------------------------------------------------------------- вход\n\ndef resolve_input(raw):\n    """Only the configured Drive directory is inspected, never all of MyDrive."""\n    raw = str(raw or "").strip().strip(\'"\').strip()\n    if not raw:\n        raise ValueError("INPUT пуст: нужна папка MyDrive/input")\n    name = raw[len("MyDrive/"):] if raw.startswith("MyDrive/") else raw\n    target = (Path(raw) if raw.startswith("/") else DRIVE_ROOT / name).resolve()\n    if not target.is_relative_to(DRIVE_ROOT.resolve()):\n        raise ValueError("INPUT должен находиться на Google Drive")\n    if not target.is_dir():\n        raise ValueError(f"не найдена папка с фото {target}; архивы и видео не являются входом")\n    return target\n\n\ndef unpack_zip(archive, dest=None):\n    import zipfile\n    archive = Path(archive)\n    if dest is None:\n        digest = hashlib.sha256(str(archive.resolve()).encode()).hexdigest()[:10]\n        dest = WORK / "input_photos" / digest\n    dest = Path(dest)\n    if archive.resolve().is_relative_to(dest.resolve()):\n        raise ValueError("архив не должен лежать в папке распаковки")\n    shutil.rmtree(dest, ignore_errors=True)\n    dest.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(archive) as zf:\n        for n in zf.namelist():\n            if n.endswith("/") or Path(n).suffix.lower() not in IMG_EXT:\n                continue\n            if n.startswith("._") or "/._" in n or "__MACOSX" in n:\n                continue\n            target = (dest / n).resolve()\n            if not str(target).startswith(str(dest.resolve()) + os.sep):\n                raise ValueError("опасный путь в архиве: " + n)\n            zf.extract(n, dest)\n    kids = [p for p in dest.iterdir() if p.is_dir()]\n    if len(kids) == 1 and not [p for p in dest.iterdir() if p.is_file()]:\n        return kids[0]\n    return dest\n\n\ndef is_photo(p):\n    return p.is_file() and p.suffix.lower() in IMG_EXT and not p.name.startswith("._")\n\n\ndef photo_moment(path):\n    """Return a sortable timestamp from the filename, or None when the date is missing/invalid."""\n    stem = Path(path).stem\n    patterns = (\n        r"(?<!\\d)(\\d{4})([_\\-.]?)(\\d{2})\\2(\\d{2})(?!\\d)",\n        r"(?<!\\d)(\\d{2})([_\\-.]?)(\\d{2})\\2(\\d{4})(?!\\d)",\n    )\n    match = None\n    for pattern in patterns:\n        match = re.search(pattern, stem)\n        if match:\n            break\n    if not match:\n        return None\n    groups = match.groups()\n    try:\n        if len(groups[0]) == 4:\n            year, month, day = int(groups[0]), int(groups[2]), int(groups[3])\n        else:\n            day, month, year = int(groups[0]), int(groups[2]), int(groups[3])\n        date = datetime(year, month, day).date()\n    except ValueError:\n        return None\n    time_match = re.search(r"(?<!\\d)(\\d{2})[:_\\-]?(\\d{2})[:_\\-]?(\\d{2})(?!\\d)", stem[match.end():])\n    if time_match:\n        try:\n            parsed_time = datetime.strptime("".join(time_match.groups()), "%H%M%S").time()\n            return datetime.combine(date, parsed_time)\n        except ValueError:\n            pass\n    return datetime.combine(date, datetime.min.time())\n\n\ndef scan_tree(root, cfg):\n    """Sort dated photos chronologically; undated photos are reported but never analyzed."""\n    cfg = cfg or {}\n    dated, undated = [], []\n    for base, dirs, files in os.walk(root):\n        dirs[:] = sorted(d for d in dirs if not d.startswith(".") and d != "__MACOSX")\n        d = Path(base)\n        for name in files:\n            if name.startswith(".") or Path(name).suffix.lower() not in IMG_EXT:\n                continue\n            path = d / name\n            if not path.is_file():\n                continue\n            rel = str(path.relative_to(root))\n            moment = photo_moment(path)\n            if moment is None:\n                undated.append(rel)\n            else:\n                dated.append((moment, path, rel))\n    dated.sort(key=lambda item: (item[0], item[2]))\n    start = str(cfg.get("start_from", "")).strip()\n    if start:\n        start_item = None\n        for item in dated:\n            if item[2] == start or Path(item[2]).name == start:\n                start_item = item\n                break\n        if start_item is None:\n            raise ValueError(f"START_FROM не найден среди датированных фото: {start!r}")\n        # Фото за тот же день сохраняются все, даже если начальная точка указана\n        # как один из вариантов "(2)" или "(3)" за эту дату.\n        start_date = start_item[0].date()\n        dated = [item for item in dated if item[0].date() >= start_date]\n    cfg["_undated_photos"] = sorted(undated)\n\n    grouped = {}\n    order = []\n    for _, path, rel in dated:\n        folder = path.parent\n        if folder not in grouped:\n            grouped[folder] = []\n            order.append(folder)\n        grouped[folder].append((path, rel))\n    batches = []\n    for folder in order:\n        rel_parts = folder.relative_to(root).parts\n        subject = slug(rel_parts[0]) if len(rel_parts) >= 1 else "flat"\n        angle = "__".join(slug(x) for x in rel_parts[1:]) if len(rel_parts) > 1 else "all"\n        bid = safe_id(subject if len(rel_parts) == 1 else f"{subject}__{angle}")\n        batches.append({\n            "id": bid,\n            "subject": subject,\n            "angle": angle,\n            "kind": "photos",\n            "folder": str(folder),\n            "n": len(grouped[folder]),\n            "frames": [{"orig": str(path), "relpath": rel} for path, rel in grouped[folder]],\n        })\n    counts = {}\n    for b in batches:\n        counts[b["id"]] = counts.get(b["id"], 0) + 1\n    for b in batches:\n        if counts[b["id"]] > 1:\n            rel = str(Path(b["folder"]).relative_to(root))\n            digest = hashlib.sha256(rel.encode()).hexdigest()[:10]\n            b["id"] = f"{b[\'id\']}__{digest}"\n    cfg["_photo_count"] = len(dated)\n    cfg["_undated_count"] = len(undated)\n    return batches\n\n\ndef scan_single(root, cfg):\n    """Один файл: видео или фото -> один батч."""\n    stem = safe_id(root.stem)\n    if root.suffix.lower() in IMG_EXT:\n        return [{"id": f"{stem}__one", "subject": stem, "angle": "one", "kind": "photos",\n                 "folder": str(root.parent), "n": 1,\n                 "frames": [{"orig": str(root), "relpath": root.name}]}]\n    if root.suffix.lower() not in VIDEO_EXT:\n        raise ValueError(f"неподдерживаемый вход: {root.name}; нужны фото, zip или видео")\n    return [{"id": stem, "subject": stem, "angle": "video", "kind": "video",\n             "folder": str(root), "n": 0, "frames": []}]\n\n\ndef build_batches(cfg, refresh=True):\n    root = resolve_input(cfg["input"])\n    batches = scan_tree(root, cfg)\n    if refresh:\n        for b in batches:\n            b["n"] = len(b["frames"])\n            stage_photos(b)\n    return [b for b in batches if b["n"] > 0]\n\n\ndef stage_video(b, root):\n    import cv2\n    capture = cv2.VideoCapture(str(root))\n    try:\n        if not capture.isOpened():\n            raise ValueError(f"видео не читается: {root}")\n        count = max(0, int(capture.get(cv2.CAP_PROP_FRAME_COUNT)))\n    finally:\n        capture.release()\n    if count > MAX_TRACK_FRAMES:\n        raise ValueError(f"видео содержит {count} кадров; разбей на части до {MAX_TRACK_FRAMES} перед plan")\n    b["n"] = count\n    dst = STAGE / (b["id"] + ".mp4")\n    dst.parent.mkdir(parents=True, exist_ok=True)\n    if Path(root).resolve() != dst.resolve():\n        shutil.copy2(root, dst)\n    b["target"] = str(dst)\n    return str(dst)\n\n\ndef stage_photos(b, keep=None):\n    """Stage one batch\'s frames as numbered images."""\n    from PIL import Image, ImageOps\n    import numpy as np\n    frames = b["frames"] if keep is None else [b["frames"][i] for i in keep]\n    dst = STAGE / b["id"]\n    if any(Path(item["orig"]).resolve().is_relative_to(dst.resolve()) for item in frames):\n        raise ValueError("INPUT находится в папке staging; перенеси исходные фото за пределы /content/stage")\n    shutil.rmtree(dst, ignore_errors=True)\n    dst.mkdir(parents=True, exist_ok=True)\n    kept = []\n    for item in frames:\n        i = len(kept)\n        progress(phase="Подготовка изображений", current=item["relpath"],\n                 stats=f"Группа: {i + 1}/{len(frames)}")\n        try:\n            with Image.open(item["orig"]) as im:\n                im = ImageOps.exif_transpose(im).convert("RGB")\n                if min(im.size) < 64 or np.asarray(im).std() < 1.0:\n                    item["note"] = (item.get("note", "") + " too_small_or_flat").strip()\n        except Exception as error:\n            item["note"] = (item.get("note", "") + f" unreadable:{type(error).__name__}").strip()\n            b.setdefault("dropped", []).append(dict(item))\n            continue\n        im.save(dst / f"{i:05d}.jpg", quality=95)\n        item["orig_frame"] = item.get("frame")\n        item["frame"] = f"{i:05d}"\n        kept.append(item)\n    b["frames"] = kept\n    b["n"] = len(kept)\n    b["target"] = str(dst)\n    write_manifest(b)\n    return str(dst)\n\n\nMANIFEST_COLS = ("frame", "subject", "angle", "batch", "orig_relpath", "orig_path", "face_score", "note")\n# плоские колонки кадра: один список на frames.csv батча и на сводный _index/frames_all.csv, иначе\n# ракурс (yaw/pitch/roll + метка из VIEW_BINS) появится в одном месте и пропадёт в другом\nFRAMES_CSV_COLS = ["frame", "batch", "subject", "angle", "orig_relpath", "orig_path", "face_score",\n                   "mesh_ply", "frame_file", "jaw_norm", "exp_norm", "shape_norm", "t", "fl", "pp",\n                   "yaw_deg", "pitch_deg", "roll_deg", "view_bin", "note"]\nFRAMES_ALL_CSV_COLS = ["batch"] + [c for c in FRAMES_CSV_COLS if c not in ("batch", "note")]\n\n\ndef manifest_path(b):\n    return WORK / "manifest" / f"{b[\'id\']}.csv"\n\n\ndef write_manifest(b):\n    path = manifest_path(b)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("w", newline="", encoding="utf-8") as fh:\n        writer = csv.DictWriter(fh, fieldnames=MANIFEST_COLS)\n        writer.writeheader()\n        if b["kind"] == "video":\n            for i in range(b.get("n_real", 0)):\n                writer.writerow({"frame": f"{i:05d}", "subject": b["subject"], "angle": "video",\n                                 "batch": b["id"], "orig_relpath": f"{Path(b[\'folder\']).name}#frame={i}",\n                                 "orig_path": b["folder"], "face_score": "", "note": "video_frame"})\n        else:\n            for item in b["frames"]:\n                writer.writerow({"frame": item["frame"], "subject": b["subject"], "angle": b["angle"],\n                                 "batch": b["id"], "orig_relpath": item["relpath"],\n                                 "orig_path": item["orig"], "face_score": item.get("score", ""),\n                                 "note": item.get("note", "")})\n\n\ndef read_manifest(bid):\n    path = manifest_path({"id": bid})\n    if not path.is_file():\n        return {}\n    with path.open(encoding="utf-8") as fh:\n        return {row["frame"]: row for row in csv.DictReader(fh)}\n\n\n# ------------------------------------------------------------------- FaceBoxes фильтр\n\ndef make_detector():\n    sys.path.insert(0, str(CODE / "src"))\n    sys.path.insert(0, str(CODE / "src/pixel3dmm/preprocessing/PIPNet/FaceBoxesV2"))\n    import torch\n    from pixel3dmm.preprocessing.PIPNet.FaceBoxesV2.faceboxes_detector import FaceBoxesDetector\n    weights = PIPNET / "FaceBoxesV2/weights/FaceBoxesV2.pth"\n    return FaceBoxesDetector("FaceBoxes", str(weights), True, torch.device("cuda"))\n\n\ndef face_score(detector, path):\n    import cv2\n    image = cv2.imread(str(path))\n    if image is None or image.ndim < 3:\n        return -1.0\n    detections, _ = detector.detect(image, 0.6, 1)\n    faces = sorted([d for d in detections if d[0] == "face"], key=lambda x: -float(x[1]))\n    return float(faces[0][1]) if faces else 0.0\n\n\ndef filter_faces(batches, cfg):\n    """Порог 0.75 из upstream: кадр ниже -> ValueError рвёт весь батч. Отсекаем заранее."""\n    try:\n        detector = make_detector()\n    except Exception as error:\n        say(f"  FaceBoxes недоступен ({error}) — фильтр пропущен, плохие кадры поймает цикл")\n        return batches, []\n    dropped = []\n    for b in batches:\n        if b["kind"] == "video":\n            continue\n        keep = []\n        for i, item in enumerate(b["frames"]):\n            score = face_score(detector, Path(b["target"]) / f"{i:05d}.jpg")\n            item["score"] = round(score, 3)\n            if score >= float(cfg["face_min_score"]):\n                keep.append(i)\n            else:\n                dropped.append({"batch": b["id"], "photo": item["orig"], "score": item["score"]})\n        stage_photos(b, keep)\n    return batches, dropped\n\n\n# ----------------------------------------------------------------------- проверка шагов\n\ndef count_files(folder, suffix=None):\n    folder = Path(folder)\n    if not folder.is_dir():\n        return 0\n    return sum(1 for p in folder.iterdir()\n               if p.is_file() and (suffix is None or p.name.endswith(suffix)))\n\n\ndef count_dirs(folder):\n    folder = Path(folder)\n    return sum(1 for p in folder.iterdir() if p.is_dir()) if folder.is_dir() else 0\n\n\ndef nonempty_file(path):\n    return path.is_file() and path.stat().st_size > 0\n\n\ndef _dir_size(folder):\n    folder = Path(folder)\n    return sum(p.stat().st_size for p in folder.rglob("*") if p.is_file()) if folder.is_dir() else 0\n\n\ndef resolve_actor_dir(bid):\n    cands = [d for d in TRACK.glob(f"{bid}_nV*") if d.is_dir()]\n    return max(cands, key=lambda d: d.stat().st_mtime) if cands else None\n\n\ndef preproc_missing(bid, n):\n    """Кадры без обязательных артефактов предобработки. network_inference молча глотает\n    ошибки по кадрам (try/except pass), поэтому rc=0 ничего не гарантирует."""\n    base = PREPRO / bid\n    import numpy as np\n    mica_entries = list((base / "mica").iterdir()) if (base / "mica").is_dir() else []\n    mica_ok = bool(mica_entries)\n    for entry in mica_entries:\n        try:\n            identity = np.load(entry / "identity.npy", allow_pickle=False).squeeze()\n            mica_ok = mica_ok and identity.shape == (300,) and bool(np.isfinite(identity).all())\n        except (OSError, ValueError):\n            mica_ok = False\n    stages = {\n        "cropped": [i for i in range(n) if not nonempty_file(base / "cropped" / f"{i:05d}.jpg")],\n        "seg_og": [i for i in range(n) if not nonempty_file(base / "seg_og" / f"{i:05d}.png")],\n        "PIPnet_landmarks": [i for i in range(n) if not nonempty_file(base / "PIPnet_landmarks" / f"{i:05d}.npy")],\n        "normals": [i for i in range(n) if not nonempty_file(base / "p3dmm/normals" / f"{i:05d}.png")],\n        "uv_map": [i for i in range(n) if not nonempty_file(base / "p3dmm/uv_map" / f"{i:05d}.png")],\n        # read_data() берёт среднее по mica/*.npy: пустая папка = падение трекинга\n        "mica": [] if mica_ok else [0],\n    }\n    return stages\n\n\ndef track_missing(actor_dir, n):\n    out = {}\n    out["mesh"] = [i for i in range(n) if not nonempty_file(actor_dir / "mesh" / f"{i:05d}.ply")]\n    out["checkpoint"] = [i for i in range(n) if not nonempty_file(actor_dir / "checkpoint" / f"{i:05d}.frame")]\n    if not nonempty_file(actor_dir / "result.mp4"):\n        out["result.mp4"] = list(range(n))\n    if not nonempty_file(actor_dir / "mesh" / "canonical.ply"):\n        out["canonical.ply"] = list(range(n))\n    return {k: v for k, v in out.items() if v}\n\n\ndef flame2023_available():\n    path = FLAME_DATA / FLAME2023_PKL\n    return path.is_file() and path.stat().st_size > 0\n\n\ndef resolve_flame2023(cfg):\n    if not flame2023_available():\n        return False\n    return True\n\n\ndef track_args(cfg, n, use2023):\n    if n <= 1:\n        return (f"iters={int(cfg[\'single_iters\'])} batch_size=1 save_landmarks=True "\n                f"use_flame2023={\'True\' if use2023 else \'False\'}")\n    joint = min(int(cfg["joint_iters_max"]),\n                max(int(cfg["joint_iters_min"]), int(cfg["joint_iters_per_frame"]) * n))\n    return (" ".join([\n        f"iters={int(cfg[\'stage_iters\'])}", f"global_iters={joint}",\n        f"batch_size={min(int(cfg[\'batch_size_cap\']), n)}",\n        "include_neck=False", "w_exp=0.1", "use_mouth_lmk=False", "w_shape=0.01",\n        "w_shape_general=0.001", "normal_super=2000.0", "sil_super=1000.0",\n        "ignore_mica=True", "is_discontinuous=True", "save_landmarks=True",\n        f"use_flame2023={\'True\' if use2023 else \'False\'}"]))\n\n\n# --------------------------------------------------------------------------- прогон\n\ndef preproc_commands(cfg, bid, target):\n    """Команды предобработки батча: кроп+ландмарки -> MICA (и arcface-слепки) -> сегментация -> нормали -> uv.\n\n    Первые три пишут в PREPRO/<bid>, последние две — p3dmm/normals и p3dmm/uv_map; порядок важен\n    (сегментация берёт cropped, сети — тоже), поэтому он один и тот же и в joint, и в photo-режиме."""\n    mica = CODE / "src/pixel3dmm/preprocessing/MICA"\n    return (\n        (["python", str(CODE / "scripts/run_cropping.py"), "--video_or_images_path", str(target)],\n         CODE / "scripts"),\n        (["python", str(mica / "demo.py"), "-video_name", bid, "-a", str(PREPRO / bid / "arcface"),\n          "-m", str(mica / "data/pretrained/mica.tar")], mica),\n        (["python", str(CODE / "scripts/run_facer_segmentation.py"), "--video_name", bid], CODE / "scripts"),\n        (["python", "scripts/network_inference.py", "model.prediction_type=normals", f"video_name={bid}"], CODE),\n        (["python", "scripts/network_inference.py", "model.prediction_type=uv_map", f"video_name={bid}"], CODE),\n    )\n\n\ndef process_batch(cfg, b, use2023):\n    t0 = time.time()\n    bid = b["id"]\n    log = LOGS / f"{bid}.log"\n    log.unlink(missing_ok=True)\n    args = ""\n    b.pop("n_done", None)\n    if b["kind"] == "photos" and b["n"] > MAX_TRACK_FRAMES:\n        return "fail", f"больше {MAX_TRACK_FRAMES} кадров — разбей папку на отдельные батчи", args, 0.0\n    attempts = max(1, int(cfg["attempts"]))\n    timeout = float(cfg["batch_timeout_min"])\n    deadline = t0 + timeout * 60 if timeout > 0 else None\n\n    def run_step(cmd, cwd=None):\n        remaining = (deadline - time.time()) / 60 if deadline is not None else None\n        if remaining is not None and remaining <= 0:\n            return -9, f"таймаут батча {timeout} мин"\n        return run_cmd(cmd, log, timeout_min=remaining, cwd=cwd)\n\n    for attempt in range(attempts):\n        shutil.rmtree(PREPRO / bid, ignore_errors=True)\n        preprocessing = preproc_commands(cfg, bid, b["target"])[:3]\n        for cmd, cwd in preprocessing:\n            rc, why = run_step(cmd, cwd=cwd)\n            if rc:\n                return "fail", f"preprocessing ({Path(cmd[1]).name}): {why}; {tail(log, 12)}", args, time.time() - t0\n        produced = count_files(PREPRO / bid / "cropped", ".jpg")\n        # ожидаемое число кадров: для фото = сколько застили, для видео = сколько нарезали\n        expected = count_files(PREPRO / bid / "rgb", ".jpg") if b["kind"] == "video" else b["n"]\n        if b["kind"] == "video":\n            b["n_real"] = expected\n            b["n"] = expected\n            write_manifest(b)\n        if produced == 0 or expected == 0:\n            return "fail", "preprocessing: ни одного кадра в cropped", args, time.time() - t0\n        holes = preproc_missing(bid, expected)\n        missing = sorted({i for key in ("cropped", "seg_og", "PIPnet_landmarks") for i in holes[key]})\n        if missing and attempt + 1 < attempts:\n            keep = [i for i in range(b["n"]) if i not in missing] if b["kind"] == "photos" else []\n            if not keep:\n                return "fail", f"не прошли предобработку кадры {missing[:10]}", args, time.time() - t0\n            say(f"  {bid}: выкидываем кадры {missing[:12]} и повторяем предобработку")\n            stage_photos(b, keep)\n            expected = b["n"]\n            continue\n        if missing:\n            return "fail", f"не прошли предобработку кадры {missing[:10]}", args, time.time() - t0\n        if holes["mica"]:\n            return "fail", "предобработка не сделала MICA (папка mica пуста) — трекинг на ней упадёт", args, time.time() - t0\n        break\n\n    if expected > MAX_TRACK_FRAMES:\n        return "fail", f"видео содержит {expected} кадров; разбей на части до {MAX_TRACK_FRAMES}", args, time.time() - t0\n\n    for cmd, cwd in preproc_commands(cfg, bid, b["target"])[3:]:\n        rc, why = run_step(cmd, cwd=cwd)\n        if rc:\n            return "fail", f"{cmd[2].split(\'=\')[1]}: {why}; {tail(log, 12)}", args, time.time() - t0\n    holes = {k: v for k, v in preproc_missing(bid, expected).items() if v and k in ("normals", "uv_map")}\n    if holes:\n        return "fail", "сеть не сгенерировала часть кадров: " + ", ".join(f"{k}:{len(v)}" for k, v in holes.items()), args, time.time() - t0\n\n    args = track_args(cfg, expected, use2023)\n    b["track_args"] = args\n    b["tracking_config"] = effective_config(cfg, b, expected, use2023)\n    for old_actor in TRACK.glob(f"{bid}_nV*"):\n        if old_actor.is_dir():\n            shutil.rmtree(old_actor)\n    rc, why = run_step(["python", "scripts/track.py", f"video_name={bid}"] + args.split())\n    actor = resolve_actor_dir(bid)\n    if rc or actor is None:\n        return "fail", f"track: {why or \'нет выходного каталога\'}; {tail(log, 20)}", args, time.time() - t0\n    n_track = expected\n    holes = track_missing(actor, n_track)\n    if holes:\n        return "fail", "трекинг не дописал: " + ", ".join(f"{k}:{len(v)}" for k, v in holes.items()), args, time.time() - t0\n    try:\n        extract_params(actor, n_track)\n    except Exception as error:\n        return "fail", f"повреждён checkpoint: {type(error).__name__}: {error}", args, time.time() - t0\n    b["n_done"] = n_track\n    if not cfg.get("keep_preproc", True):\n        shutil.rmtree(PREPRO / bid, ignore_errors=True)\n    return "ok", "", args, time.time() - t0\n\n\n# ---------------------------------------------------------------- извлечение параметров\n\nPARAM_KEYS = ("exp", "shape", "eyes", "eyelids", "jaw", "neck", "R", "R_rotation_matrix", "t")\nEXTRA_KEYS = ("joint_transforms", "img_size", "frame_id", "global_step")\n\n\ndef _as_batch(value):\n    """Значение из .frame -> (1, ...): размерности матриц (R_base 3x3, joint_transforms) сохраняем."""\n    import numpy as np\n    if hasattr(value, "detach"):\n        value = value.detach().cpu().numpy()\n    arr = np.asarray(value)\n    if arr.ndim == 0:\n        return arr.reshape(1, 1)\n    if arr.ndim == 1:\n        return arr.reshape(1, -1)\n    if arr.shape[0] == 1:\n        return arr\n    return arr[None]\n\n\ndef frame_params(actor_dir, key):\n    """Один checkpoint кадра -> (строка таблицы, {имя: массив (1, ...)}) — общий код joint и photo."""\n    import numpy as np\n    import torch\n    path = Path(actor_dir) / "checkpoint" / f"{key}.frame"\n    d = torch.load(path, map_location="cpu", weights_only=False)  # файл написал наш же трекинг\n    d["flame"] = {name: _as_batch(value) for name, value in d["flame"].items()}\n    d["camera"] = {name: _as_batch(value) for name, value in d["camera"].items()}\n    one = {name: _as_batch(d["flame"][name]) for name in PARAM_KEYS}\n    for name, value in d["camera"].items():\n        one[name] = _as_batch(value)\n    for name in EXTRA_KEYS:\n        if name in d:\n            one[name] = _as_batch(d[name])\n    jaw = np.asarray(d["flame"]["jaw"]).ravel()\n    exp = np.asarray(d["flame"]["exp"]).ravel()\n    pose = pose_of_payload({"flame": {k: one[k] for k in ("R", "R_rotation_matrix") if k in one},\n                            "camera": {k: one[k] for k in ("R_base_0",) if k in one}})\n    row = {\n        "frame": key,\n        "mesh_ply": f"mesh/{key}.ply",\n        "frame_file": f"checkpoint/{key}.frame",\n        "jaw_norm": round(float(np.linalg.norm(jaw)), 5),\n        "exp_norm": round(float(np.linalg.norm(exp)), 5),\n        "shape_norm": round(float(np.linalg.norm(np.asarray(d["flame"]["shape"]).ravel())), 5),\n        "t": ";".join(f"{x:.5f}" for x in np.asarray(d["flame"]["t"]).ravel()[:3]),\n        "fl": round(float(np.asarray(d["camera"]["fl"]).ravel()[0]), 4),\n        "pp": ";".join(f"{x:.3f}" for x in np.asarray(d["camera"]["pp"]).ravel()[:2]),\n        # ракурс кадра по матрице поворота: тот же источник, что и в params.json, только плоский для csv\n        "yaw_deg": pose.get("yaw_deg", ""), "pitch_deg": pose.get("pitch_deg", ""),\n        "roll_deg": pose.get("roll_deg", ""), "view_bin": pose.get("bin") or "",\n    }\n    return row, one\n\n\ndef extract_params(actor_dir, n):\n    """Все FLAME-параметры и камера по кадрам -> (строки frames.csv, массивы params.npz)."""\n    import numpy as np\n    per, rows = {}, []\n    for i in range(n):\n        row, one = frame_params(actor_dir, f"{i:05d}")\n        rows.append(row)\n        for name, value in one.items():\n            per.setdefault(name, []).append(value)\n    arrays = {}\n    for name, values in per.items():\n        try:\n            arrays[name] = np.concatenate(values, axis=0)\n        except ValueError:\n            say(f"  параметр {name} разной размерности — в params.npz не кладём")\n    return rows, arrays\n\n\n# -------------------------------------------------------------------- ракурс (поза) кадра\n\n_AXES_CACHE = {}\n\n\ndef unit_vector(v):\n    """Единичный вектор; нулевой оставляем нулевым — вырожденную геометрию ловит вызывающий код."""\n    import numpy as np\n    arr = np.asarray(v, dtype=np.float64)\n    norm = float(np.linalg.norm(arr))\n    return arr / norm if norm > 1e-12 else arr\n\n\ndef yaw_bin(yaw):\n    """Имя бина по VIEW_BINS: нижняя граница включительно, верхняя — нет (кроме последнего бина)."""\n    import math\n    if yaw is None:\n        return None\n    yaw = float(yaw)\n    if not math.isfinite(yaw):\n        return None\n    for index, (name, lower, upper, _) in enumerate(VIEW_BINS):\n        if lower <= yaw < upper or (index == len(VIEW_BINS) - 1 and yaw == upper):\n            return name\n    return None\n\n\ndef rotation_6d_to_matrix(vec):\n    """6D-поворот Zhou et al. -> 3x3, как pytorch3d.transforms.rotation_6d_to_matrix.\n\n    Запасной путь: трекер пишет и готовую R_rotation_matrix, но в старых чекпойнтах её может не быть.\n    От pytorch3d отличаемся только в вырожденном входе (там нулевой второй вектор -> нечисловая\n    матрица, здесь — любая перпендикулярная плоскость), потому что ронять экспорт фото из-за этого нельзя."""\n    import numpy as np\n    v = np.asarray(vec, dtype=np.float64).ravel()\n    if v.size != 6:\n        raise ValueError(f"ожидалось 6 чисел поворота, получено {v.size}")\n    a1, a2 = v[:3], v[3:6]\n    b1 = unit_vector(a1)\n    if not np.any(b1):\n        raise ValueError("вырожденный 6D-поворот: нулевой первый вектор")\n    b2 = unit_vector(a2 - b1 * float(np.dot(b1, a2)))\n    if not np.any(b2):\n        helper = np.array([0.0, 0.0, 1.0]) if abs(float(b1[2])) < 0.9 else np.array([1.0, 0.0, 0.0])\n        b2 = unit_vector(helper - b1 * float(np.dot(b1, helper)))\n    return np.stack([b1, b2, np.cross(b1, b2)], axis=0)\n\n\ndef flame2020_dir():\n    """generic_model.pkl + landmark_embedding.npy: те же файлы, что проверяет preflight по REQUIRED_FLAME2020."""\n    return Path(FLAME_DATA) / "FLAME2020"\n\n\ndef landmark_embedding_axes(flame_dir):\n    """Оси лица по шаблону FLAME2020: те же 68 вшитых ландмарок и та же построка, что в\n    pixel3dmm_analysis.build_reference (origin между глазами, lateral по скулам, up к макушке,\n    front — нормаль к этой плоскости, развёрнутая в сторону носа). Только чистый CPU: pickle + numpy."""\n    import numpy as np\n    import pickle\n    flame_dir = Path(flame_dir)\n    with (flame_dir / "generic_model.pkl").open("rb") as stream:\n        model = pickle.load(stream, encoding="latin1")\n    embedding = np.load(flame_dir / "landmark_embedding.npy", allow_pickle=True, encoding="latin1").item()\n    face_ids = np.concatenate([np.asarray(embedding["dynamic_lmk_faces_idx"])[0],\n                               np.asarray(embedding["static_lmk_faces_idx"])])\n    bary = np.concatenate([np.asarray(embedding["dynamic_lmk_bary_coords"])[0],\n                          np.asarray(embedding["static_lmk_bary_coords"])])\n    if len(face_ids) != 68 or bary.shape != (68, 3):\n        raise ValueError("неожиданная топология FLAME landmark embedding")\n    vertices = np.asarray(model["v_template"], dtype=np.float64)\n    triangles = np.asarray(model["f"], dtype=int)\n    lm = np.sum(vertices[triangles[face_ids.astype(int)]] * bary.astype(float)[..., None], axis=-2)\n    origin = (lm[21] + lm[22]) / 2\n    lateral = unit_vector(lm[45] - lm[36])\n    up = unit_vector(origin - lm[8])\n    up = unit_vector(up - lateral * float(np.dot(up, lateral)))\n    front = unit_vector(np.cross(lateral, up))\n    if float(np.dot(lm[30] - origin, front)) < 0:\n        front = -front\n    axes = np.column_stack([lateral, up, front])\n    if not np.isfinite(axes).all() or abs(float(np.linalg.det(axes))) < 0.5:\n        raise ValueError("вырожденная референсная геометрия FLAME")\n    return axes\n\n\ndef reference_axes(flame_dir=None):\n    """Кэширует результат: generic_model.pkl весит 53 МБ, перечитывать его на каждое фото нельзя.\n\n    Битый файл кэшируем как ошибку (пересчитывать бесполезно), а отсутствие файла — нет: веса\n    могут докачать следующей ячейкой, и в том же процессе оси должны появиться."""\n    key = str(flame_dir or flame2020_dir())\n    cached = _AXES_CACHE.get(key)\n    if cached is None:\n        try:\n            cached = landmark_embedding_axes(Path(key))\n        except Exception as error:\n            if not (Path(key) / "generic_model.pkl").is_file():\n                raise FileNotFoundError(f"нет {Path(key) / \'generic_model.pkl\'} — выполни ячейку «4. Веса»")\n            cached = error\n        _AXES_CACHE[key] = cached\n    if isinstance(cached, BaseException):\n        raise cached\n    return cached\n\n\ndef pose_from_camera(rotation, axes):\n    """yaw/pitch/roll по повёрнутым осям лица — дословно та же формула, что в analysis.pose_from_camera."""\n    import math\n    import numpy as np\n    lateral, up, front = (np.asarray(rotation, dtype=np.float64)\n                          @ np.asarray(axes, dtype=np.float64)).T\n    yaw = math.degrees(math.atan2(front[0], front[2]))\n    pitch = math.degrees(math.atan2(front[1], math.hypot(front[0], front[2])))\n    roll = math.degrees(math.atan2(lateral[1], lateral[0]))\n    return {"yaw_deg": yaw, "pitch_deg": pitch, "roll_deg": roll, "convention": POSE_CONVENTION}\n\n\ndef pose_of_payload(payload):\n    """Поза одного кадра: R_total = R_base_0 @ R (как в разборе корпуса), метка — по девяти бинам.\n\n    Берётся из того, что уже лежит в params.json (flame.R / flame.R_rotation_matrix + camera.R_base_0),\n    поэтому этим же путём можно дополнить и старые папки фото, не трогая torch и чекпойнт.\n    Поза не двигает фото между папками: разметка корпуса остаётся основным ярлыком, это только измерение."""\n    import numpy as np\n    try:\n        axes = reference_axes()\n    except Exception as error:\n        return {"error": f"оси FLAME2020 не получаются: {type(error).__name__}: {error}"}\n    flame = payload.get("flame") or {}\n    camera = payload.get("camera") or {}\n    rmat, r6, base = flame.get("R_rotation_matrix"), flame.get("R"), camera.get("R_base_0")\n    if base is None or (rmat is None and r6 is None):\n        return {"error": "в params.json нет camera.R_base_0 или поворота головы"}\n    try:\n        base = np.asarray(base, dtype=np.float64).reshape(3, 3)\n        if rmat is not None:\n            rot = np.asarray(rmat, dtype=np.float64).reshape(3, 3)\n            source = "camera.R_base_0 @ flame.R_rotation_matrix"\n        else:\n            rot = rotation_6d_to_matrix(r6)\n            source = "camera.R_base_0 @ rotation_6d_to_matrix(flame.R)"\n        if not (np.isfinite(base).all() and np.isfinite(rot).all()):\n            raise ValueError("не числа в матрицах поворота")\n    except Exception as error:\n        return {"error": f"матрица поворота кадра не читается: {type(error).__name__}: {error}"}\n    pose = pose_from_camera(base @ rot, axes)\n    for key in ("yaw_deg", "pitch_deg", "roll_deg"):\n        pose[key] = round(float(pose[key]), 2)\n    pose["bin"] = yaw_bin(pose["yaw_deg"])\n    pose["source"] = source\n    pose["reference"] = "FLAME2020 v_template + landmark_embedding (68 вшитых точек), оси analysis.build_reference"\n    return pose\n\n\ndef effective_config(cfg, b, n, use2023):\n    """Upstream track.py не сохраняет конфиг запуска — сохраняем сами: tracking.yaml + наши CLI-аргументы.\n    Иначе потом невозможно воспроизвести, что реально крутилось."""\n    args = b.get("track_args", track_args(cfg, n, use2023)).split()\n    try:\n        from omegaconf import OmegaConf\n        base = OmegaConf.load(CODE / "configs" / "tracking.yaml")\n        merged = OmegaConf.merge(base, OmegaConf.from_cli([f"video_name={b[\'id\']}"] + args))\n        return OmegaConf.to_container(merged, resolve=True)\n    except Exception as error:\n        return {"error": f"{type(error).__name__}: {error}", "cli_args": args}\n\n\ndef batch_metrics(cfg, b, actor_dir, n, use2023, sec):\n    import torch\n    return {\n        "batch": b["id"], "subject": b["subject"], "angle": b["angle"], "kind": b["kind"],\n        "frames": n, "photos": [item["orig"] for item in b["frames"]],\n        "relpaths": [item["relpath"] for item in b["frames"]],\n        "track_args": b.get("track_args", track_args(cfg, n, use2023)), "use_flame2023": bool(use2023),\n        "tracking_config": b.get("tracking_config", effective_config(cfg, b, n, use2023)),\n        "paths": {"preprocessed": str(PREPRO / b["id"]), "actor": str(actor_dir),\n                  "input": cfg["input"], "output": cfg["output"]},\n        "actor_dir": actor_dir.name, "sec": round(sec, 1),\n        "torch": torch.__version__, "cuda": torch.version.cuda,\n        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "",\n        "canonical_ply": (Path(actor_dir) / "mesh/canonical.ply").is_file(),\n        "landmark_files": sorted(p.name for p in (Path(actor_dir) / "mesh").glob("*.npy"))[:3],\n        "preproc_kept": list(PREPROC_SUBSET) if cfg.get("keep_preproc", True) else [],\n        "dropped": b.get("dropped", []), "ts": int(time.time()),\n    }\n\n\n# ---------------------------------------------------------------------------- sync\n\ndef batches_root(cfg):\n    """Папки корпусов живут в OUTPUT/batches/, чтобы служебные _state/_index/_logs с ними не смешивались."""\n    return Path(cfg["output"]) / "batches"\n\n\ndef batch_dir(cfg, bid):\n    return batches_root(cfg) / bid\n\n\ndef photo_names(rows):\n    """кадр -> имя папки в Drive: по исходному файлу; при коллизии имён добавляем номер кадра."""\n    base = {}\n    for row in rows:\n        hint = row.get("orig_relpath") or row.get("orig_path") or ""\n        base[row["frame"]] = photo_folder(hint) if hint else f"frame_{row[\'frame\']}"\n    counts = {}\n    for name in base.values():\n        counts[name] = counts.get(name, 0) + 1\n    return {frame: (f"{name}__{frame}" if counts[name] > 1 else name) for frame, name in base.items()}\n\n\ndef overlay_frame(video, dst_jpg, index):\n    """Кадр из result.mp4 (меш поверх лица) в JPG — качество смотрят без проигрывателя."""\n    try:\n        import cv2\n    except ImportError:\n        return False\n    capture = cv2.VideoCapture(str(video))\n    try:\n        capture.set(cv2.CAP_PROP_POS_FRAMES, int(index))\n        ok, image = capture.read()\n    finally:\n        capture.release()\n    return bool(ok) and image is not None and bool(cv2.imwrite(str(dst_jpg), image))\n\n\ndef photo_payload(b, row, arrays, index):\n    """params.json одного фото: FLAME-параметры, камера и измеренный ракурс обычными списками + связь с исходником."""\n    import numpy as np\n    flame, camera = {}, {}\n    for name, values in arrays.items():\n        try:\n            value = values[index]\n        except IndexError:\n            continue\n        (flame if name in PARAM_KEYS else camera)[name] = np.asarray(value).tolist()\n    pose = pose_of_payload({"flame": flame, "camera": camera})\n    return {\n        "batch": b["id"], "subject": b["subject"], "angle": b["angle"], "frame": row["frame"],\n        "source": row.get("orig_path", ""), "relpath": row.get("orig_relpath", ""),\n        "face_score": row.get("face_score", ""), "note": row.get("note", ""),\n        "summary": {key: row.get(key, "") for key in\n                    ("jaw_norm", "exp_norm", "shape_norm", "t", "fl", "pp")},\n        "flame": flame, "camera": camera,\n        # метка ракурса — по матрице поворота и девяти бинам корпуса, а не по догадкам о 2D-асимметрии\n        "pose": pose,\n    }\n\n\ndef landmark_artifacts(actor, frame):\n    """Ландмарки кадра: upstream пишет их номером без ведущего нуля, а np.save к имени\n    \'.ply\' приклеивает \'.npy\' — поэтому ibug68_N.ply.npy. Падает в обе ветки экспорта.\n    zeropadding на всякий случай перебирается: в другом релизе трекера имена могут оказаться\n    и с нулями, а молча потерять ландмарки из папки фото было бы дороже, чем один is_file()."""\n    actor = Path(actor)\n    variants = [str(frame)]\n    if str(frame).isdigit():                       # upstream пишет номер без нулей, но с нулями тоже встречался\n        variants = [str(int(frame)), f"{int(frame):05d}"]\n    out = []\n    for label, pattern in (("landmarks_7.npy", "landmarks_{}_0.npy"),\n                           ("landmarks_68.npy", "ibug68_{}.ply.npy"),\n                           ("landmarks_eyes.npy", "now_{}.ply.npy")):\n        paths = [actor / "mesh" / pattern.format(v) for v in variants]\n        out.append((label, next((p for p in paths if nonempty_file(p)), paths[-1])))\n    return out\n\n\ndef export_photo_folders(cfg, b, actor, rows, arrays, index_of, tmp):\n    """photos/<имя исходного файла>/ — всё извлечённое по каждому фото лежит в своей папке."""\n    names = photo_names(rows)\n    pre = PREPRO / b["id"]\n    canonical = actor / "mesh" / "canonical.ply"\n    made = []\n    for row in rows:\n        frame = row["frame"]\n        index = index_of.get(frame, 0)\n        dst = tmp / "photos" / names[frame]\n        dst.mkdir(parents=True, exist_ok=True)\n        source = Path(row.get("orig_path") or "")\n        if source.is_file():\n            shutil.copy2(source, dst / f"source{source.suffix.lower() or \'.jpg\'}")\n        for src, name in ((actor / "mesh" / f"{frame}.ply", "mesh.ply"),\n                          (actor / "checkpoint" / f"{frame}.frame", "checkpoint.frame"),\n                          (pre / "cropped" / f"{frame}.jpg", "crop.jpg"),\n                          (pre / "seg_og" / f"{frame}.png", "segmentation.png"),\n                          (pre / "seg_og" / f"{frame}.jpg", "segmentation.jpg"),\n                          (pre / "PIPnet_landmarks" / f"{frame}.npy", "landmarks_pipnet.npy")):\n            if src.is_file():\n                shutil.copy2(src, dst / name)\n        for name, src in landmark_artifacts(actor, frame):\n            if src.is_file():\n                shutil.copy2(src, dst / name)\n        if canonical.is_file():\n            shutil.copy2(canonical, dst / "canonical.ply")\n        position = int(frame) if str(frame).isdigit() else index   # нумерация кадров в result.mp4 = номер кадра\n        overlay_frame(actor / "result.mp4", dst / "overlay.jpg", position)\n        payload = photo_payload(b, row, arrays, index)\n        payload["visibility"] = write_visibility(dst)\n        payload["files"] = sorted(p.name for p in dst.iterdir() if p.is_file())\n        (dst / "params.json").write_text(json.dumps(payload, ensure_ascii=False, indent=1), encoding="utf-8")\n        made.append(names[frame])\n    if canonical.is_file():\n        shutil.copy2(canonical, tmp / "canonical.ply")\n    lines = [f"# батч {b[\'id\']}: кадров {len(rows)}", "",\n             f"вход: {cfg[\'input\']}", "",\n             "frames.csv — таблица по кадрам, params.npz — все параметры разом, result.mp4 — "\n             "наложение меша" + (", artifacts.tar.gz — mesh/ checkpoint/ предобработка"\n                                 if cfg.get("keep_archive", True) else "") + ".",\n             "",]\n    for row in rows:\n        lines.append(f"- {row.get(\'orig_relpath\') or row.get(\'orig_path\') or row[\'frame\']} "\n                     f"-> photos/{names[row[\'frame\']]}/")\n    (tmp / "summary.md").write_text("\\n".join(lines) + "\\n", encoding="utf-8")\n    return made\n\n\ndef copy_file(src, dst):\n    """Один файл в выходную папку: через .part и с проверкой размера — полунки в Drive не остаётся."""\n    src, dst = Path(src), Path(dst)\n    dst.parent.mkdir(parents=True, exist_ok=True)\n    partial = dst.parent / (dst.name + ".part")\n    shutil.copy2(src, partial)\n    if partial.stat().st_size != src.stat().st_size:\n        raise OSError(f"неполная копия: {dst}")\n    partial.replace(dst)\n    return dst.stat().st_size\n\n\ndef write_json(payload, dst):\n    """JSON в выходную папку атомарно (через тот же .part)."""\n    dst = Path(dst)\n    tmp = dst.parent / (dst.name + ".part")\n    tmp.parent.mkdir(parents=True, exist_ok=True)\n    tmp.write_text(json.dumps(payload, ensure_ascii=False, indent=1), encoding="utf-8")\n    tmp.replace(dst)\n    return dst.stat().st_size\n\n\ndef sync_dir_to_drive(src, dst):\n    """Папка (photos/...) в Drive: копируем пофайлово через .part — обрезанных файлов на Drive быть не должно."""\n    files = [p for p in sorted(src.rglob("*")) if p.is_file()]\n    total = 0\n    dst.mkdir(parents=True, exist_ok=True)\n    for item in files:\n        total += copy_file(item, dst / item.relative_to(src))\n    return len(files), total\n\n\ndef write_archive(archive, actor_dir, pre, cfg):\n    """Не нужен в unit=photo (всё лежит по папкам фото), но остаётся для joint-режима по желанию."""\n    with tarfile.open(archive, "w:gz") as tar:\n        for folder, arc in ((Path(actor_dir) / "mesh", "mesh"), (Path(actor_dir) / "checkpoint", "checkpoint")):\n            if folder.is_dir():\n                tar.add(folder, arcname=arc)\n        for optional, arc in ((Path(actor_dir) / "config.yaml", "config.yaml"),   # upstream его не пишет\n                              (Path(actor_dir) / "logs", "logs")):                  # tensorboard: кривые потерь\n            if optional.is_file():\n                tar.add(optional, arcname=arc)\n            elif optional.is_dir() and _dir_size(optional) < 40_000_000:\n                tar.add(optional, arcname=arc)\n        for rel in (PREPROC_SUBSET if cfg.get("keep_preproc", True) else ()):\n            folder = Path(pre) / rel\n            if folder.is_dir():\n                tar.add(folder, arcname="preproc/" + rel.replace("/", "_"))\n    return archive\n\n\ndef sync_batch(cfg, b, actor_dir, n, sec, use2023, photos=None):\n    """Собирает результат батча в Drive: мелкие файлы отдельно + artifacts.tar.gz\n    с mesh/, checkpoint/ и предобработкой. photos=None -> по cfg["export"];\n    демо просит папки по фото всегда — иначе непонятно, где результат конкретного файла."""\n    out = Path(cfg["output"])\n    dst = batch_dir(cfg, b["id"])\n    tmp = WORK / "sync" / b["id"]\n    shutil.rmtree(tmp, ignore_errors=True)\n    tmp.mkdir(parents=True, exist_ok=True)\n    rows, arrays = extract_params(actor_dir, n)\n    index_of = {row["frame"]: i for i, row in enumerate(rows)}   # позиции в массивах params.npz\n    manifest = read_manifest(b["id"])\n    for row in rows:\n        row.update(manifest.get(row["frame"], {}))\n    if manifest:\n        rows.sort(key=lambda r: r["frame"])\n    with (tmp / "frames.csv").open("w", newline="", encoding="utf-8") as fh:\n        writer = csv.DictWriter(fh, fieldnames=FRAMES_CSV_COLS, extrasaction="ignore")\n        writer.writeheader()\n        writer.writerows(rows)\n    if manifest_path(b).is_file():\n        shutil.copy2(manifest_path(b), tmp / "manifest.csv")\n    if arrays:\n        import numpy as np\n        np.savez_compressed(tmp / "params.npz", **arrays)\n    shutil.copy2(actor_dir / "result.mp4", tmp / "result.mp4")\n    (tmp / "metrics.json").write_text(json.dumps(batch_metrics(cfg, b, actor_dir, n, use2023, sec),\n                                                 ensure_ascii=False, indent=1), encoding="utf-8")\n    pre = PREPRO / b["id"]\n    if cfg.get("keep_archive", True):\n        write_archive(tmp / "artifacts.tar.gz", actor_dir, pre, cfg)\n    want_photos = (str(cfg.get("export", "photos")) != "bundle") if photos is None else bool(photos)\n    if want_photos:\n        made = export_photo_folders(cfg, b, actor_dir, rows, arrays, index_of, tmp)\n        say(f"  {b[\'id\']}: папок по фото {len(made)}")\n    written = []\n    if cfg.get("sync_to_drive", True):\n        dst.mkdir(parents=True, exist_ok=True)\n        for src in sorted(tmp.iterdir(), key=lambda p: (p.name == "metrics.json", p.name)):\n            target = dst / src.name\n            if src.is_dir():\n                count, size = sync_dir_to_drive(src, target)\n                written.append((src.name + "/", size))\n                say(f"  {b[\'id\']}: {src.name}/ — {count} файлов в Drive")\n                continue\n            partial = dst / (src.name + ".part")\n            shutil.copy2(src, partial)\n            if partial.stat().st_size != src.stat().st_size:\n                raise OSError(f"неполная копия в Drive: {src.name}")\n            partial.replace(target)\n            written.append((src.name, target.stat().st_size))\n    if written:\n        say(f"  {b[\'id\']}: в Drive {len(written)} файлов")\n    log_src = LOGS / f"{b[\'id\']}.log"\n    if cfg.get("sync_to_drive", True) and log_src.is_file():\n        (out / "_logs").mkdir(parents=True, exist_ok=True)\n        shutil.copy2(log_src, out / "_logs" / log_src.name)\n    return written\n\n\n# --------------------------------------------------------------------------- лог\n\ndef state_dir(cfg):\n    path = Path(cfg["output"]) / "_state"\n    path.mkdir(parents=True, exist_ok=True)\n    return path\n\n\ndef read_log(cfg, name="batch_log.jsonl", local=None):\n    """Журнал прогона:Drive-копия + локальная. name/локальный путь разные для batch и photo."""\n    local = LOCAL_LOG if local is None else Path(local)\n    records = {}\n    for path in (Path(cfg["output"]) / "_state" / name, local):\n        if not path.is_file():\n            continue\n        for line in path.read_text(encoding="utf-8", errors="ignore").splitlines():\n            try:\n                rec = json.loads(line)\n            except ValueError:\n                continue\n            if not isinstance(rec, dict):\n                continue\n            if path == local and "output" not in rec:\n                continue\n            if rec.get("output", cfg["output"]) != cfg["output"] or rec.get("input", cfg["input"]) != cfg["input"]:\n                continue\n            uid = rec.get("key") or rec.get("id")     # unit=photo: ключ фото, иначе — id батча\n            if not uid:\n                continue\n            previous = records.get(uid, {})\n            if rec.get("updated_ns", 0) >= previous.get("updated_ns", 0):\n                records[uid] = rec\n    return [records[k] for k in sorted(records)]\n\n\ndef append_log(cfg, rec, name="batch_log.jsonl", local=None):\n    rec = dict(rec, input=cfg["input"], output=cfg["output"], updated_ns=time.time_ns())\n    local = LOCAL_LOG if local is None else Path(local)\n    local.parent.mkdir(parents=True, exist_ok=True)\n    with local.open("a", encoding="utf-8") as fh:\n        fh.write(json.dumps(rec, ensure_ascii=False) + "\\n")\n        fh.flush()\n    last_error = None\n    for _ in range(3):\n        try:\n            (state_dir(cfg) / name).write_text(\n                "".join(json.dumps(r, ensure_ascii=False) + "\\n" for r in read_log(cfg, name, local)),\n                encoding="utf-8")\n            log = LOGS / f"{rec.get(\'log\') or rec.get(\'id\')}.log"\n            if rec.get("status") == "fail" and log.is_file():\n                (Path(cfg["output"]) / "_logs").mkdir(parents=True, exist_ok=True)\n                shutil.copy2(log, Path(cfg["output"]) / "_logs" / log.name)\n            return rec\n        except OSError as error:\n            last_error = error\n            time.sleep(1)\n    raise RuntimeError(f"журнал не записан на Drive: {last_error}")\n\n\nBUNDLE_FILES = ("frames.csv", "manifest.csv", "params.npz", "metrics.json", "result.mp4", "artifacts.tar.gz")\n\n\ndef bundle_files(cfg):\n    """Что обязательно должно лежать в папке батча: архив нужен только если его явно просили."""\n    return tuple(name for name in BUNDLE_FILES\n                 if name != "artifacts.tar.gz" or cfg.get("keep_archive", True))\n\n\ndef bundle_missing(folder, synced=(), cfg=None):\n    folder = Path(folder)\n    expected = dict(synced)\n    names = bundle_files(cfg) if cfg is not None else BUNDLE_FILES\n    return [name for name in names\n            if not (folder / name).is_file() or (folder / name).stat().st_size == 0\n            or (name in expected and (folder / name).stat().st_size != expected[name])]\n\n\ndef save_plan(cfg, plan):\n    PLAN_FILE.parent.mkdir(parents=True, exist_ok=True)\n    text = json.dumps(plan, ensure_ascii=False, indent=1)\n    PLAN_FILE.write_text(text, encoding="utf-8")\n    try:\n        (state_dir(cfg) / "plan.json").write_text(text, encoding="utf-8")\n    except OSError as error:\n        say(f"план в Drive не записан: {error}")\n\n\n# ------------------------------------------------------------------ слепок среды (квота Colab)\n# После сброса рантайма /content пуст: сборка энва + скачивание весов занимают ~45–55 минут,\n# и всё это время идёт счётчик GPU-квоты. freeze упаковывает готовую среду в OUTPUT/_env,\n# warm распаковывает её обратно — минуты вместо десятков минут сборки.\nENV_ARCHIVE_DIR = "_env"\nENV_MARKER = ENV_ROOT / ".pixel3dmm_setup_v2"\nCACHE_ROOT = Path.home() / ".cache"\nINSIGHTFACE_ROOT = Path.home() / ".insightface"\nENV_CACHE_VERSION = 2\nPIXEL_REV = "fcd1fa973c7715b02a8948dfc679dff53cf85924"\nEXTRACT_ROOT = "/"              # куда распаковывать слепок; в тестах подменяется на tmp_path\n\n\ndef freeze_parts():\n    """(имя архива, что кладём, зачем) — пути читаются из глобалов, чтобы тесты могли их подменить."""\n    return (\n        ("p3dmm-env.tar.gz", (ENV_ROOT,), "микроокружение python 3.9 с torch cu118, pytorch3d и nvdiffrast в сборе"),\n        ("pixel3dmm.tar.gz", (CODE,), "клон pixel3dmm с патчами, FLAME и весами сетей"),\n        ("root-cache.tar.gz", (INSIGHTFACE_ROOT, CACHE_ROOT / "insightface", CACHE_ROOT / "torch_extensions",\n                               CACHE_ROOT / "huggingface", CACHE_ROOT / "torch"),\n         "веса insightface/FARL и JIT-сборка nvdiffrast"),\n    )\n\n\ndef env_dir(cfg, make=False):\n    path = Path(cfg["output"]) / ENV_ARCHIVE_DIR\n    if make:\n        path.mkdir(parents=True, exist_ok=True)\n    return path\n\n\ndef env_manifest_path(cfg):\n    return env_dir(cfg) / "manifest.json"\n\n\ndef read_env_manifest(cfg):\n    path = env_manifest_path(cfg)\n    if not path.is_file():\n        return {}\n    try:\n        data = json.loads(path.read_text(encoding="utf-8"))\n    except (OSError, ValueError):\n        return {}\n    return data if isinstance(data, dict) else {}\n\n\ndef archive_prefixes(sources):\n    """Префиксы путей внутри tar: распаковка идёт в указанный корень, поэтому чужие пути запрещены."""\n    return tuple(str(Path(p).absolute()).lstrip("/") for p in sources)\n\n\ndef cache_excluded(name):\n    parts = Path(name).parts\n    return (Path(name).name in {"token", "stored_tokens", "lock"}\n            or Path(name).name.endswith(".lock") or ".locks" in parts)\n\n\ndef safe_extract(archive, sources, target="/", missing_only=False):\n    """Validate the complete archive before extracting, including legacy conda/HF links."""\n    prefixes = archive_prefixes(sources)\n\n    def allowed(name):\n        return any(name == p or name.startswith(p + "/") for p in prefixes)\n\n    with tarfile.open(archive, "r:*") as tar:\n        members = []\n        for member in tar.getmembers():\n            norm = posixpath.normpath(member.name)\n            if (not allowed(norm) or member.name.startswith("/")\n                    or ".." in member.name.split("/")):\n                raise ValueError(f"{archive.name}: подозрительный путь в архиве: {member.name}")\n            if cache_excluded(norm):\n                continue\n            if not (member.isfile() or member.isdir() or member.issym() or member.islnk()):\n                raise ValueError(f"{archive.name}: подозрительный тип файла в архиве: {norm}")\n            if member.issym() or member.islnk():\n                link = (member.linkname.lstrip("/") if member.linkname.startswith("/")\n                        else posixpath.join(posixpath.dirname(norm), member.linkname)\n                        if member.issym() else member.linkname)\n                link = posixpath.normpath(link)\n                if not allowed(link) or cache_excluded(link):\n                    raise ValueError(f"{archive.name}: ссылка за пределы кеша: {norm} -> {member.linkname}")\n                member.linkname = (posixpath.relpath(link, posixpath.dirname(norm))\n                                   if member.issym() else link)\n            resolved = (Path(target) / norm).resolve()\n            roots = [Path(target).absolute() / p for p in prefixes]\n            if not any(resolved.is_relative_to(p) for p in roots):\n                raise ValueError(f"{archive.name}: существующая ссылка выводит путь из кеша: {norm}")\n            members.append(member)\n        links = {m.name for m in members if m.issym() or m.islnk()}\n        for member in members:\n            if any(str(p) in links for p in Path(member.name).parents):\n                raise ValueError(f"{archive.name}: запись внутрь ссылки: {member.name}")\n        if missing_only:\n            members = [m for m in members if m.isdir() or not (Path(target) / m.name).exists()]\n        kwargs = {"members": members, "path": str(target)}\n        if sys.version_info >= (3, 12):\n            kwargs["filter"] = "fully_trusted"\n        tar.extractall(**kwargs)\n    return len(members)\n\n\ndef snapshot_signature():\n    return {"version": ENV_CACHE_VERSION, "pixel_rev": PIXEL_REV, "hf_rev": HF_REV,\n            "cuda_arch": os.environ.get("TORCH_CUDA_ARCH_LIST", ""),\n            "flame_archives": {name: [p.stat().st_size, p.stat().st_mtime_ns]\n                               for name in ("FLAME2020.zip", "FLAME2023.zip")\n                               for p in (DRIVE_ROOT / name,) if p.is_file()}}\n\n\ndef snapshot_parts(cfg, manifest):\n    """The manifest cannot choose extraction destinations or arbitrary archive paths."""\n    allowed = {name: tuple(map(Path, sources)) for name, sources, _ in freeze_parts()}\n    out = []\n    for part in manifest.get("parts", []):\n        if not isinstance(part, dict):\n            raise ValueError("Некорректная часть слепка")\n        name = str(part.get("name", ""))\n        match = re.fullmatch(r"(p3dmm-env|pixel3dmm|root-cache)(?:-[0-9]+)?\\.tar\\.gz", name)\n        if not match:\n            raise ValueError(f"Некорректное имя архива: {name}")\n        kind = match[1] + ".tar.gz"\n        sources = [Path(p) for p in part.get("sources", [])]\n        if not sources or any(p not in allowed[kind] for p in sources):\n            raise ValueError(f"Недопустимые пути в слепке: {name}")\n        archive = env_dir(cfg) / name\n        if not archive.resolve().is_relative_to(env_dir(cfg).resolve()):\n            raise ValueError(f"Архив находится за пределами _env: {name}")\n        out.append((part, archive, sources, kind))\n    return out\n\n\ndef snapshot_ready(cfg, manifest):\n    if (manifest.get("invalid") or manifest.get("signature") != snapshot_signature()\n            or manifest.get("output") != cfg["output"]):\n        return False\n    try:\n        parts = snapshot_parts(cfg, manifest)\n        return ({kind for _, _, _, kind in parts} == {p[0] for p in freeze_parts()}\n                and all(archive.is_file() and archive.stat().st_size == part.get("bytes")\n                        and re.fullmatch(r"[a-f0-9]{64}", str(part.get("sha256", "")))\n                        for part, archive, _, _ in parts))\n    except (OSError, ValueError, TypeError):\n        return False\n\n\ndef dir_size_gb(folder):\n    total = 0\n    for root, dirs, files in os.walk(folder):\n        for name in files:\n            try:\n                total += os.path.getsize(Path(root) / name)\n            except OSError:\n                pass\n    return total / 1e9\n\n\ndef free_gb_at(path):\n    """Свободно ГБ на устройстве, куда пишем. Для fuse-маунта Drive обычно показывает свободное /content,\n    поэтому это только страховка от явного переполнения, а не квота Google Drive."""\n    try:\n        return shutil.disk_usage(path).free / 1e9\n    except OSError:\n        return 0.0\n\n\ndef session_elapsed_hours():\n    """Сколько живёт текущий рантайм: /proc/uptime — время контейнера, не ноутбука."""\n    try:\n        return float(Path("/proc/uptime").read_text(encoding="utf-8").split()[0]) / 3600.0\n    except (OSError, IndexError, ValueError):\n        return 0.0\n\n\nGUARD_MARGIN_S = 900          # запас на запись последнего батча в Drive перед принудительным отключением\n\n\ndef session_deadline(cfg, now=None):\n    """До какого момента можно считать: минимум из бюджета ячейки и остатка жизни рантайма Colab.\n\n    Бесплатная среда отключается принудительно (~12–13 ч), и обрыв посреди батча означал бы\n    потерю всего времени, потраченного на него, поэтому цикл останавливается сам чуть раньше."""\n    now = time.time() if now is None else now\n    limits = []\n    budget = float(cfg.get("session_max_hours", 0) or 0) * 3600\n    hard = float(cfg.get("session_hard_limit_hours", 0) or 0) * 3600\n    if budget > 0:\n        limits.append(budget)\n    if hard > 0:\n        limits.append(max(0.0, hard - session_elapsed_hours() * 3600 - GUARD_MARGIN_S))\n    return now + (min(limits) if limits else 365 * 86400.0)\n\n\ndef median_sec_per_frame(records):\n    """Медиана секунд на кадр по успешно посчитанным батчам; 0.0 если опыта ещё нет."""\n    vals = sorted(float(r["sec"]) / int(r["n"]) for r in records\n                  if r.get("status") == "ok" and int(r.get("n") or 0) > 0 and float(r.get("sec") or 0) > 0)\n    if not vals:\n        return 0.0\n    mid = len(vals) // 2\n    return vals[mid] if len(vals) % 2 else (vals[mid - 1] + vals[mid]) / 2.0\n\n\ndef weights_stamp():\n    """Файл-отметка проверенных весов (живёт в /content/pixel3dmm, то есть и в слепке среды тоже)."""\n    return CODE / ".weights_sha256.json"\n\n\n_VERIFIED_ASSETS = set()\n\n\ndef verified_asset(name, path):\n    """Reuse a SHA256 verification only for the same revision, size and nanosecond mtime."""\n    path = Path(path)\n    size, expected = HF_FILES[name]\n    if not path.is_file():\n        return False\n    stat = path.stat()\n    if stat.st_size != size:\n        return False\n    token = (expected, stat.st_dev, stat.st_ino, size, stat.st_mtime_ns)\n    try:\n        stamp = json.loads(weights_stamp().read_text(encoding="utf-8"))\n    except (OSError, ValueError):\n        stamp = {}\n    if not isinstance(stamp, dict) or stamp.get("revision") != HF_REV:\n        stamp = {"revision": HF_REV, "files": {}}\n    if not isinstance(stamp.get("files"), dict):\n        stamp["files"] = {}\n    files = stamp["files"]\n    row = {"sha256": expected, "size": size, "mtime_ns": stat.st_mtime_ns}\n    if files.get(str(path)) == row:\n        _VERIFIED_ASSETS.add(token)\n        return True\n    if token not in _VERIFIED_ASSETS:\n        progress(phase="Проверка SHA256", current=name)\n        if file_sha256(path) != expected:\n            return False\n        _VERIFIED_ASSETS.add(token)\n    files[str(path)] = row\n    write_json(stamp, weights_stamp())\n    return True\n\n\ndef check_weights():\n    """Проверяет веса по размеру и SHA256; повторное хеширование 4+ ГБ не нужно, если файл не менялся.\n\n    Отметка живёт в /content/pixel3dmm/.weights_sha256.json и поэтому попадает и в слепок среды: после\n    warm/preflight считает только то, что реально изменилось. Возвращает список (rel, размер, ок, вывод)."""\n    out = []\n    for rel, minimum in WEIGHTS:\n        path = CODE / rel\n        name = Path(rel).name\n        if not path.is_file():\n            out.append((rel, 0, False, "НЕТ"))\n            continue\n        size = path.stat().st_size\n        ok = verified_asset(name, path)\n        out.append((rel, size, ok, "ok" if ok else "НЕТ"))\n    return out\n\n\n# ------------------------------------------------------------------ стадии\n\ndef st_preflight(cfg):\n    import torch\n    say("=" * 72)\n    say("ПРЕДФЛАЙТ — готов ли рантайм к корпусу")\n    say("=" * 72)\n    problems = []\n    say(f"python   : {sys.executable}")\n    if not sys.executable.startswith(str(ENV_ROOT)):\n        problems.append("ядро пайплайна запускается не из /content/p3dmm-env")\n    say(f"torch    : {torch.__version__} | cuda: {torch.cuda.is_available()} | "\n        f"{torch.cuda.get_device_name(0) if torch.cuda.is_available() else \'БЕЗ GPU (Runtime -> Change runtime type -> T4)\'}")\n    if not torch.cuda.is_available():\n        problems.append("GPU недоступен — трекинг не пойдёт")\n    if not CODE.is_dir():\n        problems.append("нет /content/pixel3dmm — выполни ячейку «3. Окружение»")\n    needed = dict(WEIGHTS)\n    for rel, size, ok, flag in check_weights():\n        if not ok:\n            problems.append(f"{rel} — неверный размер/SHA256 ({size} байт, ожидается {needed[rel]}), выполни ячейку «4. Веса»")\n        say(f"вес      : {flag:<3} {rel} ({size / 1e6:.0f} МБ)")\n    for rel in REQUIRED_FLAME2020:\n        path = FLAME_DATA / "FLAME2020" / rel\n        if not path.is_file() or not path.stat().st_size:\n            problems.append(f"FLAME2020/{rel} нет — повтори ячейку «4. Веса»")\n        say(f"flame    : {\'ok\' if path.is_file() else \'НЕТ \':<3} {rel}")\n    use2023 = flame2023_available()\n    try:\n        use2023 = resolve_flame2023(cfg)\n        say(f"flame2023: {\'ok\' if use2023 else \'НЕТ \':<3} {FLAME2023_PKL} -> "\n            f"батчи пойдут с use_flame2023={\'True\' if use2023 else \'False\'}")\n    except ValueError as error:\n        problems.append(str(error))\n        say(f"flame2023: НЕТ {FLAME2023_PKL} (в CONFIG стоит use_flame2023=True)")\n    nms = list((PIPNET / "FaceBoxesV2/utils").rglob("*nms*.so"))\n    say(f"faceboxes: {\'ok\' if nms else \'НЕТ \':<3} {\', \'.join(p.name for p in nms) or \'не собран (ячейка 4)\'}")\n    if not nms:\n        problems.append("FaceBoxes не собран — выполни ячейку «4. Веса»")\n    for name, (size, digest) in HF_FILES.items():\n        if name.startswith("insightface/"):\n            path = Path.home() / ("." + name)\n            if not path.is_file() or path.stat().st_size != size:\n                problems.append(f"{name} отсутствует или недокачан — повтори ячейку «4. Веса»")\n    say("torch.compile -> " + patch_compile(bool(cfg.get("torch_compile"))))\n    say("трекинг одного кадра -> " + patch_single_frame_support())\n    try:\n        import nvdiffrast.torch as dr\n        dr.RasterizeCudaContext()\n        say("nvdiffrast: JIT-плагин собран ( gcc 11 из энва )")\n    except Exception as error:\n        problems.append(f"nvdiffrast не собирается: {error}")\n        say(f"nvdiffrast: FAIL {type(error).__name__}: {str(error)[:200]}")\n    try:\n        import chumpy, cv2, facer, insightface, pixel3dmm  # noqa: F401\n        from pytorch3d import _C  # noqa: F401\n        from pytorch3d.ops import knn_points\n        if torch.cuda.is_available():\n            points = torch.zeros((1, 1, 3), device="cuda")\n            knn_points(points, points, K=1)\n            make_detector()\n        say("импорты  : ok (chumpy cv2 facer insightface pixel3dmm pytorch3d)")\n    except Exception as error:\n        problems.append(f"импорт пакета: {error}")\n    say(f"диск     : свободно {free_gb()} ГБ на /content")\n    if free_gb() < float(cfg["min_free_gb"]):\n        problems.append(f"свободно меньше {cfg[\'min_free_gb\']} ГБ — освободи диск перед run")\n    root = None\n    try:\n        root = resolve_input(cfg["input"])\n        say(f"вход     : {root}")\n    except ValueError as error:\n        problems.append(str(error))\n    out = Path(cfg["output"])\n    out.mkdir(parents=True, exist_ok=True)\n    say(f"выход    : {out} | {\'доступен\' if out.is_dir() else \'НЕ ДОСТУПЕН — проверь монтирование Drive\'}")\n    say(f"режим    : source={cfg.get(\'source\', \'drive\')} | unit={cfg.get(\'unit\', \'folder\')} | "\n        f"папки батчей в {batches_root(cfg)} | "\n        + (f"на каждое фото: {batches_root(cfg)}/<папка>/photos/<имя файла>/ "\n           f"чанками по {cfg.get(\'photos_per_chunk\', 40)}"\n           if str(cfg.get(\'unit\', \'folder\')) == \'photo\' else\n           f"результат на фото: {\'папка по имени файла\' if str(cfg.get(\'export\', \'photos\')) != \'bundle\' else \'только архив\'}"))\n    if root is not None and root.is_dir():\n        photos = sum(1 for p in root.rglob("*") if is_photo(p))\n        say(f"фото     : {photos} в {root}")\n    manifest = read_env_manifest(cfg)\n    if manifest.get("parts"):\n        gb = sum(int(p.get("bytes", 0)) for p in manifest["parts"]) / 1e9\n        age = max(0.0, time.time() - float(manifest.get("ts", 0) or 0))\n        say(f"слепок   : {len(manifest[\'parts\'])} архива, {gb:.1f} ГБ, {human(age)} назад — "\n            f"перезапуск среды займёт минуты (ячейка 3 распакует сама)")\n    else:\n        say("слепок   : OUTPUT/_env пуст — после первой успешной сборки жми ячейку freeze, "\n            "и дальше каждый перезапуск среды будет в разы дешевле по квоте")\n    state_dir(cfg)\n    (state_dir(cfg) / "preflight.json").write_text(json.dumps(\n        {"problems": problems, "flame2023": bool(use2023), "free_gb": free_gb(),\n         "torch": torch.__version__, "ts": int(time.time())}, ensure_ascii=False, indent=1), encoding="utf-8")\n    say("-" * 72)\n    if problems:\n        say("ЧТО МЕШАЕТ:")\n        for item in problems:\n            say("  ! " + item)\n        return 1\n    say("ГОТОВО: можно гнать корпус (стадия plan -> run)")\n    return 0\n\n\ndef st_plan(cfg):\n    say("=" * 72)\n    say("ПЛАН — сканирование входа, фильтр лиц, staging, манифест")\n    say("=" * 72)\n    t0 = time.time()\n    batches = build_batches(cfg)\n    if cfg["face_filter"] and any(b["kind"] == "photos" for b in batches):\n        say("проверяю лица FaceBoxes (одна загрузка модели на всю выборку)...")\n        batches, dropped = filter_faces(batches, cfg)\n        if dropped:\n            (WORK / "face_dropped.json").write_text(json.dumps(dropped, ensure_ascii=False, indent=1), encoding="utf-8")\n            say(f"отброшено кадров без лица/с низким скором: {len(dropped)} (список в /content/work/face_dropped.json)")\n    batches = [b for b in batches if b["n"] or b["kind"] == "video"]\n    unit = str(cfg.get("unit", "folder"))\n    if unit == "photo":\n        batches = chunk_batches(batches, cfg)\n        say(f"unit=photo: предобработка чанками по <= {cfg.get(\'photos_per_chunk\', 40)} фото, "\n            f"трекинг — отдельно на каждое фото (свой canonical.ply)")\n    use2023 = resolve_flame2023(cfg)\n    total = sum(b["n"] for b in batches)\n    save_plan(cfg, {"batches": batches, "use_flame2023": use2023, "total_photos": total, "config": cfg})\n    undated = cfg.get("_undated_photos", [])\n    if undated:\n        write_json({"undated": undated, "count": len(undated)}, state_dir(cfg) / "undated.json")\n        say(f"Без корректной даты в имени: {len(undated)} фото. Они не анализируются.")\n        for name in undated[:20]:\n            say("  " + name)\n        if len(undated) > 20:\n            say(f"  ... ещё {len(undated) - 20}")\n    dated = cfg.get("_photo_count", total)\n    say(f"К анализу: {dated} фото по дате в имени; пропущено без даты: {len(undated)}")\n    first = batches[0]["frames"][0]["relpath"] if batches and batches[0]["frames"] else ""\n    if first:\n        write_json({"next_photo": {"relpath": first}, "last_completed": None,\n                    "updated": int(time.time())}, state_dir(cfg) / "resume.json")\n    say(f"{\'батч\':<26}{\'кадров\':>8}  {\'аргументы трекинга\'}")\n    for b in batches:\n        preview = (" ".join(photo_track_args(cfg, b["frames"][0].get("frame", "0"), use2023))\n                   if unit == "photo" and b["kind"] == "photos" else track_args(cfg, b["n"], use2023))\n        say(f"{b[\'id\']:<26}{b[\'n\']:>8}  {preview[:88]}")\n    say("-" * 72)\n    over = [b["id"] for b in batches if b["n"] > MAX_TRACK_FRAMES]\n    if over:\n        say(f"!! больше {MAX_TRACK_FRAMES} кадров — разбей папки перед run: "\n            + ", ".join(over))\n    say(f"батчей: {len(batches)} | кадров: {total} | flame2023: {use2023} | готовилось {human(time.time() - t0)}")\n    photo_log = read_log(cfg, "photos_log.jsonl", PHOTO_LOG) if unit == "photo" else []\n    per_frame = median_sec_per_frame(photo_log or read_log(cfg))\n    source = ("по журналу фото" if photo_log else "по журналу Drive") if per_frame else "оценка по умолчанию"\n    default_per = PHOTO_DEFAULT_S if unit == "photo" else 45.0\n    say(f"ориентир по времени: ~{total * (per_frame or default_per) / 3600:.1f} ч GPU "\n        f"({(per_frame or default_per):.0f} с/кадр, {source})")\n    say(f"свободно на /content: {free_gb()} ГБ")\n    return 1 if not batches or over else 0\n\n\ndef load_plan(cfg):\n    for path in (PLAN_FILE, Path(cfg["output"]) / "_state/plan.json"):\n        if path.is_file():\n            try:\n                data = json.loads(path.read_text(encoding="utf-8"))\n            except (OSError, ValueError):\n                continue\n            if not isinstance(data, dict) or not isinstance(data.get("batches"), list):\n                continue\n            saved = data.get("config", {})\n            if any(saved.get(key) != cfg.get(key) for key in ("input", "output", "source", "subjects",\n                                                              "max_batches", "face_filter", "face_min_score",\n                                                              "unit", "photos_per_chunk")):\n                continue\n            for b in data["batches"]:\n                if not re.fullmatch(r"[A-Za-z0-9_-]+", str(b.get("id", ""))) or b.get("kind") not in ("photos", "video"):\n                    raise ValueError("план содержит некорректный ID/тип батча — пересоздай plan")\n                b["target"] = str(STAGE / b["id"] if b["kind"] == "photos" else STAGE / (b["id"] + ".mp4"))\n            return data\n    return None\n\n\ndef st_test1(cfg):\n    say("=" * 72)\n    say("ТЕСТ 1 ФОТО — полный цикл на одном кадре (корпус не трогаем)")\n    say("=" * 72)\n    root = resolve_input(cfg["input"])\n    if root.is_file() and root.suffix.lower() == ".zip":\n        root = unpack_zip(root)\n    video_source = None\n    if root.is_file() and root.suffix.lower() in VIDEO_EXT:\n        import cv2\n        capture = cv2.VideoCapture(str(root))\n        try:\n            success, image = capture.read()\n        finally:\n            capture.release()\n        if not success:\n            say(f"видео не читается: {root}")\n            return 1\n        video_source = root\n        root = WORK / "test1_video_frame.jpg"\n        if not cv2.imwrite(str(root), image):\n            raise OSError("не удалось сохранить тестовый кадр видео")\n    photos = sorted(p for p in (root.rglob("*") if root.is_dir() else [root]) if is_photo(p))\n    if not photos:\n        say("фото не найдены — проверь INPUT")\n        return 1\n    src = photos[len(photos) // 2] if len(photos) > 1 else photos[0]\n    say(f"тестируем: {src}")\n    if str(cfg.get("unit", "folder")) == "photo":\n        return test1_photo(cfg, src, video_source)\n    b = {"id": "test1", "subject": "test", "angle": "one", "kind": "photos",\n          "folder": str(src.parent), "frames": [{"orig": str(src), "relpath": src.name}], "n": 1}\n    if video_source:\n        b["frames"][0]["relpath"] = video_source.name + "#frame=0"\n    stage_photos(b)\n    use2023 = resolve_flame2023(cfg)\n    status, err, args, sec = process_batch(cfg, b, use2023)\n    actor = resolve_actor_dir("test1")\n    say(f"test1 -> {status} за {human(sec)} | args: {args or \'(не дошли)\'}")\n    if status != "ok":\n        say(err)\n        say(tail(LOGS / "test1.log", 30))\n        return 1\n    n = b.get("n_done", b["n"])\n    rows, arrays = extract_params(actor, n)\n    say(f"артефакты: mesh {count_files(actor / \'mesh\', \'.ply\')} ply, "\n        f"{count_files(actor / \'checkpoint\', \'.frame\')} .frame, "\n        f"ландмарок {count_files(actor / \'mesh\', \'.npy\')} npy, "\n        f"canonical.ply {(actor / \'mesh/canonical.ply\').is_file()}, параметров {len(arrays)} ключей")\n    say("файлы батча: " + ", ".join(sorted(p.name for p in actor.iterdir())))\n    written = sync_batch(cfg, b, actor, n, sec, use2023)\n    append_log(cfg, {"id": b["id"], "status": "ok", "n": n, "sec": round(sec, 1),\n                     "args": args, "flame2023": use2023, "synced": written, "ts": int(time.time())})\n    say("ИТОГ: УСПЕХ — можно запускать plan + run")\n    return 0\n\n\ndef write_demo_preview(cfg, b, sec):\n    """Локальное превью + репорт: ячейка «показать результат» читает /content, а не Drive —\n    иначе на примонтированном Drive просмотр демо превращается в минуту ожидания."""\n    src = WORK / "sync" / b["id"] / "photos"\n    dst = WORK / "preview"\n    shutil.rmtree(dst, ignore_errors=True)\n    if src.is_dir():\n        shutil.copytree(src, dst)\n    entries = []\n    for folder in sorted(p for p in dst.iterdir() if p.is_dir()) if dst.is_dir() else []:\n        params = folder / "params.json"\n        payload = json.loads(params.read_text(encoding="utf-8")) if params.is_file() else {}\n        entries.append({"folder": folder.name, "frame": payload.get("frame", ""),\n                        "source": payload.get("source", ""), "relpath": payload.get("relpath", ""),\n                        "summary": payload.get("summary", {}),\n                        "flame_keys": sorted(payload.get("flame", {})),\n                        "files": sorted(p.name for p in folder.iterdir() if p.is_file())})\n    report = {"batch": b["id"], "status": "ok", "sec": round(sec, 1),\n              "drive": str(batch_dir(cfg, b["id"])), "preview": str(dst),\n              "bundle": str(WORK / "sync" / b["id"]),\n              "input": cfg["input"], "output": cfg["output"],\n              "photos": entries}\n    (WORK / "demo_report.json").write_text(json.dumps(report, ensure_ascii=False, indent=1), encoding="utf-8")\n    return report\n\n\ndef st_demo(cfg):\n    """ДЕМО — один полный цикл на первом фото из INPUT: preprocessing → сети → трекинг → папка в Drive.\n    Результат раскладывается по папкам с именами исходных фото (photos/<имя>/)."""\n    say("=" * 72)\n    say("ДЕМО — полный цикл на первом фото из INPUT")\n    say("=" * 72)\n    t0 = time.time()\n    try:\n        batches = build_batches(cfg, refresh=False)\n    except ValueError as error:\n        say(f"вход недоступен: {error}")\n        return 1\n    batches = [b for b in batches if b["kind"] == "photos" and b["n"]]\n    if not batches:\n        say(f"фото не найдены: поправь INPUT (сейчас это {cfg[\'input\']!r}) — нужна папка с фото на Drive")\n        return 1\n    b = batches[0]\n    limit = max(1, int(cfg.get("demo_photos", 1) or 1))\n    total_in_folder = int(b["n"])\n    if total_in_folder > limit:\n        say(f"в первой папке {total_in_folder} фото — демо берёт первые {limit}")\n        if str(cfg.get("unit", "folder")) == "photo":\n            say("   unit=photo: демо считает ровно эти фото, и run их пропустит — засчитано")\n        else:\n            say("   demo считает батч целиком (joint-трекинг), поэтому run пересчитает эту папку заново —")\n            say("   так что демо не «кусает» корпус, а проверяет цикл на малом числе кадров")\n        b["frames"] = b["frames"][:limit]\n        b["n"] = limit\n    say(f"батч {b[\'id\']}: " + ", ".join(Path(item["orig"]).name for item in b["frames"][:8]))\n    stage_photos(b)\n    use2023 = resolve_flame2023(cfg)\n    if str(cfg.get("unit", "folder")) == "photo":\n        return demo_photos(cfg, b, use2023)\n    status, err, args, sec = process_batch(cfg, b, use2023)\n    actor = resolve_actor_dir(b["id"])\n    say(f"демо -> {status} за {human(sec)} | args: {args or \'(не дошли)\'}")\n    if status != "ok":\n        say(err)\n        say(tail(LOGS / f"{b[\'id\']}.log", 30))\n        append_log(cfg, {"id": b["id"], "status": "fail", "n": b["n"], "sec": round(sec, 1), "args": args,\n                         "flame2023": use2023, "synced": [], "err": err[:500],\n                         "partial": b["n"] < total_in_folder, "n_total": total_in_folder,\n                         "ts": int(time.time())})\n        return 1\n    n = b.get("n_done", b["n"])\n    partial = n < total_in_folder          # демо считало часть папки: run обязан пересчитать её целиком\n    written = sync_batch(cfg, b, actor, n, sec, use2023, photos=True)\n    append_log(cfg, {"id": b["id"], "status": "ok", "n": n, "sec": round(sec, 1), "args": args,\n                     "flame2023": use2023, "synced": written, "partial": partial,\n                     "n_total": total_in_folder, "ts": int(time.time())})\n    report = write_demo_preview(cfg, b, time.time() - t0)\n    say("-" * 72)\n    say(f"папка результата в Drive: {report[\'drive\']}")\n    for entry in report["photos"]:\n        name = Path(entry["source"]).name if entry["source"] else f"кадр {entry[\'frame\']}"\n        say(f"  {name} -> photos/{entry[\'folder\']}/ [{\', \'.join(entry[\'files\']) or \'пусто\'}]")\n    say(f"превью для показа в ноутбуке: {report[\'preview\']}")\n    say("ИТОГ: цикл проходит — дальше жми plan + run по всей папке из INPUT")\n    return 0\n\n\ndef demo_photos(cfg, b, use2023):\n    """ДЕМО в unit=photo: те же независимые примерки, что и у корпуса; посчитанное засчитывается —\n    run потом пропустит эти папки по done.json, пересчёт не нужен."""\n    names = photo_names_map(b)\n    ok, fail, done_folders = 0, 0, []\n    for i, item in enumerate(b["frames"]):\n        name = names[item.get("frame", f"{i:05d}")]\n        status, err, sec, args, extra = fit_one_photo(cfg, b, item, name, use2023)\n        folder, files, shape = photo_folder_report(cfg, b, name)\n        say(f"  {Path(item[\'orig\']).name} -> {status} за {human(sec)} -> {folder}")\n        say(f"    файлы: {\', \'.join(files) or \'пусто\'}" + (f" | shape_norm {shape}" if shape else ""))\n        if status == "ok":\n            ok += 1\n            done_folders.append(str(folder))\n        else:\n            fail += 1\n            say(f"    ! {err[:240]}")\n    preview = WORK / "preview"\n    shutil.rmtree(preview, ignore_errors=True)\n    src = photos_root(cfg, b.get("out", b["id"]))\n    if src.is_dir():\n        shutil.copytree(src, preview)\n    report = {"batch": b["id"], "unit": "photo", "status": "ok" if not fail else "fail",\n              "drive": str(src), "preview": str(preview), "ok": ok, "fail": fail,\n              "folders": done_folders, "input": cfg["input"], "output": cfg["output"]}\n    write_json(report, WORK / "demo_report.json")\n    say("-" * 72)\n    say(f"превью для показа в ноутбуке: {preview}")\n    say("ИТОГ: цикл проходит — жми plan + run по всей папке, эти фото уже посчитаны" if not fail\n        else "ИТОГ: есть провалы — смотри log выше")\n    return 1 if fail else 0\n\n\ndef fit_one_photo(cfg, b, item, name, use2023):\n    """Один полный фото-цикл без плана и без журнала (демо и test1): предобработка + примерка + папка."""\n    frame = item["frame"]\n    if not preproc_ok(PREPRO / b["id"], frame):\n        preprocess_subset(cfg, b, [frame])\n    sync_preproc(cfg, b, frame, name)\n    return process_photo(cfg, b, item, name, use2023)\n\n\ndef photo_folder_report(cfg, b, name):\n    """Что легло в папку фото и чем этот каноникал отличается от чужого (shape-норма)."""\n    folder = photo_dir(cfg, b.get("out", b["id"]), name)\n    files = sorted(p.name for p in folder.iterdir() if p.is_file()) if folder.is_dir() else []\n    shape = ""\n    params = folder / "params.json"\n    if params.is_file():\n        try:\n            shape = str(json.loads(params.read_text(encoding="utf-8")).get("summary", {}).get("shape_norm", ""))\n        except (OSError, ValueError):\n            shape = ""\n    return folder, files, shape\n\n\ndef test1_photo(cfg, src, video_source):\n    """test1 в unit=photo — ровно тот цикл, что и для корпуса, но на одном снимке и без журнала."""\n    rel = video_source.name + "#frame=0" if video_source else src.name\n    b = {"id": "test1", "out": "test1", "subject": "test", "angle": "one", "kind": "photos",\n         "folder": str(src.parent), "frames": [{"orig": str(src), "relpath": rel}], "n": 1}\n    stage_photos(b)\n    use2023 = resolve_flame2023(cfg)\n    frame = b["frames"][0]["frame"]\n    name = photo_names_map(b)[frame]\n    status, err, sec, args, extra = fit_one_photo(cfg, b, b["frames"][0], name, use2023)\n    say(f"test1 -> {status} за {human(sec)} | args: {\' \'.join(args)}")\n    if status != "ok":\n        say(err)\n        say(tail(LOGS / f"test1__{frame}.log", 30))\n        return 1\n    folder, files, shape = photo_folder_report(cfg, b, name)\n    say(f"папка фото: {folder}")\n    say("файлы: " + ", ".join(files))\n    holes = [n for n in PHOTO_REQUIRED if not nonempty_file(folder / n)]\n    if holes:\n        say(f"не хватает: {holes}")\n        return 1\n    say(f"каноническая модель этого фото: shape_norm {shape} — она своя на каждый снимок")\n    say("ИТОГ: цикл проходит — дальше жми plan + run по корпусу")\n    return 0\n\n\ndef ensure_staged(b):\n    """После перезапуска рантайма /content пуст: раскладываем фото заново по плану.\n    Нумерация кадров детерминирована (тот же порядок b["frames"]), манифест перезаписывается."""\n    target = Path(b.get("target") or (STAGE / b["id"]))\n    if b["kind"] == "video":\n        if not target.is_file():\n            try:\n                stage_video(b, Path(b["folder"]))\n            except (OSError, ValueError):\n                return False\n        return Path(b["target"]).is_file()\n    expected = {f"{i:05d}.jpg" for i in range(b["n"])}\n    staged = {p.name for p in target.glob("*.jpg") if p.is_file()}\n    manifest = read_manifest(b["id"])\n    if not (staged == expected and len(manifest) == b["n"]\n            and all(manifest.get(f"{i:05d}", {}).get("orig_path") == item["orig"]\n                    for i, item in enumerate(b["frames"]))):\n        original_frames, original_n = b["frames"], b["n"]\n        stage_photos(b)\n        if b["n"] == 0:\n            b["frames"], b["n"] = original_frames, original_n\n            return False\n    return b["n"] > 0\n\n\n# --------------------------------------------------------- unit=photo: папка и прогон на каждое фото\n\ndef chunk_batches(batches, cfg):\n    """unit=photo: пачка предобработки = чанк не больше photos_per_chunk, а результат всё равно\n    лежит в папке исходной папки (b["out"]), чтобы фото не разъезжались по разным местам."""\n    size = max(1, int(cfg.get("photos_per_chunk", 40) or 40))\n    out = []\n    for b in batches:\n        out_id = b.setdefault("out", b["id"])      # папка результата = исходная папка, а не чанк\n        if b["kind"] != "photos" or b["n"] <= size:\n            out.append(b)\n            continue\n        for j in range(0, b["n"], size):\n            piece = dict(b)\n            piece["frames"] = b["frames"][j:j + size]\n            piece["n"] = len(piece["frames"])\n            piece["out"] = out_id\n            piece["id"] = f"{out_id}__c{1 + j // size:02d}"\n            piece["target"] = str(STAGE / piece["id"])        # чанк стейджится сам, по своим номерам\n            piece.pop("track_args", None)\n            out.append(piece)\n    return out\n\n\ndef photos_root(cfg, out_id):\n    """Папка фото исходной папки: без синка в Drive — локально в /content/work, чтобы тесты не трогали Drive."""\n    base = batches_root(cfg) if cfg.get("sync_to_drive", True) else WORK / "sync"\n    return base / out_id / "photos"\n\n\ndef photo_dir(cfg, out_id, name):\n    return photos_root(cfg, out_id) / name\n\n\ndef photo_key(out_id, name):\n    return f"{out_id}/photos/{name}"\n\n\ndef photo_done(folder):\n    """Фото считается готовым, только если в папке есть все обязательные файлы и done.json —\n    именно по этому признаку run пропускает фото после сброса среды."""\n    folder = Path(folder)\n    return folder.is_dir() and all(nonempty_file(folder / name) for name in PHOTO_REQUIRED)\n\n\ndef photo_names_map(b):\n    """Номер кадра чанка -> имя папки фото (та же логика, что и в joint-экспорте)."""\n    rows = [{"frame": item.get("frame", f"{i:05d}"), "orig_relpath": item.get("relpath", ""),\n             "orig_path": item.get("orig", "")} for i, item in enumerate(b["frames"])]\n    return photo_names(rows)\n\n\ndef preproc_files(base, frame, mica=True):\n    """(имя в папке фото, путь в предобработке) для одного кадра — то, что трекер читает по номеру.\n\n    mica=False отдаёт только ядро: MICA делает identity по подвыборке кадров, поэтому отсутствие\n    своей папки mica — не пропуск, а просто инициализация формы средним по чанку."""\n    base = Path(base)\n    out = [(name, base / sub / f"{frame}{ext}") for name, sub, ext in PHOTO_CORE]\n    if not mica:\n        return out\n    out += [("arcface.npy", base / "arcface" / f"{frame}.npy")]\n    out += [(name, base / "mica" / frame / rel) for name, rel in PHOTO_MICA]\n    return out\n\n\ndef preproc_ok(base, frame):\n    return all(nonempty_file(path) for _, path in preproc_files(base, frame, mica=True))\n\n\ndef resolve_run_dir(run_root):\n    """output_folder=<TRACK>/<чанок>__<кадр>, а внутри ещё каталог актёра с суффиксами конфига."""\n    run_root = Path(run_root)\n    if not run_root.is_dir():\n        return None\n    if (run_root / "mesh").is_dir():\n        return run_root\n    cands = [d for d in run_root.iterdir() if d.is_dir() and (d / "mesh").is_dir()]\n    return max(cands, key=lambda d: d.stat().st_mtime) if cands else None\n\n\ndef photo_missing(actor, frame):\n    """Чего не довёз трекер по этому кадру (пустой список = кадр полный)."""\n    holes = []\n    if not nonempty_file(actor / "mesh" / f"{frame}.ply"):\n        holes.append("mesh.ply")\n    if not nonempty_file(actor / "mesh" / "canonical.ply"):\n        holes.append("canonical.ply")\n    if not nonempty_file(actor / "checkpoint" / f"{frame}.frame"):\n        holes.append("checkpoint.frame")\n    return holes\n\n\ndef photo_track_args(cfg, frame, use2023):\n    """Аргументы независимой примерки одного кадра: ровно один кадр, свои итерации, свои ландмарки."""\n    return (f"start_frame={int(frame)}", "num_frames=1", f"iters={int(cfg[\'single_iters\'])}",\n            "batch_size=1", "save_landmarks=True",\n            f"use_flame2023={\'True\' if use2023 else \'False\'}")\n\n\ndef restore_preproc(cfg, b, need):\n    """Достаёт предобработку кадров из их папок в Drive — после сброса среды её не пересчитываем.\n    Верит только метке preproc.json своего чанка: сдвинулась нумерация — считаем заново."""\n    base = PREPRO / b["id"]\n    got = set()\n    for frame, name in need:\n        folder = photo_dir(cfg, b.get("out", b["id"]), name)\n        marker = folder / "preproc.json"\n        if not marker.is_file():\n            continue\n        try:\n            meta = json.loads(marker.read_text(encoding="utf-8"))\n        except (OSError, ValueError):\n            continue\n        if str(meta.get("cid")) != b["id"] or str(meta.get("frame")) != str(frame):\n            continue\n        if not all(nonempty_file(folder / photo_name)\n                   for photo_name, _ in preproc_files(base, frame, mica=False)):\n            continue\n        for photo_name, src in preproc_files(base, frame):        # MICA может не быть — кладём что есть\n            if nonempty_file(src) or not nonempty_file(folder / photo_name):\n                continue\n            src.parent.mkdir(parents=True, exist_ok=True)\n            shutil.copy2(folder / photo_name, src)\n        got.add(frame)\n    return got\n\n\ndef preprocess_subset(cfg, b, frames):\n    """Предобработка только нужных кадров, раскладка по АБСОЛЮТным номерам чанка.\n\n    Пересчитывать весь чанк после каждого сброса среды — значит каждый раз платить десятки минут\n    на уже готовые фото; поэтому во временный батч кладывается только недостающее, а результаты\n    переносятся под своими номерами, и нумерация start_frame не съезжает. Временных батчей всегда\n    несколько по <= MICA_GROUP кадров: иначе MICA проредит кадры и часть фото останется без своего\n    mica_identity, а >10 кадров за один вызов mica_demo.py обрабатывает как раз только с прореживанием."""\n    by_frame = {item["frame"]: item for item in b["frames"]}\n    subset = [by_frame[f] for f in frames if f in by_frame]\n    if not subset:\n        return\n    base = PREPRO / b["id"]\n    base.mkdir(parents=True, exist_ok=True)\n    log = LOGS / f"{b[\'id\']}.log"\n    timeout = float(cfg.get("batch_timeout_min", 0)) or None\n    for j in range(0, len(subset), MICA_GROUP):\n        group = subset[j:j + MICA_GROUP]\n        want = [item["frame"] for item in group]          # stage_photos перенумерует кадры внутри группы\n        tid = f"{b[\'id\']}__pre{j // MICA_GROUP:02d}"\n        tmp = {"id": tid, "subject": b["subject"], "angle": b["angle"], "kind": "photos",\n               "folder": b["folder"], "frames": [dict(i) for i in group], "n": len(group)}\n        stage_photos(tmp)\n        shutil.rmtree(PREPRO / tid, ignore_errors=True)\n        for cmd, cwd in preproc_commands(cfg, tid, tmp["target"]):\n            rc, why = run_cmd(cmd, log, timeout_min=timeout, cwd=cwd)\n            if rc:\n                raise RuntimeError(f"предобработка ({Path(cmd[1]).name}) {tid}: {why}; {tail(log, 10)}")\n        src = PREPRO / tid\n        for index, tmp_item in enumerate(tmp["frames"]):\n            here = f"{index:05d}"\n            dst_frame = tmp_item.get("orig_frame")\n            if dst_frame is None:\n                continue\n            for (_, dst), (_, source) in zip(preproc_files(base, dst_frame), preproc_files(src, here)):\n                if nonempty_file(source):\n                    dst.parent.mkdir(parents=True, exist_ok=True)\n                    shutil.copy2(source, dst)\n            shutil.rmtree(base / "mica" / dst_frame, ignore_errors=True)\n            if (src / "mica" / here).is_dir():\n                shutil.copytree(src / "mica" / here, base / "mica" / dst_frame)\n        shutil.rmtree(src, ignore_errors=True)\n        shutil.rmtree(STAGE / tid, ignore_errors=True)\n\n\ndef sync_preproc(cfg, b, frame, name):\n    """Кладёт предобработку кадра в папку фото и ставит метку чанка — это и страховка от сброса среды."""\n    folder = photo_dir(cfg, b.get("out", b["id"]), name)\n    copied = []\n    for photo_name, path in preproc_files(PREPRO / b["id"], frame):\n        if nonempty_file(path):\n            copy_file(path, folder / photo_name)\n            copied.append(photo_name)\n    write_json({"cid": b["id"], "frame": frame, "relpath": name, "items": sorted(copied),\n                "ts": int(time.time())}, folder / "preproc.json")\n    return copied\n\n\ndef seg_masks(seg_path, size=MASK_SIZE):\n    """Маски наблюдаемости по ровно тем правилам классов, что и read_data(): fg / uv / normal.\n\n    Считается по уже сохранённой segmentation.png, поэтому папку фото можно дополнить ими и потом,\n    без GPU и без пересчёта трекинга."""\n    import numpy as np\n    from PIL import Image\n    with Image.open(seg_path) as im:\n        seg = np.asarray(im.convert("L").resize((size, size), Image.NEAREST))\n    return {"fg": np.isin(seg, SEG_FG), "uv": np.isin(seg, SEG_UV), "normal": np.isin(seg, SEG_NORMAL)}\n\n\ndef write_visibility(folder, size=MASK_SIZE):\n    """Фиксирует, какая сторона лица реально наблюдалась на этом снимке — чтобы сравнение потом\n    шло по пересечению видимых областей, а не по всей модели.\n\n    left/right здесь — стороны КАДРА (как видит зритель), а не стороны головы: их связь с анатомией\n    даёт pose.yaw_deg / pose.bin из params.json, а не признак ниже. Всё берётся из segmentation.png\n    и landmarks_pipnet.npy этой же папки."""\n    import numpy as np\n    from PIL import Image\n    folder = Path(folder)\n    if not nonempty_file(folder / "segmentation.png"):\n        return {"error": "нет segmentation.png — наблюдаемость не зафиксирована"}\n    try:\n        masks = seg_masks(folder / "segmentation.png", size)\n    except Exception as error:      # битая сегментация не имеет права ронять экспорт уже посчитанного фото\n        return {"error": f"segmentation.png не читается: {type(error).__name__}"}\n    for key, mask in masks.items():\n        Image.fromarray((mask.astype(np.uint8)) * 255, mode="L").save(folder / f"mask_{key}.png")\n    out = {"size": size, "masks": [f"mask_{k}.png" for k in masks],\n           "fg_fraction": round(float(masks["fg"].mean()), 4),\n           "uv_fraction": round(float(masks["uv"].mean()), 4),\n           "normal_fraction": round(float(masks["normal"].mean()), 4)}\n    ys, xs = np.nonzero(masks["fg"])\n    if len(xs):\n        x0, x1, y0, y1 = int(xs.min()), int(xs.max()), int(ys.min()), int(ys.max())\n        out["fg_box"] = [x0, y0, x1, y1]\n        mid = (x0 + x1) // 2\n        for side, sl in (("left", slice(x0, max(x0 + 1, mid))), ("right", slice(mid, x1 + 1))):\n            fg_part = int(masks["fg"][y0:y1 + 1, sl].sum())\n            uv_part = int(masks["uv"][y0:y1 + 1, sl].sum())\n            out[f"{side}_uv_of_fg"] = round(uv_part / fg_part, 4) if fg_part else 0.0\n        out["visible_side"] = max(("left", "right"), key=lambda s: out.get(f"{s}_uv_of_fg", 0.0))\n    lms_path = folder / "landmarks_pipnet.npy"\n    if not nonempty_file(lms_path):\n        return out\n    try:\n        lms = np.load(lms_path, allow_pickle=False).astype(np.float64)\n        if lms.ndim != 2 or lms.shape[1] < 2 or not len(lms):\n            return out\n    except Exception as error:\n        out["points_error"] = f"landmarks_pipnet.npy не читается: {type(error).__name__}"\n        return out\n    inside = (lms[:, 0] >= 0) & (lms[:, 0] <= 1) & (lms[:, 1] >= 0) & (lms[:, 1] <= 1)\n    px = np.clip(np.rint(lms * (size - 1)), 0, size - 1).astype(int)\n    hit_fg = np.where(inside, masks["fg"][px[:, 1], px[:, 0]], False).astype(np.uint8)\n    hit_uv = np.where(inside, masks["uv"][px[:, 1], px[:, 0]], False).astype(np.uint8)\n    np.save(folder / "visibility_points.npy",\n            np.stack([px[:, 0], px[:, 1], hit_fg, hit_uv], axis=1).astype(np.float32))\n    out["pipnet_points"] = int(len(px))\n    out["points_out_of_frame"] = int((~inside).sum())\n    zones = {}\n    for index_range, name in PIPNET_ZONES:\n        idx = [i for i in index_range if i < len(px)]\n        if not idx:\n            continue\n        zones[name] = {"points": len(idx),\n                       "in_silhouette": int(hit_fg[idx].sum()),\n                       "in_uv": int(hit_uv[idx].sum()),\n                       "fraction_uv": round(float(hit_uv[idx].mean()), 3)}\n    out["zones"] = zones\n    if len(px) > 45 and inside[30] and inside[36] and inside[45]:\n        outer_l, outer_r = int(px[36, 0]), int(px[45, 0])       # внешние уголки правого и левого глаза\n        span = outer_r - outer_l\n        if abs(span) > 8:\n            # 2D-диагностика кривой подгонки (нос относительно середины глаз), не ярлык ракурса:\n            # метку ракурса даёт только pose по матрице поворота и девяти бинам корпуса\n            out["yaw_proxy"] = round(float((px[30, 0] - (outer_l + outer_r) / 2.0) / span), 4)\n    return out\n\n\ndef ensure_visibility(folder):\n    """Дописывает в уже готовую папку фото то, что считается на CPU: маски наблюдаемости и метку ракурса.\n    Пересчёт трекинга и torch не нужны (поза берётся из матриц, которые уже лежат в params.json),\n    поэтому collect может довести старые корпуса без нового прогона."""\n    params = Path(folder) / "params.json"\n    if not params.is_file():\n        return False\n    try:\n        payload = json.loads(params.read_text(encoding="utf-8"))\n    except (OSError, ValueError):\n        return False\n    changed = False\n    if not (payload.get("visibility") and nonempty_file(Path(folder) / "mask_uv.png")) \\\n            and nonempty_file(Path(folder) / "segmentation.png"):\n        payload["visibility"] = write_visibility(folder)\n        changed = True\n    pose = payload.get("pose") or {}\n    if not pose.get("bin"):            # старые папки: учим ракурс по матрицам, которые уже лежат в params.json\n        fresh = pose_of_payload(payload)\n        if not fresh.get("error"):\n            payload["pose"] = fresh\n            changed = True\n    if not changed:\n        return False\n    payload["files"] = sorted(p.name for p in Path(folder).iterdir() if p.is_file())\n    write_json(payload, params)\n    return True\n\n\ndef export_photo(cfg, b, item, name, actor, sec, args, use2023):\n    """Пишет в папку фото всё, что даёт библиотека: матрицы поворота и ракурс по ним, каноническую\n    форму, параметры, наблюдаемость, ландмарки."""\n    folder = photo_dir(cfg, b.get("out", b["id"]), name)\n    frame = item["frame"]\n    row, one = frame_params(actor, frame)\n    for key in ("orig", "relpath", "face_score", "note"):\n        if item.get(key) is not None:\n            row[key] = item.get(key)\n    source = Path(item.get("orig") or "")\n    if source.is_file():\n        copy_file(source, folder / f"source{source.suffix.lower() or \'.jpg\'}")\n    for label, src in (("mesh.ply", actor / "mesh" / f"{frame}.ply"),\n                       ("canonical.ply", actor / "mesh" / "canonical.ply"),\n                       ("checkpoint.frame", actor / "checkpoint" / f"{frame}.frame")):\n        if nonempty_file(src):\n            copy_file(src, folder / label)\n    for label, src in landmark_artifacts(actor, frame):\n        if nonempty_file(src):\n            copy_file(src, folder / label)\n    overlay_frame(actor / "result.mp4", folder / "overlay.jpg", 0)\n    payload = photo_payload(b, row, {key: value for key, value in one.items()}, 0)\n    payload["unit"] = "photo"\n    payload["canonical"] = "canonical.ply"\n    payload["run"] = {"args": " ".join([f"video_name={b[\'id\']}"] + list(args)), "sec": round(sec, 1),\n                      "chunk": b["id"], "frame": frame}\n    payload["tracking_config"] = effective_config(cfg, dict(b, track_args=" ".join(args)), 1, use2023)\n    payload["mica_own"] = nonempty_file(folder / "mica_identity.npy")\n    # кривые потерь (tensorboard) живут только в папке прогона и удаляются вместе с ней: это единственный\n    # измеримый признак того, насколько меш сел именно на этом фото\n    events = sorted((actor / "logs").glob("events*")) if (actor / "logs").is_dir() else []\n    if events:\n        copy_file(events[0], folder / "tb_events.tfevents")\n        payload["loss_curves"] = "tb_events.tfevents"\n    payload["visibility"] = write_visibility(folder)\n    payload["files"] = sorted(p.name for p in folder.iterdir() if p.is_file())\n    write_json(payload, folder / "params.json")\n    run_log = LOGS / f"{b[\'id\']}__{frame}.log"\n    if run_log.is_file():\n        copy_file(run_log, folder / "run.log")\n    return folder, row, payload\n\n\ndef process_photo(cfg, b, item, name, use2023):\n    """Одно фото: независимая примерка -> экспорт в свою папку -> done.json -> запись в журнал."""\n    frame = item["frame"]\n    args = photo_track_args(cfg, frame, use2023)\n    run_root = TRACK / f"{b[\'id\']}__{frame}"\n    log = LOGS / f"{b[\'id\']}__{frame}.log"\n    # Если предыдущий запуск упал после дорогого трекинга, не удаляем уже посчитанный actor:\n    # сначала проверяем, можно ли экспортировать его повторно без нового GPU.\n    actor = resolve_run_dir(run_root)\n    holes = photo_missing(actor, frame) if actor else ["нет выходного каталога трекинга"]\n    rc, why, sec = 0, "", 0.0\n    if holes:\n        shutil.rmtree(run_root, ignore_errors=True)\n        log.unlink(missing_ok=True)\n        t0 = time.time()\n        rc, why = run_cmd(["python", "scripts/track.py", f"video_name={b[\'id\']}",\n                           f"output_folder={run_root}"] + list(args), log,\n                          timeout_min=float(cfg.get("photo_timeout_min", 15)))\n        sec = time.time() - t0\n        actor = resolve_run_dir(run_root)\n        holes = photo_missing(actor, frame) if actor else ["нет выходного каталога трекинга"]\n    # Лог трекинга сразу кладём на Drive, иначе при сбросе runtime он исчезает вместе с /content/logs.\n    folder = photo_dir(cfg, b.get("out", b["id"]), name)\n    folder.mkdir(parents=True, exist_ok=True)\n    if log.is_file():\n        try:\n            copy_file(log, folder / "run.log")\n        except OSError as error:\n            say(f"  {name}: run.log не сохранён на Drive: {error}")\n    if rc or holes:\n        return "fail", (why or "") + (" | нет: " + ", ".join(holes) if holes else ""), sec, args, {}\n    try:\n        folder, row, payload = export_photo(cfg, b, item, name, actor, sec, args, use2023)\n        vis, pose = payload["visibility"], payload["pose"]\n        if not cfg.get("keep_preproc", True):\n            shutil.rmtree(PREPRO / b["id"] / "mica" / frame, ignore_errors=True)\n        try:\n            done_payload = {"key": photo_key(b.get("out", b["id"]), name), "batch": b.get("out", b["id"]),\n                            "chunk": b["id"], "frame": frame, "photo": name,\n                            "source": item.get("orig", ""), "relpath": item.get("relpath", ""),\n                            "sec": round(sec, 1), "args": " ".join([f"video_name={b[\'id\']}"] + list(args)),\n                            "files": sorted(p.name for p in folder.iterdir() if p.is_file()),\n                            "has_canonical": (folder / "canonical.ply").is_file(),\n                            "pipeline": "unit=photo", "ts": int(time.time())}\n            write_json(done_payload, folder / "done.json")\n        except Exception as error:\n            raise RuntimeError(f"done.json не записан: {type(error).__name__}: {error}") from error\n    except Exception as error:\n        return "fail", f"экспорт папки фото: {type(error).__name__}: {error}", sec, args, {}\n    shutil.rmtree(run_root, ignore_errors=True)\n    return "ok", "", sec, args, {"summary": {key: row.get(key, "") for key in\n                                             ("jaw_norm", "exp_norm", "shape_norm", "t", "fl", "pp")},\n                                 "yaw_deg": pose.get("yaw_deg", ""), "pitch_deg": pose.get("pitch_deg", ""),\n                                 "roll_deg": pose.get("roll_deg", ""), "view_bin": pose.get("bin") or "",\n                                 "pose_error": pose.get("error", ""),\n                                 "visible_side": vis.get("visible_side", ""),\n                                 "yaw_proxy": vis.get("yaw_proxy", ""),\n                                 "uv_fraction": vis.get("uv_fraction", "")}\n\n\ndef photo_todo(cfg, b, records, attempts):\n    """Кадры чанка, которые ещё не посчитаны: готовая папка с done.json пропускается насовсем."""\n    names = photo_names_map(b)\n    out, skipped, exhausted = [], 0, []\n    for i, item in enumerate(b["frames"]):\n        frame = item.get("frame", f"{i:05d}")\n        name = names[frame]\n        key = photo_key(b.get("out", b["id"]), name)\n        if photo_done(photo_dir(cfg, b.get("out", b["id"]), name)):\n            skipped += 1\n            continue\n        if int(records.get(key, {}).get("fails", 0)) >= attempts:\n            exhausted.append(name)\n            continue\n        out.append((item, name, key))\n    return out, skipped, exhausted\n\n\ndef photo_record(cfg, records, b, item, name, key, status, err="", sec=0.0, args_text="", extra=None):\n    """Одна запись в photos_log.jsonl: по ней status/collect видят прогресс, а fails — сколько раз\n    фото уже пробовали (после cfg["attempts"] неудач его больше не трогаем)."""\n    previous = int(records.get(key, {}).get("fails", 0) or 0)\n    rec = {"key": key, "id": b["id"], "out": b.get("out", b["id"]), "photo": name,\n           "frame": item.get("frame", ""), "relpath": item.get("relpath", ""), "n": 1,\n           "log": f"{b[\'id\']}__{item.get(\'frame\', \'\')}", "status": status,\n           "fails": previous + (0 if status == "ok" else 1), "err": str(err)[:400],\n           "sec": round(sec, 1), "args": args_text, "ts": int(time.time())}\n    rec.update(extra or {})\n    append_log(cfg, rec, "photos_log.jsonl", PHOTO_LOG)\n    records[key] = rec\n    return rec\n\n\ndef run_photos(cfg, plan, batches, use2023):\n    """Цикл unit=photo: по одной примерке на фото, журнал и пропуск по папкам фото, без архивов."""\n    name_log, local_log = "photos_log.jsonl", PHOTO_LOG\n    records = {r["key"]: r for r in read_log(cfg, name_log, local_log) if r.get("key")}\n    resume = {}\n    try:\n        resume = json.loads((state_dir(cfg) / "resume.json").read_text(encoding="utf-8"))\n    except (OSError, ValueError):\n        resume = {}\n    if resume.get("next_photo", {}).get("relpath"):\n        say(f"Продолжение с: {resume[\'next_photo\'][\'relpath\']}")\n\n    def write_resume(current=None, completed=None, nxt=None):\n        payload = {"updated": int(time.time())}\n        if current:\n            payload["current"] = {"relpath": current, "ts": int(time.time())}\n        if completed:\n            payload["last_completed"] = {"relpath": completed, "ts": int(time.time())}\n        if nxt:\n            payload["next_photo"] = {"relpath": nxt}\n        try:\n            write_json(payload, state_dir(cfg) / "resume.json")\n        except OSError as error:\n            say(f"Точка продолжения не записана: {error}")\n\n    attempts = max(1, int(cfg.get("attempts", 3)))\n    work, total_done, gave_up = [], 0, 0\n    for b in batches:\n        if not ensure_staged(b):\n            say(f"{b[\'id\']}: нет исходников — проверь INPUT и пересобери plan")\n            continue\n        todo, skipped, exhausted = photo_todo(cfg, b, records, attempts)\n        total_done += skipped\n        for name in exhausted:\n            say(f"  {name}: пробовали {attempts} раз(а) и бросили — правка в журнале {name_log}")\n        gave_up += len(exhausted)\n        if todo or exhausted:\n            work.append((b, todo))\n    left = sum(len(todo) for _, todo in work)\n    say("=" * 72)\n    say(f"КОРПУС (unit=photo): фото всего {sum(b[\'n\'] for b in batches)} | готово {total_done} | "\n        f"считаем {left} | свободно {free_gb()} ГБ | flame2023={use2023}")\n    say(f"чанков предобработки: {len(work)} по <= {cfg.get(\'photos_per_chunk\', 40)} фото | "\n        f"бюджет сессии {cfg[\'session_max_hours\']} ч, рантайм живёт {session_elapsed_hours():.1f} ч "\n        f"из {cfg[\'session_hard_limit_hours\']} ч | стоп-файл: {STOP_FILE}")\n    say("=" * 72)\n    if not left:\n        if gave_up:\n            say(f"считать нечего: готово {total_done}, брошено после {attempts} попыток ещё {gave_up} фото "\n                f"(правь fails в OUTPUT/_state/{name_log}, если их надо повторить)")\n            return 1\n        say("всё посчитано — ничего не делаем; свежие результаты смотри в status/collect")\n        return 0\n    per_photo = median_sec_per_frame(records.values())\n    deadline = session_deadline(cfg)\n    ok_n, fail_n = 0, 0\n    track_queue = []\n    for b, todo in work:\n        if not todo:\n            continue\n        base = PREPRO / b["id"]\n        need = [(item["frame"], name) for item, name, _ in todo]\n        say(f"[чанк {b[\'id\']}] фото {len(todo)} из {b[\'n\']}: готовлю предобработку")\n        missing = [f for f, _ in need if not preproc_ok(base, f)]\n        if missing:\n            restored = restore_preproc(cfg, b, [(f, n) for f, n in need if f in missing])\n            missing = [f for f in missing if f not in restored and not preproc_ok(base, f)]\n        if missing:\n            say(f"  предобработка {len(missing)} кадр(ов) (остальные уже в Drive)")\n            try:\n                preprocess_subset(cfg, b, missing)\n            except (RuntimeError, OSError, ValueError) as error:\n                say(f"  FAIL предобработка чанка: {str(error)[:300]}")\n                for item, name, key in todo:\n                    photo_record(cfg, records, b, item, name, key, "fail", err=str(error)[:400])\n                fail_n += 1\n                continue\n        if not count_dirs(base / "mica"):\n            err = f"чанк {b[\'id\']}: MICA не вернул ни одного identity — трекер на этом падает"\n            say(f"  FAIL {err}")\n            for item, name, key in todo:\n                photo_record(cfg, records, b, item, name, key, "fail", err=err)\n            fail_n += 1\n            continue\n        for frame, name in need:\n            if not preproc_ok(base, frame):\n                continue\n            sync_preproc(cfg, b, frame, name)\n        for item, name, key in todo:\n            if not photo_done(photo_dir(cfg, b.get("out", b["id"]), name)):\n                track_queue.append((photo_moment(Path(item.get("orig", ""))) or datetime.min, b, item, name, key))\n    track_queue.sort(key=lambda x: (x[0], x[2].get("relpath", "")))\n    max_photos = int(cfg.get("max_photos_per_run", 500) or 500)\n    new_attempts = 0\n    stop_run = False\n    for _, b, item, name, key in track_queue:\n        if new_attempts >= max_photos:\n            say(f"лимит запуска: {max_photos} фото за один run — остальное в следующий раз")\n            stop_run = True\n            break\n        base = PREPRO / b["id"]\n        folder = photo_dir(cfg, b.get("out", b["id"]), name)\n        if photo_done(folder):\n            continue\n        if not preproc_ok(base, item["frame"]):\n            holes = [p.name for _, p in preproc_files(base, item["frame"], mica=False)\n                     if not nonempty_file(p)]\n            say(f"  {name}: предобработка неполная (нет {\', \'.join(holes[:4])}) — в журнал пропуском")\n            photo_record(cfg, records, b, item, name, key, "fail",\n                         err="предобработка: нет " + ", ".join(holes[:6]))\n            fail_n += 1\n            continue\n        if STOP_FILE.exists():\n            say(f"найден {STOP_FILE} — останавливаюсь на {name}")\n            stop_run = True\n            break\n        if time.time() > deadline:\n            say(f"бюджет сессии исчерпан — стоп на {name}, жми run ещё раз")\n            stop_run = True\n            break\n        est = (per_photo or float(cfg.get("photo_timeout_min", 15)) * 60) + GUARD_MARGIN_S\n        if est > deadline - time.time():\n            say(f"не начинаю {name}: оценка {human(est)} больше остатка "\n                f"{human(max(deadline - time.time(), 0.0))} — запускай run в свежей среде")\n            stop_run = True\n            break\n        new_attempts += 1\n        write_resume(current=item.get("relpath", name))\n        status, err, sec, args, extra = process_photo(cfg, b, item, name, use2023)\n        photo_record(cfg, records, b, item, name, key, status, err=err, sec=sec,\n                     args_text=" ".join(args), extra=dict(extra, canonical=status == "ok"))\n        if status == "ok":\n            ok_n += 1\n            done_here = 0\n            per_photo = per_photo or sec\n            left_now = left - ok_n - fail_n\n            write_resume(completed=item.get("relpath", name),\n                        nxt=next((x[2].get("relpath", x[3]) for x in track_queue\n                                  if x[2].get("relpath", x[3]) != item.get("relpath", name)), ""))\n            say(f"  {name} -> ok за {human(sec)} | осталось {left_now} фото "\n                f"(~{human(per_photo * left_now) if per_photo else \'?\'})")\n        else:\n            fail_n += 1\n            say(f"  {name} -> fail за {human(sec)} | {err[:200]}")\n            if "таймаут" in err or free_gb() < float(cfg["min_free_gb"]):\n                stop_run = True\n                break\n    if stop_run:\n        pass\n    save_plan(cfg, plan)\n    say("-" * 72)\n    say(f"сессия: ok {ok_n} | провалов {fail_n}. Жми ту же ячейку run — посчитанные фото пропускаются "\n        f"по папкам (done.json) и по журналу OUTPUT/_state/{name_log}")\n    return 1 if fail_n else 0\n\n\ndef run_folders(cfg, plan, use2023, batches=None):\n    """Joint-режим: один прогон на всю папку (для видео другого и не бывает)."""\n    batches = plan["batches"] if batches is None else batches\n    records = {r["id"]: r for r in read_log(cfg)}\n    done, sync_failed = set(), set()\n    for b in batches:\n        rec = records.get(b["id"], {})\n        if rec.get("status") not in ("ok", "ok_no_sync"):\n            continue\n        if rec.get("partial"):\n            say(f"{b[\'id\']}: демо посчитало {rec.get(\'n\')} из {b[\'n\']} кадров — joint-трекинг не дополняется, "\n                f"пересчитываю батч целиком")\n            continue\n        folder = (batch_dir(cfg, b["id"]) if cfg.get("sync_to_drive", True)\n                  else WORK / "sync" / b["id"])\n        if rec.get("status") == "ok" and not bundle_missing(folder, rec.get("synced", []), cfg):\n            done.add(b["id"])\n            continue\n        actor = resolve_actor_dir(b["id"])\n        n = int(rec.get("n", 0))\n        if actor and n > 0 and not track_missing(actor, n):\n            say(f"досылаю результат {b[\'id\']} (в выходной папке он неполный)")\n            manifest = read_manifest(b["id"])\n            if b["kind"] == "photos" and len(manifest) == n:\n                b["frames"] = [{"orig": row["orig_path"], "relpath": row["orig_relpath"],\n                                "frame": row["frame"], "note": row.get("note", ""),\n                                "score": row.get("face_score", "")} for row in manifest.values()]\n                b["n"] = n\n            b["n_done"] = n\n            rec = dict(rec)\n            try:\n                b["track_args"] = rec.get("args") or b.get("track_args", track_args(cfg, n, use2023))\n                rec["synced"] = sync_batch(cfg, b, actor, n, rec.get("sec", 0), rec.get("flame2023", use2023))\n                rec["status"] = "ok"\n                rec["err"] = ""\n                done.add(b["id"])\n            except Exception as error:\n                rec["status"] = "ok_no_sync"\n                rec["err"] = f"результат есть, но не записан: {type(error).__name__}: {error}"[:500]\n                sync_failed.add(b["id"])\n                say(f"  {b[\'id\']}: {rec[\'err\']}")\n            append_log(cfg, rec)\n            save_plan(cfg, plan)\n        else:\n            say(f"{b[\'id\']}: полного результата нет ни в выходной папке, ни на /content — пересчитываю")\n    todo = [b for b in batches if b["id"] not in (done | sync_failed)]\n    say("=" * 72)\n    say(f"КОРПУС: батчей {len(batches)} | готово {len(done)} | в работе {len(todo)} | "\n        f"кадров {sum(b[\'n\'] for b in todo)} | свободно {free_gb()} ГБ | flame2023={use2023}")\n    say(f"бюджет сессии {cfg[\'session_max_hours\']} ч, рантайм живёт {session_elapsed_hours():.1f} ч "\n        f"из {cfg[\'session_hard_limit_hours\']} ч, стоп-файл: {STOP_FILE} (создай, чтобы остановить цикл)")\n    say("=" * 72)\n    if not todo:\n        say("все батчи уже в логе — ничего не делаем; свежие результаты смотри в status/collect")\n        return 1 if sync_failed else 0\n    if str(cfg["input"]).lower().endswith(".zip") and any(\n            not Path(item["orig"]).is_file() for b in todo for item in b["frames"]):\n        root = unpack_zip(resolve_input(cfg["input"]))\n        for b in todo:\n            for item in b["frames"]:\n                item["orig"] = str(root / item["relpath"])\n    deadline = session_deadline(cfg)\n    per_frame = median_sec_per_frame(records.values())\n    failed = len(sync_failed)\n    ok_now = 0\n    for k, b in enumerate(todo, 1):\n        if STOP_FILE.exists():\n            say(f"[{k}/{len(todo)}] найден {STOP_FILE} — останавливаюсь на {b[\'id\']}")\n            break\n        if time.time() > deadline:\n            say(f"[{k}/{len(todo)}] бюджет сессии исчерпан — стоп на {b[\'id\']}")\n            break\n        if free_gb() < float(cfg["min_free_gb"]):\n            say(f"[{k}/{len(todo)}] свободно {free_gb()} ГБ (< {cfg[\'min_free_gb\']}) — стоп на {b[\'id\']}")\n            break\n        # батч нельзя прервать на середине и дополнить позже, поэтому не начинаем тот,\n        # который заведомо не влезает в остаток сессии — иначе потеряем и его целиком, и квоту\n        est = per_frame * b["n"] + GUARD_MARGIN_S if per_frame else float(cfg["batch_timeout_min"]) * 60\n        left = deadline - time.time()\n        if est > left:\n            say(f"[{k}/{len(todo)}] не начинаю {b[\'id\']} ({b[\'n\']} кадр(ов)): оценка {human(est)} "\n                f"больше остатка {human(max(left, 0.0))} — дождись свежей среды и жми run ещё раз")\n            break\n        say(f"[{k}/{len(todo)}] {b[\'id\']} ({b[\'n\']} кадр(ов)) старт, свободно {free_gb()} ГБ")\n        planned = b["n"]\n        if not ensure_staged(b):\n            rec = {"id": b["id"], "n": 0, "planned": planned, "status": "fail",\n                   "err": f"нет исходников для {b[\'id\']} (проверь INPUT / пересобери plan)",\n                   "args": "", "sec": 0.0, "flame2023": bool(use2023), "ts": int(time.time()),\n                   "synced": [], "notes": []}\n            append_log(cfg, rec)\n            failed += 1\n            say(f"[{k}/{len(todo)}] {b[\'id\']} -> fail: {rec[\'err\']}")\n            continue\n        started = time.time()\n        try:\n            status, err, args, sec = process_batch(cfg, b, use2023)\n        except Exception as error:\n            status, err, args, sec = "fail", f"{type(error).__name__}: {error}", "", time.time() - started\n        n_done = b.get("n_done", 0)\n        plan["total_photos"] = sum(batch["n"] for batch in plan["batches"])\n        save_plan(cfg, plan)\n        rec = {"id": b["id"], "n": n_done, "planned": planned, "status": status, "err": err[:500],\n               "args": args, "sec": round(sec, 1), "flame2023": bool(use2023), "ts": int(time.time()),\n               "synced": [], "notes": [i["relpath"] for i in b["frames"] if i.get("note")][:20]}\n        if status == "ok":\n            actor = resolve_actor_dir(b["id"])\n            try:\n                rec["synced"] = sync_batch(cfg, b, actor, n_done, sec, use2023)\n            except Exception as error:\n                import traceback\n                traceback.print_exc()\n                status = "ok_no_sync"\n                rec["status"] = status\n                rec["err"] = f"результат есть, но в Drive не записан: {type(error).__name__}: {error}"[:500]\n        append_log(cfg, rec)\n        if status != "ok":\n            failed += 1\n            say(f"[{k}/{len(todo)}] {b[\'id\']} -> {status} за {human(sec)} | {rec[\'err\'][:240]}")\n        else:\n            ok_now += 1\n            left = sum(x["n"] for x in todo[k:])\n            per = sec / max(n_done, 1)\n            say(f"[{k}/{len(todo)}] {b[\'id\']} -> ok за {human(sec)} ({per:.0f} с/кадр) | "\n                f"осталось ~{human(left * per)} | свободно {free_gb()} ГБ | в Drive {len(rec[\'synced\'])} файлов")\n    say("-" * 72)\n    say(f"цикл завершен за сессию: ok {ok_now}, провалов/без синка {failed}. "\n        f"Жми ту же ячейку run ещё раз — готовые батчи пропускаются (лог живёт в Drive).")\n    say("Если среда сбросилась: 1 → 2 → 3 (она сама распакует слепок _env) → 4 → run; "\n        "не считанное не потеряно — курсор в OUTPUT/_state/batch_log.jsonl")\n    return 1 if failed else 0\n\n\ndef st_run(cfg):\n    plan = load_plan(cfg)\n    if not plan:\n        say("плана нет или INPUT/OUTPUT/фильтры изменились — сначала выполни стадию plan")\n        return 1\n    use2023 = resolve_flame2023(cfg)\n    plan["use_flame2023"] = use2023\n    if str(cfg.get("unit", "folder")) != "photo":\n        return run_folders(cfg, plan, use2023)\n    photos = [b for b in plan["batches"] if b["kind"] == "photos"]\n    others = [b for b in plan["batches"] if b["kind"] != "photos"]\n    rc = run_photos(cfg, plan, photos, use2023) if photos else 0\n    if others:\n        rc = max(rc, run_folders(cfg, plan, use2023, others))\n    return rc\n\n\ndef photo_groups(cfg, plan, records):\n    """unit=photo: сводка по исходным папкам — сколько фото посчитано, сколько сдохло.\n    Источник истины — папки в выходе (done.json), журнал даёт скорости и ошибки."""\n    groups = {}\n    for b in (plan["batches"] if plan else []):\n        if b["kind"] != "photos":\n            continue\n        names = photo_names_map(b)\n        out = b.get("out", b["id"])\n        group = groups.setdefault(out, {"total": 0, "done": 0, "fail": 0, "errors": []})\n        for i, item in enumerate(b["frames"]):\n            name = names[item.get("frame", f"{i:05d}")]\n            group["total"] += 1\n            if photo_done(photo_dir(cfg, out, name)):\n                group["done"] += 1\n                continue\n            rec = records.get(photo_key(out, name), {})\n            if int(rec.get("fails", 0)) >= max(1, int(cfg.get("attempts", 3))):\n                group["fail"] += 1\n                group["errors"].append(f"{name}: {str(rec.get(\'err\', \'\'))[:70]}")\n    return groups\n\n\ndef status_photos(cfg, plan):\n    log = read_log(cfg, "photos_log.jsonl", PHOTO_LOG)\n    records = {r["key"]: r for r in log if r.get("key")}\n    groups = photo_groups(cfg, plan, records)\n    resume = {}\n    try:\n        resume = json.loads((state_dir(cfg) / "resume.json").read_text(encoding="utf-8"))\n    except (OSError, ValueError):\n        resume = {}\n    if resume.get("current"):\n        say(f"Текущее фото: {resume[\'current\'].get(\'relpath\')}")\n    if resume.get("last_completed"):\n        say(f"Последнее готовое: {resume[\'last_completed\'].get(\'relpath\')}")\n    if resume.get("next_photo"):\n        say(f"Следующее фото: {resume[\'next_photo\'].get(\'relpath\')}")\n    total = sum(g["total"] for g in groups.values())\n    done = sum(g["done"] for g in groups.values())\n    say(f"{\'папка\':<26}{\'фото\':>7}{\'готово\':>8}{\'с броском\':>11}")\n    for out, group in sorted(groups.items()):\n        say(f"{out:<26}{group[\'total\']:>7}{group[\'done\']:>8}{group[\'fail\']:>11}")\n        for line in group["errors"][:3]:\n            say(f"    ! {line}")\n    say("-" * 72)\n    per = median_sec_per_frame(records.values())\n    left = total - done\n    say(f"фото: посчитано {done} из {total} | осталось {left} | свободно {free_gb()} ГБ | "\n        f"журнал: OUTPUT/_state/photos_log.jsonl")\n    if left:\n        estimate = per or PHOTO_DEFAULT_S\n        say(f"по {\'медиане журнала\' if per else \'умолчанию\'} {estimate:.0f} с/фото — осталось "\n            f"~{human(estimate * left)} GPU-времени (без учёта перезапусков среды)")\n    hard = float(cfg.get("session_hard_limit_hours", 0) or 0)\n    snap = read_env_manifest(cfg)\n    snap_age = max(0.0, time.time() - float(snap.get("ts", 0) or 0)) if snap.get("parts") else 0.0\n    say(f"среда живёт {session_elapsed_hours():.1f} ч" + (f" из {hard:.0f} ч" if hard else "") +\n        f" | слепок среды: {f\'{human(snap_age)} назад\' if snap.get(\'parts\') else \'нет (сделай ячейку 4b)\'}")\n    if not groups:\n        say("плана с фото нет (стадия plan) — показываю только журнал")\n        for key, rec in list(records.items())[:40]:\n            say(f"  {key:<50}{rec.get(\'status\', \'?\'):<8}{human(rec.get(\'sec\', 0))}")\n    return 0\n\n\ndef st_status(cfg):\n    plan = load_plan(cfg)\n    records = {r["id"]: r for r in read_log(cfg)}\n    say("=" * 72)\n    say("ДАШБОРД")\n    say("=" * 72)\n    if str(cfg.get("unit", "folder")) == "photo":\n        return status_photos(cfg, plan)\n    batches = plan["batches"] if plan else []\n    if not batches:\n        say("плана нет (стадия plan) — показываю только лог")\n    say(f"{\'батч\':<26}{\'кадр\':>6} {\'статус\':<10}{\'mesh\':>6}{\'frame\':>7}{\'лмк\':>5} {\'в Drive\':>9}  {\'время\':>8}")\n    totals = {"ok": 0, "fail": 0, "todo": 0, "photos": 0, "done_photos": 0}\n    out = Path(cfg["output"])\n    for b in batches:\n        rec = records.get(b["id"])\n        actor = resolve_actor_dir(b["id"]) if actor_root_exists() else None\n        meshes = count_files(actor / "mesh", ".ply") if actor else 0\n        frames = count_files(actor / "checkpoint", ".frame") if actor else 0\n        lmks = count_files(actor / "mesh", ".npy") if actor else 0\n        folder = batch_dir(cfg, b["id"])\n        drive = len(list(folder.iterdir())) if folder.is_dir() else 0\n        if rec is None:\n            state, sec = "не сделан", ""\n            totals["todo"] += 1\n        else:\n            base = rec.get("status", "?")\n            state = f"демо {rec.get(\'n\')}/{b[\'n\']}" if base == "ok" and rec.get("partial") else base\n            sec = human(rec.get("sec", 0))\n            totals["ok" if base == "ok" else "fail"] += 1\n            totals["done_photos"] += rec.get("n", 0) if base == "ok" else 0\n        totals["photos"] += b["n"]\n        say(f"{b[\'id\']:<26}{b[\'n\']:>6} {state:<10}{meshes:>6}{frames:>7}{lmks:>5} {drive:>9}  {sec:>8}")\n        if rec and rec.get("err"):\n            say(f"    ! {rec[\'err\'][:200]}")\n    say("-" * 72)\n    say(f"батчей: {len(batches)} | ok {totals[\'ok\']} | провал {totals[\'fail\']} | "\n        f"не сделано {totals[\'todo\']} | кадров {totals[\'done_photos\']}/{totals[\'photos\']} | "\n        f"свободно {free_gb()} ГБ | Drive: {out.is_dir()}")\n    per = median_sec_per_frame(records.values())\n    left_frames = max(0, totals["photos"] - totals["done_photos"])\n    if per and left_frames:\n        say(f"по медиане {per:.0f} с/кадр осталось ~{human(per * left_frames)} GPU-времени "\n            f"(без учёта перезапусков среды)")\n    hard = float(cfg.get("session_hard_limit_hours", 0) or 0)\n    snap = read_env_manifest(cfg)\n    snap_age = max(0.0, time.time() - float(snap.get("ts", 0) or 0)) if snap.get("parts") else 0.0\n    say(f"среда живёт {session_elapsed_hours():.1f} ч" + (f" из {hard:.0f} ч" if hard else "") +\n        f" | слепок среды: {f\'{human(snap_age)} назад\' if snap.get(\'parts\') else \'нет (сделай ячейку 4b)\'}")\n    if not plan:\n        for bid, rec in records.items():\n            say(f"  {bid:<26}{rec.get(\'n\', 0):>6} {rec.get(\'status\', \'?\'):<10} {human(rec.get(\'sec\', 0))}")\n    return 0\n\n\ndef actor_root_exists():\n    return TRACK.is_dir()\n\n\ndef folder_pose(folder):\n    """Ракурс из params.json готовой папки фото — на случай журнала, написанного до появления позы."""\n    params = Path(folder) / "params.json"\n    if not params.is_file():\n        return {}\n    try:\n        return json.loads(params.read_text(encoding="utf-8")).get("pose") or {}\n    except (OSError, ValueError):\n        return {}\n\n\ndef label_bin(text):\n    """Бин из разметки корпуса: имя файла/папки вида `right_light__2016_05_20.jpg`.\n    Метка человека остаётся главной — сверяем её с измеренной позой, чтобы видеть расход, а не перезаписывать."""\n    low = str(text or "").lower()\n    for name, _, _, _ in VIEW_BINS:\n        if re.search(rf"(?<![a-z]){name}(?![a-z])", low):\n            return name\n    return ""\n\n\ndef collect_photos(cfg, out):\n    """unit=photo: индекс строится по журналу фото, папки сверяются одним списком каталога."""\n    records = {r["key"]: r for r in read_log(cfg, "photos_log.jsonl", PHOTO_LOG) if r.get("key")}\n    present = set()\n    root = batches_root(cfg)\n    topped_up = 0\n    if root.is_dir():\n        for batch in sorted(p for p in root.iterdir() if p.is_dir() and not p.name.startswith("_")):\n            photos = batch / "photos"\n            if not photos.is_dir():\n                continue\n            for name in sorted(p.name for p in photos.iterdir() if p.is_dir()):\n                present.add(photo_key(batch.name, name))\n                topped_up += 1 if ensure_visibility(photos / name) else 0\n    cols = ["key", "batch", "photo", "frame", "relpath", "status", "fails", "sec",\n            "yaw_deg", "pitch_deg", "roll_deg", "view_bin", "label_bin", "bin_matches_label",\n            "visible_side", "uv_fraction", "yaw_proxy",\n            "jaw_norm", "exp_norm", "shape_norm", "t", "fl", "pp", "folder", "ts"]\n    pose_keys = ("yaw_deg", "pitch_deg", "roll_deg", "visible_side", "yaw_proxy", "uv_fraction")\n    rows = []\n    for key, rec in sorted(records.items()):\n        row = {"key": key, "batch": rec.get("out", ""), "photo": rec.get("photo", ""),\n               "frame": rec.get("frame", ""), "relpath": rec.get("relpath", ""),\n               "status": rec.get("status", ""), "fails": rec.get("fails", 0),\n               "sec": rec.get("sec", ""), "folder": "есть" if key in present else "НЕТ",\n               "ts": rec.get("ts", "")}\n        pose = rec\n        if not rec.get("view_bin") and rec.get("status") == "ok":\n            pose = folder_pose(photos_root(cfg, rec.get("out", "")) / rec.get("photo", "")) or rec\n        row.update({k: pose.get(k, rec.get(k, "")) for k in pose_keys})\n        row["view_bin"] = pose.get("view_bin") or pose.get("bin") or ""\n        row["label_bin"] = label_bin(rec.get("relpath") or rec.get("photo"))\n        row["bin_matches_label"] = ("" if not row["label_bin"] or not row["view_bin"] else\n                                   "да" if row["label_bin"] == row["view_bin"] else "нет")\n        row.update({k: v for k, v in (rec.get("summary") or {}).items() if k in cols})\n        rows.append(row)\n    (out / "_index").mkdir(parents=True, exist_ok=True)\n    with (out / "_index/photos.csv").open("w", newline="", encoding="utf-8") as fh:\n        writer = csv.DictWriter(fh, fieldnames=cols, extrasaction="ignore")\n        writer.writeheader()\n        writer.writerows(rows)\n    gone = sorted(present - set(records))\n    ok = sum(1 for r in records.values() if r.get("status") == "ok")\n    bins = {name: sum(1 for r in rows if r["view_bin"] == name) for name, *_ in VIEW_BINS}\n    bins["без метки"] = sum(1 for r in rows if not r["view_bin"])\n    check = Counter(r["bin_matches_label"] for r in rows)\n    confusion = Counter(f"{r[\'label_bin\']} -> {r[\'view_bin\']}" for r in rows if r["bin_matches_label"] == "нет")\n    report = {"config": {k: cfg.get(k) for k in ("input", "output", "unit", "photos_per_chunk",\n                                                 "single_iters", "use_flame2023", "subjects")},\n              "totals": {"photos_in_log": len(records), "ok": ok,\n                         "fail": len(records) - ok, "folders_in_drive": len(present),\n                         "without_log_entry": len(gone), "free_gb": free_gb()},\n              "view_bins": bins,\n              "bin_vs_label": {"совпало": check["да"], "разошлось": check["нет"],\n                              "не с чем сверять": check[""],\n                              "топ расхождений": dict(confusion.most_common(8))},\n              "per_batch": {batch: sum(1 for r in records.values() if r.get("out") == batch and\n                                       r.get("status") == "ok")\n                            for batch in sorted({r.get("out") for r in records.values() if r.get("out")})}}\n    (out / "_index/report.json").write_text(json.dumps(report, ensure_ascii=False, indent=1),\n                                            encoding="utf-8")\n    say(f"фото в журнале: {len(records)} (ok {ok}) | папок в выходе: {len(present)} | "\n        f"папок без записи в журнале: {len(gone)}")\n    for key in gone[:10]:\n        say(f"  {key}: папка есть, в photos_log.jsonl её нет — так и оставим, run её не тронет")\n    say(f"индекс: {out}/_index/photos.csv ({len(rows)} строк) и report.json")\n    say("ракурсы по матрице поворота (9 бинов корпуса): " +\n        ", ".join(f"{name} {count}" for name, count in bins.items() if count))\n    say(f"сверка с твоей разметкой в именах файлов: совпало {check[\'да\']}, разошлось {check[\'нет\']}, "\n        f"без метки ракурса в имени {check[\'\']}")\n    for pair, count in confusion.most_common(5):\n        say(f"  расхождение {pair}: {count}")\n    say("в папке каждого фото: source, crop.jpg, segmentation.png, landmarks_pipnet.npy, normals.png, "\n        "uv_map.png, arcface.npy, mica_* (identity/mesh/kpt7/kpt68), mesh.ply, canonical.ply, "\n        "landmarks_7/68/eyes.npy, checkpoint.frame, mask_fg/uv/normal.png, visibility_points.npy, "\n        "params.json (в нём pose: yaw/pitch/roll + bin), overlay.jpg, run.log, done.json")\n    if topped_up:\n        say(f"дописал наблюдаемости в {topped_up} папк(ах) — это был CPU без пересчёта трекинга")\n    return 1 if (len(records) - ok) else 0\n\n\ndef st_collect(cfg):\n    out = Path(cfg["output"])\n    say("=" * 72)\n    say(f"СБОРКА ИНДЕКСА в {out}")\n    say("=" * 72)\n    if str(cfg.get("unit", "folder")) == "photo":\n        return collect_photos(cfg, out)\n    records = {r["id"]: r for r in read_log(cfg)}\n    root = batches_root(cfg)\n    batch_dirs = sorted(p for p in root.glob("*") if p.is_dir() and not p.name.startswith("_")) \\\n        if root.is_dir() else []\n    index_rows = []\n    report = {"config": {k: cfg[k] for k in ("input", "output", "subjects", "use_flame2023",\n                                             "face_filter", "torch_compile")},\n              "batches": {}, "missing": []}\n    for folder in batch_dirs:\n        bid = folder.name\n        files = {p.name: p.stat().st_size for p in folder.iterdir() if p.is_file()}\n        frames_file = folder / "frames.csv"\n        rows = 0\n        if frames_file.is_file():\n            with frames_file.open(encoding="utf-8") as fh:\n                for row in csv.DictReader(fh):\n                    row["batch"] = bid\n                    index_rows.append(row)\n                    rows += 1\n        archive_ok = (not cfg.get("keep_archive", True) or\n                      ((folder / "artifacts.tar.gz").is_file() and\n                       tarfile.is_tarfile(folder / "artifacts.tar.gz")))\n        missing_files = bundle_missing(folder, records.get(bid, {}).get("synced", []), cfg)\n        report["batches"][bid] = {"files": files, "frames": rows, "log_status": records.get(bid, {}).get("status"),\n                                  "archive": archive_ok, "missing_files": missing_files}\n        if not archive_ok or rows == 0 or missing_files:\n            report["missing"].append(bid)\n        say(f"{bid:<26} кадров {rows:>5} | archive {\'ok\' if archive_ok else \'НЕТ\':<4} | "\n            f"лог: {records.get(bid, {}).get(\'status\', \'нет записи\')}")\n    plan = load_plan(cfg)\n    expected = set(records) | {b["id"] for b in plan["batches"]} if plan else set(records)\n    for bid in sorted(expected):\n        if bid not in report["batches"]:\n            report["missing"].append(bid)\n            say(f"{bid:<26} В ДРАЙВЕ НЕТ — перезапусти run для этого батча")\n    (out / "_index").mkdir(parents=True, exist_ok=True)\n    with (out / "_index/frames_all.csv").open("w", newline="", encoding="utf-8") as fh:\n        writer = csv.DictWriter(fh, fieldnames=FRAMES_ALL_CSV_COLS, extrasaction="ignore")\n        writer.writeheader()\n        writer.writerows(sorted(index_rows, key=lambda r: (r["batch"], r.get("frame", ""))))\n    report["frames_indexed"] = len(index_rows)\n    report["totals"] = {"batches_in_drive": len(batch_dirs), "frames": len(index_rows),\n                        "log_ok": sum(1 for r in records.values() if r.get("status") == "ok"),\n                        "log_fail": sum(1 for r in records.values() if r.get("status") not in ("ok", "ok_no_sync")),\n                        "free_gb": free_gb()}\n    (out / "_index").mkdir(parents=True, exist_ok=True)\n    (out / "_index/report.json").write_text(json.dumps(report, ensure_ascii=False, indent=1), encoding="utf-8")\n    say("-" * 72)\n    say(f"в Drive батчей: {len(batch_dirs)} | строк в _index/frames_all.csv: {len(index_rows)} | "\n        f"проблемных батчей: {len(report[\'missing\'])}")\n    say(f"что лежит по каждому батчу: frames.csv (кадр -> исходное фото + параметры + ракурс), manifest.csv, "\n        f"params.npz (exp/shape/jaw/neck/eyes/eyelids/R/t/camera/joint_transforms), metrics.json (в т.ч. "\n        f"эффективный конфиг трекинга), result.mp4"\n        + (", artifacts.tar.gz (mesh/ checkpoint/ logs/ preproc/)" if cfg.get("keep_archive", True) else "")\n        + f", summary.md и photos/<имя фото>/ (mesh.ply, params.json, ландмарки, кроп, overlay)")\n    return 1 if report["missing"] else 0\n\n\ndef st_freeze(cfg):\n    """Publish a complete cache generation last; failed saves never replace the old manifest."""\n    previous = read_env_manifest(cfg)\n    if snapshot_ready(cfg, previous):\n        say("Проверенный кеш среды уже сохранен: повторная упаковка не нужна")\n        return 0\n    if not ENV_MARKER.is_file() or not (ENV_ROOT / "bin/python").is_file() or not CODE.is_dir():\n        say("Нет проверенного окружения: сначала выполни ячейки 3 и 4")\n        return 1\n    if any(not ok for _, _, ok, _ in check_weights()):\n        say("Веса не прошли проверку: неполный кеш не публикуется")\n        return 1\n    dest = env_dir(cfg, make=True)\n    generation = str(time.time_ns())\n    parts, created = [], []\n    roots = tuple(p.absolute() for _, sources, _ in freeze_parts() for p in sources)\n\n    def include(info):\n        if cache_excluded(info.name):\n            return None\n        return info\n\n    try:\n        for index, (base, sources, note) in enumerate(freeze_parts()):\n            present = [Path(s) for s in sources if Path(s).is_dir()]\n            if not present:\n                raise ValueError(f"Нет данных для обязательной части {base}")\n            raw = sum(dir_size_gb(p) for p in present)\n            if free_gb_at(dest) < raw + 1.0:\n                raise OSError(f"Мало места для {base}: нужно до {raw + 1:.1f} ГБ свободного места")\n            # Dereference only links into the fixed cache roots, never user data or credentials.\n            for source in present:\n                for parent, dirs, files in os.walk(source):\n                    dirs[:] = [d for d in dirs if not cache_excluded(str(Path(parent) / d))]\n                    for filename in dirs + files:\n                        path = Path(parent) / filename\n                        if not cache_excluded(str(path)) and path.is_symlink():\n                            resolved = path.resolve()\n                            if not any(resolved.is_relative_to(root) for root in roots):\n                                raise ValueError(f"Ссылка за пределы кеша: {path} -> {resolved}")\n            name = base.removesuffix(".tar.gz") + "-" + generation + ".tar.gz"\n            archive, partial = dest / name, dest / (name + ".part")\n            created.extend((partial, archive))\n            progress(phase="Сохранение кеша: " + base, current=f"{raw:.2f} ГБ до сжатия",\n                     completed=index, total=len(freeze_parts()))\n            digest = hashlib.sha256()\n            with partial.open("wb") as stream:\n                class HashWriter:\n                    def write(self, data):\n                        digest.update(data)\n                        return stream.write(data)\n\n                    def __getattr__(self, key):\n                        return getattr(stream, key)\n\n                with tarfile.open(fileobj=HashWriter(), mode="w:gz", compresslevel=1, dereference=True) as tar:\n                    for source in present:\n                        tar.add(source, arcname=str(source.absolute()).lstrip("/"), filter=include)\n            partial.replace(archive)\n            parts.append({"name": name, "bytes": archive.stat().st_size, "sha256": digest.hexdigest(),\n                          "raw_gb": round(raw, 2), "sources": [str(p) for p in present], "note": note})\n            say(f"Сохранен {base}: {archive.stat().st_size / 1e9:.2f} ГБ")\n        manifest = {"ts": int(time.time()), "output": cfg["output"], "parts": parts,\n                    "pixel_rev": PIXEL_REV, "signature": snapshot_signature()}\n        write_json(manifest, env_manifest_path(cfg))\n    except (OSError, tarfile.TarError, ValueError, RecursionError) as error:\n        for path in created:\n            path.unlink(missing_ok=True)\n        say(f"Кеш не сохранен: {type(error).__name__}: {error}. Предыдущий слепок оставлен без изменений.")\n        return 1\n    # Only archives named in our previous manifest are retired after successful publication.\n    try:\n        for _, archive, _, _ in snapshot_parts(cfg, previous):\n            if archive.name not in {p["name"] for p in parts}:\n                archive.unlink(missing_ok=True)\n    except (OSError, ValueError, TypeError):\n        pass\n    progress(completed=len(parts), total=len(parts), current="")\n    say(f"Кеш следующей сессии: {sum(p[\'bytes\'] for p in parts) / 1e9:.2f} ГБ в {dest}")\n    return 0\n\n\ndef st_warm(cfg):\n    """Restore fixed cache roots; a separate CUDA verification decides whether setup is usable."""\n    manifest = read_env_manifest(cfg)\n    if not manifest.get("parts"):\n        say("Кеша среды пока нет: первая установка будет выполнена полностью")\n        return 1\n    if manifest.get("output") != cfg["output"] or manifest.get("pixel_rev") != PIXEL_REV:\n        say("Кеш другого выхода или другой ревизии кода: восстановление пропущено")\n        return 1\n    try:\n        parts = snapshot_parts(cfg, manifest)\n    except (ValueError, TypeError) as error:\n        say(f"Некорректный кеш: {error}")\n        return 1\n    t0 = time.time()\n    restored, bad = [], []\n    for index, (part, archive, sources, kind) in enumerate(parts):\n        if kind == "p3dmm-env.tar.gz" and (ENV_ROOT / "bin/python").is_file() and ENV_MARKER.is_file():\n            continue\n        local = WORK / "env_restore" / archive.name\n        progress(phase="Восстановление кеша: " + kind, current=archive.name, completed=index, total=len(parts))\n        try:\n            if not archive.is_file() or archive.stat().st_size != part.get("bytes"):\n                raise ValueError("архив отсутствует или имеет неправильный размер")\n            if free_gb() < archive.stat().st_size / 1e9 + float(part.get("raw_gb", 0)) + 1:\n                raise OSError("мало места на /content для восстановления")\n            local.parent.mkdir(parents=True, exist_ok=True)\n            digest = hashlib.sha256()\n            with archive.open("rb") as incoming, local.open("wb") as outgoing:\n                for block in iter(lambda: incoming.read(8 * 1024 * 1024), b""):\n                    digest.update(block)\n                    outgoing.write(block)\n            if part.get("sha256") and digest.hexdigest() != part["sha256"]:\n                raise ValueError("SHA256 архива не совпадает: кеш поврежден")\n            safe_extract(local, sources, EXTRACT_ROOT, missing_only=kind == "root-cache.tar.gz")\n        except (OSError, tarfile.TarError, ValueError) as error:\n            bad.append(f"{part[\'name\']}: {type(error).__name__}: {error}")\n            say("FAIL: " + bad[-1])\n        else:\n            restored.append(kind)\n        finally:\n            local.unlink(missing_ok=True)\n    if bad:\n        write_json(dict(manifest, invalid=bad), env_manifest_path(cfg))\n    progress(completed=len(parts), total=len(parts), current="")\n    say(f"Восстановлено за {human(time.time() - t0)}: {\', \'.join(restored) or \'ничего\'}")\n    say("Далее проверяются версии, зависимости и CUDA; наличие Python само по себе не означает успех.")\n    return 0 if not bad and (ENV_ROOT / "bin/python").is_file() and CODE.is_dir() else 1\n\n\nSTAGES = {"preflight": st_preflight, "plan": st_plan,\n          "run": st_run, "status": st_status, "collect": st_collect,\n          "freeze": st_freeze, "warm": st_warm}\n\n\ndef main():\n    parser = argparse.ArgumentParser(description="Pixel3DMM colab pipeline")\n    parser.add_argument("stage", choices=sorted(STAGES))\n    args = parser.parse_args()\n    cfg = load_config()\n    if args.stage in {"preflight", "run", "freeze"}:\n        apply_env(cfg)\n    else:\n        for folder in (WORK, LOGS):\n            folder.mkdir(parents=True, exist_ok=True)\n    try:\n        rc = STAGES[args.stage](cfg)\n    except KeyboardInterrupt:\n        say("остановлено вручную")\n        rc = 130\n    except Exception as error:\n        import traceback\n        traceback.print_exc()\n        say(f"ПАДЕНИЕ на стадии {args.stage}: {type(error).__name__}: {error}")\n        rc = 1\n    sys.exit(rc)\n\n\nif __name__ == "__main__":\n    main()'

with CellStatus("2. Ядро и хронологическая очередь") as status:
    status.update(phase="Запись исполняемого модуля")
    MODULE = pathlib.Path("/content/p3dmm_pipeline.py")
    MODULE.write_text(PIPELINE_SOURCE + "\n", encoding="utf-8")
    status.run([sys.executable, "-u", str(MODULE), "plan"], "Проверка имен и построение очереди")


## 3. Окружение
Сначала пробует восстановить кеш с Drive, затем проверяет реальные CUDA-операции. Только недостающее
собирается заново. При первой сборке PyTorch3D панель и таймер продолжают обновляться даже без нового вывода.


In [ ]:
ENV_SCRIPT = 'set -euo pipefail\ntrap \'printf "ОШИБКА: строка %s: %s\\n" "$LINENO" "$BASH_COMMAND" >&2\' ERR\nexport MAMBA_ROOT_PREFIX=/content/micromamba-root PYTHONUNBUFFERED=1\nexport PIP_DISABLE_PIP_VERSION_CHECK=1 PIP_PREFER_BINARY=1\nENV=/content/p3dmm-env\nPY=$ENV/bin/python\nREADY=$ENV/.pixel3dmm_setup_v2\nPIXEL_REV=fcd1fa973c7715b02a8948dfc679dff53cf85924\nMICA_REV=af22e7a5810d474bc28a1433db533723d6bd2b07\nFACER_REV=ddd35c76ff840174b8a5403ad1c1255e37b8782b\nPIPNET_REV=b9eab58816437403a34aa5bc3adeafe5081fd36b\n\nphase() {\n  printf \'__P3DMM_STATUS__{"phase":"%s","completed":%d,"total":12}\\n\' "$2" "$1"\n}\n\nconfigure_toolchain() {\n  export CONDA_PREFIX=$ENV CUDA_HOME=$ENV\n  export PATH="$ENV/bin:$PATH" LD_LIBRARY_PATH="$ENV/lib:${LD_LIBRARY_PATH:-}"\n  export CC="$ENV/bin/x86_64-conda-linux-gnu-gcc" CXX="$ENV/bin/x86_64-conda-linux-gnu-g++"\n  export CUDAHOSTCXX="$CXX" FORCE_CUDA=1\n  export MAX_JOBS\n  MAX_JOBS=$(python3 -c \'import os; gb=os.sysconf("SC_AVPHYS_PAGES")*os.sysconf("SC_PAGE_SIZE")/2**30; print(max(1,min(2,os.cpu_count() or 2,int((gb-2)/3))))\')\n  export TORCH_CUDA_ARCH_LIST\n  TORCH_CUDA_ARCH_LIST=$("$PY" -c \'import torch; a=torch.cuda.get_device_capability(); print("%d.%d" % a if a <= (9,0) else "8.9+PTX")\')\n  printf \'CUDA arch: %s | потоков компиляции: %s\\n\' "$TORCH_CUDA_ARCH_LIST" "$MAX_JOBS"\n}\n\nverify_environment() {\n  "$PY" - <<\'PY\'\nimport sys\nimport numpy, torch, torchvision, chumpy, cv2, facer, insightface, pixel3dmm\nfrom pytorch3d.ops import knn_points\nimport nvdiffrast.torch as dr\nassert sys.version_info[:2] == (3, 9), sys.version\nassert numpy.__version__ == "1.23.0", numpy.__version__\nassert torch.__version__ == "2.7.0+cu118", torch.__version__\nassert torchvision.__version__ == "0.22.0+cu118", torchvision.__version__\nassert torch.cuda.is_available(), "Выбери GPU T4/L4 в настройках Colab"\npoints = torch.zeros((1, 1, 3), device="cuda", requires_grad=True)\nknn_points(points, points, K=1).dists.sum().backward()\nvertices = torch.tensor([[[-.5, -.5, 0., 1.], [.5, -.5, 0., 1.], [0., .5, 0., 1.]]],\n                        device="cuda", requires_grad=True)\nfaces = torch.tensor([[0, 1, 2]], device="cuda", dtype=torch.int32)\nraster, _ = dr.rasterize(dr.RasterizeCudaContext(), vertices, faces, resolution=[4, 4])\nraster.sum().backward()\ntorch.cuda.synchronize()\nprint("Окружение проверено:", torch.__version__, torch.cuda.get_device_name(0), flush=True)\nPY\n}\n\nphase 0 \'Проверка GPU\'\nif ! nvidia-smi; then\n  printf \'Нужен GPU: Runtime -> Change runtime type -> T4/L4\\n\' >&2\n  exit 1\nfi\n\nphase 1 \'Системные библиотеки: установка только отсутствующих\'\nmissing=()\nfor pkg in unzip libgl1 libglib2.0-0 build-essential; do\n  dpkg -s "$pkg" >/dev/null 2>&1 || missing+=("$pkg")\ndone\nif [ ${#missing[@]} -gt 0 ]; then\n  apt-get update -qq\n  apt-get install -y -qq "${missing[@]}"\nfi\n\nphase 2 \'Восстановление сохраненного окружения с Drive\'\nif [ ! -x "$PY" ] || [ ! -f /content/pixel3dmm/scripts/run_cropping.py ]; then\n  python3 -u /content/p3dmm_pipeline.py warm || printf \'Полного кеша нет: собираю недостающее\\n\'\nfi\n\nphase 3 \'Проверка существующего окружения без переустановки\'\nif [ -x "$PY" ] && [ "$(git -C /content/pixel3dmm rev-parse HEAD 2>/dev/null || true)" = "$PIXEL_REV" ]; then\n  if configure_toolchain && verify_environment && "$PY" -m pip check; then\n    touch "$READY"\n    phase 12 \'Окружение готово: установка пропущена\'\n    exit 0\n  fi\nfi\n\nphase 4 \'Micromamba и Python 3.9\'\nmkdir -p /content/micromamba-bin\n[ -x /content/micromamba-bin/bin/micromamba ] || curl -LsSf https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /content/micromamba-bin bin/micromamba\nMM=/content/micromamba-bin/bin/micromamba\n[ -x "$PY" ] || "$MM" create -y -p "$ENV" -c conda-forge --override-channels python=3.9 pip\n"$PY" -c \'import sys; assert sys.version_info[:2] == (3, 9), "Другой Python в p3dmm-env: перезапусти runtime"\'\n\nphase 5 \'CUDA 11.8 и GCC 11: совместимый инструмент сборки\'\nif ! "$ENV/bin/x86_64-conda-linux-gnu-g++" -dumpversion 2>/dev/null | grep -Eq \'^11(\\.|$)\' \\\n   || ! "$ENV/bin/nvcc" --version 2>/dev/null | grep -q \'release 11.8\' \\\n   || [ ! -s "$ENV/include/cuda_runtime.h" ]; then\n  "$MM" install -y -p "$ENV" --override-channels -c nvidia/label/cuda-11.8.0 -c conda-forge cuda-nvcc=11.8 cuda-cccl=11.8 cuda-cudart cuda-cudart-dev libcusparse libcusparse-dev libcublas libcublas-dev libcurand libcurand-dev libcusolver libcusolver-dev gcc_linux-64=11 gxx_linux-64=11\nfi\n\nphase 6 \'PyTorch cu118 и базовые зависимости\'\n"$PY" -c \'import torch, torchvision; assert torch.__version__ == "2.7.0+cu118" and torchvision.__version__ == "0.22.0+cu118"\' 2>/dev/null \\\n  || "$PY" -m pip install "torch==2.7.0" "torchvision==0.22.0" --index-url https://download.pytorch.org/whl/cu118\nconfigure_toolchain\n"$PY" -m pip install "numpy==1.23.0" "setuptools<81" ninja packaging fvcore iopath gdown "Cython<3"\n\nclone_repo() {\n  local url=$1 path=$2 revision=$3\n  if [ ! -e "$path" ]; then\n    git init -q "$path"\n    git -C "$path" remote add origin "$url"\n  fi\n  if [ "$(git -C "$path" rev-parse HEAD 2>/dev/null || true)" != "$revision" ]; then\n    git -C "$path" fetch -q --depth 1 origin "$revision"\n    git -C "$path" checkout -q --detach "$revision"\n  fi\n}\n\nphase 7 \'Загрузка только закрепленных ревизий исходного кода\'\nclone_repo https://github.com/SimonGiebenhain/pixel3dmm.git /content/pixel3dmm "$PIXEL_REV"\nBASE=/content/pixel3dmm/src/pixel3dmm/preprocessing\nclone_repo https://github.com/FacePerceiver/facer.git "$BASE/facer" "$FACER_REV"\nclone_repo https://github.com/Zielon/MICA.git "$BASE/MICA" "$MICA_REV"\nclone_repo https://github.com/jhb86253817/PIPNet.git "$BASE/PIPNet" "$PIPNET_REV"\ncp -f "$BASE/replacement_code/farl.py" "$BASE/facer/facer/face_parsing/farl.py"\ncp -f "$BASE/replacement_code/facer_transform.py" "$BASE/facer/facer/transform.py"\ncp -f "$BASE/replacement_code/mica_demo.py" "$BASE/MICA/demo.py"\ncp -f "$BASE/replacement_code/mica.py" "$BASE/MICA/micalib/models/mica.py"\n\nphase 8 \'PyTorch3D: CUDA-сборка только если проверка не проходит\'\n"$PY" -c \'import torch; from pytorch3d.ops import knn_points; p=torch.zeros((1,1,3),device="cuda"); knn_points(p,p,K=1)\' 2>/dev/null \\\n  || "$PY" -m pip install --force-reinstall --no-deps --no-build-isolation "git+https://github.com/facebookresearch/pytorch3d.git@75ebeeaea0908c5527e7b1e305fbc7681382db47"\n\nphase 9 \'Nvdiffrast: установка только если CUDA-проверка не проходит\'\n"$PY" -c \'import nvdiffrast.torch as dr; dr.RasterizeCudaContext()\' 2>/dev/null \\\n  || "$PY" -m pip install --force-reinstall --no-deps --no-build-isolation "git+https://github.com/NVlabs/nvdiffrast.git@v0.3.3"\n\nphase 10 \'Зависимости Pixel3DMM и FaceR без замены PyTorch\'\nprintf \'torch==2.7.0+cu118\\ntorchvision==0.22.0+cu118\\nnumpy==1.23.0\\nopencv-python<4.12\\nopencv-python-headless<4.12\\nCython<3\\nsetuptools<81\\n\' > /content/p3dmm-constraints.txt\n"$PY" -m pip install --no-build-isolation -c /content/p3dmm-constraints.txt -r /content/pixel3dmm/requirements.txt -r "$BASE/facer/requirements.txt"\n"$PY" -m pip install --no-build-isolation --no-deps -e /content/pixel3dmm\n"$PY" -m pip install --no-build-isolation --no-deps -e "$BASE/facer"\n\nphase 11 \'Итоговая проверка импортов и реальных CUDA-операций\'\n"$PY" -m pip check\nverify_environment\ntouch "$READY"\nphase 12 \'Окружение готово\'\n'

with CellStatus("3. Окружение и CUDA", total=12) as status:
    status.run(["bash", "-c", ENV_SCRIPT], "Подготовка окружения")


## 4. Веса, FLAME и кеш следующей сессии
FLAME берется только из двух архивов в корне `MyDrive`. Используется FLAME2023, без переключения на другую модель.
Остальные веса скачиваются с закрепленного HF-зеркала с проверкой размера и SHA256.
В конце автоматически сохраняется кеш, если подходящего еще нет; повторно упаковывать исправный кеш не нужно.


In [ ]:
WEIGHTS_SCRIPT = 'import json, pickle, shutil, stat, subprocess, sys, zipfile\nfrom pathlib import Path, PurePosixPath\nimport numpy as np\nsys.path.insert(0, "/content")\nfrom p3dmm_pipeline import (CODE, DRIVE_ROOT, FLAME_DATA, PIPNET, HF_FILES, HF_REV,\n                            apply_env, load_config, progress, verified_asset, write_json)\napply_env(load_config())\nBASE = CODE / "src/pixel3dmm/preprocessing"\nMICA_REV = "af22e7a5810d474bc28a1433db533723d6bd2b07"\nHF = f"https://huggingface.co/alexnasa/pixel3dmm/resolve/{HF_REV}"\ncompleted, total = 0, 21\n\n\ndef step(label, function, *args, **kwargs):\n    global completed\n    progress(phase=label, current="", completed=completed, total=total)\n    function(*args, **kwargs)\n    completed += 1\n    print("OK:", label, flush=True)\n    progress(completed=completed, total=total)\n\n\ndef download(name, target):\n    target = Path(target)\n    if verified_asset(name, target):\n        print("Проверенный файл уже есть:", name, flush=True)\n        return\n    target.parent.mkdir(parents=True, exist_ok=True)\n    partial = target.with_name(target.name + ".part")\n    subprocess.run(["curl", "--fail", "--location", "--retry", "3", "--connect-timeout", "30",\n                    "--speed-time", "60", "--speed-limit", "1024", "--progress-bar",\n                    "--output", str(partial), f"{HF}/{name}"], check=True)\n    if not verified_asset(name, partial):\n        raise ValueError(f"Неверный размер/SHA256 {name}; повтори ячейку 4")\n    partial.replace(target)\n    verified_asset(name, target)\n\n\ndef valid_model(path):\n    if not path.is_file():\n        return False\n    try:\n        # Only the user\'s licensed FLAME archives are deserialized.\n        with path.open("rb") as stream:\n            model = pickle.load(stream, encoding="latin1")\n        vertices, directions = np.asarray(model["v_template"]), np.asarray(model["shapedirs"])\n        return (vertices.shape == (5023, 3) and directions.shape[:2] == (5023, 3)\n                and directions.shape[2] >= 400 and np.isfinite(vertices).all()\n                and np.isfinite(directions).all())\n    except Exception:\n        return False\n\n\ndef install_flame(version, filename):\n    source, target = DRIVE_ROOT / (version + ".zip"), FLAME_DATA / version / filename\n    if not source.is_file():\n        raise FileNotFoundError(f"Нет {source}; FLAME берется только из корня MyDrive")\n    stamp_path = CODE / ".flame_sources.json"\n    try:\n        stamp = json.loads(stamp_path.read_text(encoding="utf-8"))\n    except (OSError, ValueError):\n        stamp = {}\n    if not isinstance(stamp, dict):\n        stamp = {}\n    source_meta = [source.stat().st_size, source.stat().st_mtime_ns]\n    previous = stamp.get(version) or {}\n    if previous.get("source") == source_meta and target.is_file():\n        target_meta = [target.stat().st_size, target.stat().st_mtime_ns]\n        if previous.get("target") == target_meta or valid_model(target):\n            print("FLAME уже распакован из этого архива и проверен:", version, flush=True)\n            stamp[version] = {"source": source_meta, "target": target_meta}\n            write_json(stamp, stamp_path)\n            return\n    target.parent.mkdir(parents=True, exist_ok=True)\n    partial = target.with_name(target.name + ".part")\n    with zipfile.ZipFile(source) as archive:\n        for info in archive.infolist():\n            path = PurePosixPath(info.filename)\n            if path.is_absolute() or ".." in path.parts or stat.S_ISLNK(info.external_attr >> 16):\n                raise ValueError(f"Опасный путь в FLAME-архиве: {info.filename}")\n        members = [info for info in archive.infolist() if PurePosixPath(info.filename).name == filename\n                   and "__MACOSX" not in PurePosixPath(info.filename).parts and not info.is_dir()]\n        if len(members) != 1:\n            found = [info.filename for info in archive.infolist() if info.filename.lower().endswith(".pkl")]\n            raise ValueError(f"В {source.name} нужен ровно один {filename}. Найдены: {found}. "\n                             "Нужен обычный FLAME2023, не Open; другая модель не подставляется.")\n        with archive.open(members[0]) as incoming, partial.open("wb") as outgoing:\n            shutil.copyfileobj(incoming, outgoing, length=8 * 1024 * 1024)\n    if not valid_model(partial):\n        raise ValueError(f"{source.name} не содержит корректную модель {filename}")\n    partial.replace(target)\n    stamp[version] = {"source": source_meta, "target": [target.stat().st_size, target.stat().st_mtime_ns]}\n    write_json(stamp, stamp_path)\n\n\ndef restore_mica_asset(relative, minimum):\n    target = FLAME_DATA / "FLAME2020" / relative\n    content = subprocess.run(["git", "-C", str(BASE / "MICA"), "show",\n                              f"{MICA_REV}:data/FLAME2020/{relative}"],\n                             stdout=subprocess.PIPE, check=True).stdout\n    if len(content) < minimum:\n        raise ValueError(f"Неполный файл в MICA: {relative}")\n    if target.is_file() and target.read_bytes() == content:\n        return\n    target.parent.mkdir(parents=True, exist_ok=True)\n    partial = target.with_name(target.name + ".part")\n    partial.write_bytes(content)\n    partial.replace(target)\n\n\nstep("FLAME2020 из MyDrive/FLAME2020.zip", install_flame, "FLAME2020", "generic_model.pkl")\nstep("FLAME2023 из MyDrive/FLAME2023.zip", install_flame, "FLAME2023", "flame2023_no_jaw.pkl")\nfor relative, minimum in (("landmark_embedding.npy", 31292), ("FLAME_masks/FLAME_masks.pkl", 215062),\n                          ("head_template.obj", 10000)):\n    step(relative, restore_mica_asset, relative, minimum)\ntargets = {"mica.tar": FLAME_DATA / "pretrained/mica.tar",\n           "epoch59.pth": PIPNET / "snapshots/WFLW/pip_32_16_60_r18_l2_l1_10_1_nb10/epoch59.pth",\n           "uv.ckpt": CODE / "pretrained_weights/uv.ckpt", "normals.ckpt": CODE / "pretrained_weights/normals.ckpt"}\nfor name, target in targets.items():\n    step(name, download, name, target)\nfor name in HF_FILES:\n    if name.startswith("insightface/"):\n        step(name, download, name, Path.home() / ("." + name))\n\n\ndef build_faceboxes():\n    if list((PIPNET / "FaceBoxesV2/utils").glob("*nms*.so")):\n        print("FaceBoxes уже собран", flush=True)\n        return\n    subprocess.run([sys.executable, "build.py", "build_ext", "--inplace"],\n                   cwd=PIPNET / "FaceBoxesV2/utils", check=True)\n\n\ndef verify_cuda():\n    import torch, chumpy, cv2, facer, insightface, pixel3dmm\n    from pytorch3d.ops import knn_points\n    import nvdiffrast.torch as dr\n    points = torch.zeros((1, 1, 3), device="cuda")\n    knn_points(points, points, K=1)\n    dr.RasterizeCudaContext()\n\n\nstep("FaceBoxes CPU NMS", build_faceboxes)\nstep("Импорты, PyTorch3D CUDA и nvdiffrast JIT", verify_cuda)\n\n# Заранее загружаем lazy-веса FaceR, чтобы они попали в кеш среды.\nprewarm_code = r"""\nfrom pathlib import Path\nimport gc\nimport torch\nimport torch.utils.model_zoo\nimport facer\nfrom facer.face_parsing import farl\nbase = Path("/content/pixel3dmm/src/pixel3dmm/preprocessing")\nassert Path(farl.__file__).read_bytes() == (base / "replacement_code/farl.py").read_bytes(), "Загружен facer без нужного replacement"\ncache = Path(torch.hub.get_dir()) / "checkpoints"\nassert cache.resolve() == (Path.home() / ".cache/torch/hub/checkpoints").resolve(), cache\ndetector = facer.face_detector("retinaface/mobilenet", device="cuda")\nparser = facer.face_parser("farl/celebm/448", device="cuda")\ndel detector, parser\ngc.collect()\ntorch.cuda.empty_cache()\n"""\nexec(compile(prewarm_code, "prewarm_facer.py", "exec"))\nprint("Lazy-веса FaceR загружены в кеш", flush=True)\n\nprint("Все веса и сборки готовы", flush=True)\n'

with CellStatus("4. Веса, FLAME и кеш", total=21) as status:
    PY = "/content/p3dmm-env/bin/python"
    if not pathlib.Path(PY).is_file():
        raise RuntimeError("Сначала выполни ячейку 3: изолированное окружение отсутствует.")
    status.run([PY, "-u", "-c", WEIGHTS_SCRIPT], "Проверка и установка весов")
    rc = status.run([PY, "-u", "/content/p3dmm_pipeline.py", "freeze"],
                    "Сохранение кеша для следующей сессии", check=False)
    if rc:
        status.warn("Веса готовы, но кеш среды не сохранен. Анализ доступен; смотри причину в последних событиях.")


## 5. Анализ по хронологии
Проверяет готовность, затем обрабатывает выбранный диапазон. На панели всегда видно текущее фото,
готово/осталось/ошибки и оценку времени. Каждый готовый результат сразу сохраняется на Drive.
При нехватке времени цикл останавливается целиком, не начинает предобработку следующих групп.
Остановка кнопкой Colab безопасна для уже сохраненных фото; незавершенное фото повторится в следующей сессии.


In [ ]:
with CellStatus("5. Хронологический анализ") as status:
    PY, MODULE = "/content/p3dmm-env/bin/python", "/content/p3dmm_pipeline.py"
    if not pathlib.Path(PY).is_file() or not pathlib.Path(MODULE).is_file():
        raise RuntimeError("После сброса Colab снова выполни ячейки 1, 2, 3 и 4.")
    status.run([PY, "-u", MODULE, "preflight"], "Проверка готовности к анализу")
    rc = status.run([PY, "-u", MODULE, "run"], "Обработка фото по дате в имени", check=False)
    if rc:
        status.warn("У части фото есть ошибки. Готовые результаты сохранены; подробности в журналах Drive.")
    index_rc = status.run([PY, "-u", MODULE, "collect"], "Обновление сводного индекса", check=False)
    if index_rc:
        status.warn("Индекс содержит неготовые или ошибочные фото; это не отменяет сохраненные результаты.")


## 6. Сохраненный статус и последнее готовое фото
Не запускает анализ. Работает без GPU и без установленного окружения, если выполнены ячейки 1 и 2.
Показывает точное имя для `START_FROM`, общий прогресс и исходник/кроп/наложение последнего результата.


In [ ]:
with CellStatus("6. Статус и последнее фото", total=2) as status:
    MODULE = pathlib.Path("/content/p3dmm_pipeline.py")
    if not MODULE.is_file():
        raise RuntimeError("Сначала выполни ячейки 1 и 2.")
    status.run([sys.executable, "-u", str(MODULE), "status"], "Чтение состояния с Drive")
    status.update(phase="Превью последнего готового результата", completed=1, total=2)
    conf = json.loads(pathlib.Path("/content/p3dmm_config.json").read_text(encoding="utf-8"))
    resume_file = pathlib.Path(conf["output"]) / "_state/resume.json"
    saved = json.loads(resume_file.read_text(encoding="utf-8")) if resume_file.is_file() else {}
    last = saved.get("last_completed") or {}
    folder = pathlib.Path(last["folder"]) if last.get("folder") else None
    if folder and folder.is_dir():
        status.note("Последнее готовое фото: " + last.get("relpath", folder.name))
        for title, pattern in (("Исходник", "source.*"), ("Кроп", "crop.*"), ("Меш поверх лица", "overlay.*")):
            images = sorted(p for p in folder.glob(pattern) if p.suffix.lower() in (".jpg", ".jpeg", ".png"))
            if images:
                display(HTML("<strong>" + html.escape(title) + "</strong>"))
                display(Image(filename=str(images[0]), width=300))
    else:
        status.note("Готового фото пока нет. Точка продолжения показана в сохраненном статусе.")


## Что сохраняется
`OUTPUT/batches/<исходная папка>/photos/<имя фото>/` содержит `mesh.ply`, свой `canonical.ply`,
`checkpoint.frame`, `params.json` (FLAME, камера, ракурс, конфиг, маски наблюдаемости),
ландмарки, исходник, кроп, сегментацию, нормали, UV, MICA/arcface, `overlay.jpg`, `run.log` и `done.json`.
Предобработка тоже сохраняется в папке фото и может использоваться после перезапуска Colab.

`OUTPUT/_state/` содержит план, журнал попыток, `resume.json` и список фото без даты.
`OUTPUT/_index/photos.csv` и `report.json` содержат сводные результаты. Входные фото и существующие
готовые результаты не удаляются. Архивы результатов, общий joint-трекинг, видео, демо и случайные тестовые
режимы в этом ноутбуке не используются.

Если установка падает, причина видна в панели, а полный локальный лог лежит в `/content/logs/notebook/`.
При повреждении кеша он не считается готовой средой: ячейка 3 проверяет зависимости и CUDA и собирает недостающее.
Лимит Google Colab может наступить раньше заданного бюджета; это внешний лимит, а не обещанная длительность сессии.
